# BreastVision-XAI V3 — BI-RADS Acoustic & Counterfactually Faithful Multi-Task Breast Ultrasound Analysis

> **Phiên bản nâng cấp V3 (từ V2):** Bổ sung các đóng góp mới chuyên biệt cho siêu âm vú và XAI nhân quả nhằm nâng cao đồng thời **tính mới khoa học** và **hiệu năng thực nghiệm** trên BUS-BRA & BUSI:
> 1. **BI-RADS Posterior Acoustic Zone + SCSE Attention (`V3_birads`)**: Bổ sung vùng hậu âm phía dưới khối u (dựa trên vật lý truyền tia siêu âm, tính trực tiếp từ mask dự đoán — không dùng GT mask) kết hợp SCSE Attention tại decoder/neck và **Boundary-Aware Loss** tại đường biên tổn thương.
> 2. **Hierarchical Evidence Coupling Across All Regions**: Khắc phục điểm nghẽn "lách nhánh" của V2 bằng cách đưa bản đồ bằng chứng $E$ điều chế cả 4 vùng (`global`, `lesion`, `perilesional`, `posterior`), vẫn bảo toàn **phân rã cộng chính xác logit** ($\approx 10^{-8}$).
> 3. **Counterfactual Faithfulness Loss (`V3_full`)**: Tối ưu trực tiếp tính trung thực nhân quả ngay trong lúc huấn luyện bằng cách yêu cầu che vùng bằng chứng cao nhất phải làm giảm độ tự tin dự đoán rõ rệt so với che vùng bằng chứng thấp.
> 4. **Chống Overconfidence & Domain Shift**: Label Smoothing ($\epsilon=0.05$), nhiễu nhân *speckle* siêu âm, EMA trọng số mô hình, và 2-view Test-Time Augmentation (TTA).

## 1. Giới thiệu đề tài, RQ và H

**RQ1.** Gộp đặc trưng theo vùng (toàn ảnh, tổn thương, quanh tổn thương) có cải thiện phân loại benign/malignant so với baseline đa nhiệm dùng chung encoder không?
**RQ2.** Evidence map nối trực tiếp vào logit có cho giải thích *trung thực* hơn Grad-CAM/đối chứng ngẫu nhiên không?
**RQ3.** Ràng buộc giải thích không gian (dùng GT mask chỉ trong loss) có cải thiện *định vị* evidence mà không làm giảm hiệu năng và không gây evidence collapse không?
**RQ4.** Consistency regularization (dự đoán và evidence tách riêng) có làm giải thích ổn định hơn dưới biến đổi bảo toàn nhãn không?
**RQ5.** Pipeline huấn luyện trên BUS-BRA có tổng quát hóa sang BUSI (kiểm định ngoài, khóa trước) không?
**RQ6 (Mới ở V3).** Việc kết hợp vùng hậu âm BI-RADS, evidence coupling toàn vùng và Counterfactual Faithfulness Loss (`V3_full`) có khắc phục được hiện tượng suy giảm can thiệp ở V2 (`X6`: 0.0096) và cải thiện AUROC/Dice/ECE/BUSI không?

| Giả thuyết | Phát biểu (có thể bị bác bỏ) | Kiểm tra |
|---|---|---|
| H1 | X1/X2 > X0 về AUROC (BUS-BRA test) | Δ paired bootstrap, nhiều seed |
| H2 | Evidence của X3/V3 có deletion AUC thấp hơn và insertion AUC cao hơn xếp hạng ngẫu nhiên; can thiệp evidence làm giảm xác suất nhiều hơn xoá ngẫu nhiên | mục 20 |
| H3 | X4/V3 có Relevance Mass trong lesion+context cao hơn X3 | mục 20 |
| H4 | X5/X6/V3 có cosine evidence sau inverse-warp cao hơn X3 | mục 20 |
| H5 | AUROC trên BUSI không thấp hơn BUS-BRA quá một biên chấp nhận được (báo cáo mô tả) | mục 24 |
| H6 (Mới ở V3) | `V3_full` cải thiện đồng thời `evidence_intervention`, `ECE`, `HD95` và `AUROC` (cả nội bộ & BUSI) so với `X6` | mục 19, 20, 21, 24, 25 |

**Giao thức dữ liệu:** BUS-BRA (train/val/test chia theo bệnh nhân, 0 rò rỉ) → BUSI (đánh giá ngoài độc lập, loại Normal, khóa pipeline trước khi đọc BUSI). Kết quả không phải kết luận lâm sàng.


## 2. Cài đặt thư viện
Kaggle đã có PyTorch, torchvision, OpenCV, scikit-learn, SciPy, pandas, matplotlib. Cell chỉ cài nếu thiếu. (Streamlit chỉ cần khi chạy ứng dụng, không cần trong notebook.)

In [ ]:
import importlib, subprocess, sys
REQ = {'torch': 'torch', 'torchvision': 'torchvision', 'cv2': 'opencv-python-headless', 'sklearn': 'scikit-learn', 'scipy': 'scipy',
       'pandas': 'pandas', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm', 'numpy': 'numpy'}
miss = []
for mod, pipn in REQ.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        miss.append(pipn)
if miss:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *miss])
print('Thiếu và đã cài:' if miss else 'Đủ thư viện.', miss)

### Ghi các module mã nguồn
Các cell dưới ghi `bus_core.py`, `bus_trust.py` (kế thừa từ phiên bản trước, đã kiểm thử), `losses_xai.py`, `models_xai.py`, `data_audit.py`, `bus_core_v2.py`, `evaluation_xai.py` và `tests_xai.py`. Chúng cũng là các file trong gói ZIP, nên notebook và mã nguồn luôn đồng nhất.

In [ ]:
%%writefile bus_core.py
# bus_core.py - module dùng chung cho huấn luyện, đánh giá và ứng dụng
import os, csv
from datetime import datetime
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ------------------------- Tiền xử lý -------------------------
def ensure_rgb_uint8(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[2] == 4:
        img = img[:, :, :3]
    if img.dtype != np.uint8:
        mx = float(img.max()) if img.size else 1.0
        img = (img.astype(np.float32) / (mx if mx > 0 else 1.0) * 255.0).clip(0, 255).astype(np.uint8)
    return img


def letterbox_square(arr, pad_value=0):
    h, w = arr.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    canvas = np.full((s, s) + arr.shape[2:], pad_value, dtype=arr.dtype)
    canvas[top:top + h, left:left + w] = arr
    return canvas


def preprocess_image(img_rgb, size):
    sq = letterbox_square(ensure_rgb_uint8(img_rgb))
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    return cv2.resize(sq, (size, size), interpolation=interp)


def preprocess_mask(mask_bin, size):
    sq = letterbox_square((np.asarray(mask_bin) > 0).astype(np.uint8) * 255)
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    r = cv2.resize(sq, (size, size), interpolation=interp)
    return (r > 127).astype(np.uint8)


def normalize_to_tensor(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))


# ------------------------- Mô hình -------------------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, drop=0.0):
        super().__init__()
        layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        if drop > 0:
            layers.append(nn.Dropout2d(drop))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    # g: tín hiệu gating (đã upsample, cùng kích thước với x); x: đặc trưng skip-connection
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.w_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
        self.last_alpha = None

    def forward(self, g, x):
        a = self.relu(self.w_g(g) + self.w_x(x))
        alpha = self.psi(a)
        self.last_alpha = alpha.detach()
        return x * alpha


class UNetFamily(nn.Module):
    # attention=False -> U-Net cơ bản; attention=True -> Attention U-Net
    # num_cls>0 -> thêm đầu phân loại trên bottleneck (mô hình đa nhiệm); forward trả (seg_logits, cls_logits)
    def __init__(self, in_ch=3, base=32, attention=True, num_cls=0, cls_drop=0.3):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 8, base * 16]
        self.attention = attention
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ConvBlock(in_ch, ch[0])
        self.enc2 = ConvBlock(ch[0], ch[1])
        self.enc3 = ConvBlock(ch[1], ch[2])
        self.enc4 = ConvBlock(ch[2], ch[3])
        self.bott = ConvBlock(ch[3], ch[4], drop=0.2)
        self.ups = nn.ModuleList()
        self.atts = nn.ModuleList()
        self.decs = nn.ModuleList()
        for i in [4, 3, 2, 1]:
            self.ups.append(nn.ConvTranspose2d(ch[i], ch[i - 1], 2, stride=2))
            if attention:
                self.atts.append(AttentionGate(ch[i - 1], ch[i - 1], max(ch[i - 1] // 2, 8)))
            self.decs.append(ConvBlock(ch[i - 1] * 2, ch[i - 1]))
        self.out = nn.Conv2d(ch[0], 1, 1)
        self.cls_head = nn.Sequential(nn.Dropout(cls_drop), nn.Linear(ch[4], num_cls)) if num_cls > 0 else None

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bott(self.pool(e4))
        d = b
        for k, skip in enumerate([e4, e3, e2, e1]):
            d = self.ups[k](d)
            if self.attention:
                skip = self.atts[k](d, skip)
            d = self.decs[k](torch.cat([d, skip], dim=1))
        seg = self.out(d)
        if self.cls_head is not None:
            return seg, self.cls_head(b.mean(dim=(2, 3)))
        return seg

    def attention_maps(self):
        return [a.last_alpha for a in self.atts] if self.attention else []


def build_effnet_b3(num_classes, pretrained=True):
    loaded = False
    if pretrained:
        try:
            weights = torchvision.models.EfficientNet_B3_Weights.IMAGENET1K_V1
            m = torchvision.models.efficientnet_b3(weights=weights)
            loaded = True
        except Exception as e:
            print('[CẢNH BÁO] Không tải được trọng số pretrained (' + type(e).__name__ + ': ' + str(e)[:150] + ').')
            print('           EfficientNet-B3 sẽ được khởi tạo NGẪU NHIÊN -> không còn là transfer learning.')
            print('           Bật Internet trên Kaggle (Settings) rồi chạy lại nếu muốn dùng pretrained.')
            m = torchvision.models.efficientnet_b3(weights=None)
    else:
        m = torchvision.models.efficientnet_b3(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, num_classes))
    return m, loaded


class FusionClassifier(nn.Module):
    # E4: EfficientNet-B3 nhận 4 kênh = RGB + xác suất phân vùng do seg_net (Attention U-Net, đóng băng) dự đoán.
    # Không dùng mask ground-truth khi suy luận. Kênh thứ 4 khởi tạo trọng số 0 nên lúc bắt đầu mạng tương đương E0.
    def __init__(self, num_classes, seg_base=32, pretrained=False):
        super().__init__()
        self.seg_net = UNetFamily(3, seg_base, attention=True)
        for p in self.seg_net.parameters():
            p.requires_grad = False
        net, self.pretrained_loaded = build_effnet_b3(num_classes, pretrained)
        conv = net.features[0][0]
        new = nn.Conv2d(4, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=False)
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :3] = conv.weight
        net.features[0][0] = new
        self.cls_net = net
        self.last_seg_prob = None

    @property
    def features(self):
        return self.cls_net.features

    def train(self, mode=True):
        super().train(mode)
        self.seg_net.eval()          # phân vùng luôn ở chế độ eval (BN, dropout cố định)
        return self

    def forward(self, x):
        with torch.no_grad():
            sp = torch.sigmoid(self.seg_net(x).float())
        self.last_seg_prob = sp.detach()
        return self.cls_net(torch.cat([x, sp.to(x.dtype)], dim=1))


def build_model(arch, num_classes, base=32, pretrained=False):
    # trả về (model, pretrained_loaded)
    if arch == 'effnet_b3':
        return build_effnet_b3(num_classes, pretrained)
    if arch == 'fusion':
        m = FusionClassifier(num_classes, seg_base=base, pretrained=pretrained)
        return m, m.pretrained_loaded
    if arch == 'unet':
        return UNetFamily(3, base, attention=False), False
    if arch == 'attunet':
        return UNetFamily(3, base, attention=True), False
    if arch == 'attunet_mt':
        return UNetFamily(3, base, attention=True, num_cls=num_classes), False
    raise ValueError('arch không hợp lệ: ' + str(arch))


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ------------------------- Checkpoint -------------------------
def torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)   # chỉ dùng cho file do chính notebook tạo
    except TypeError:
        return torch.load(path, map_location=device)


def save_checkpoint(path, model, meta):
    torch.save({'model': model.state_dict(), 'meta': meta}, path)


def load_checkpoint(path, device):
    ck = torch_load(path, device)
    meta = ck['meta']
    model, _ = build_model(meta['arch'], meta['num_classes'], base=meta.get('base', 32), pretrained=False)
    model.load_state_dict(ck['model'])
    return model.to(device).eval(), meta


# ------------------------- Grad-CAM -------------------------
def gradcam(model, x, class_idx=None, target_layer=None):
    # x: tensor (1,3,H,W). Mặc định dùng khối đặc trưng cuối của EfficientNet (model.features[-1]).
    layer = target_layer if target_layer is not None else model.features[-1]
    store = {}

    def fwd_hook(module, inp, out):
        store['a'] = out
        out.register_hook(lambda g: store.__setitem__('g', g))

    handle = layer.register_forward_hook(fwd_hook)
    try:
        model.eval()
        with torch.enable_grad():
            xin = x.detach().clone().requires_grad_(True)
            logits = model(xin)
            logits = logits[1] if isinstance(logits, tuple) else logits
            probs = torch.softmax(logits.float(), 1)[0].detach().cpu().numpy()
            idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)
            model.zero_grad(set_to_none=True)
            logits[0, idx].backward()
        acts = store['a'].detach()
        grads = store['g'].detach()
        w = grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * acts).sum(1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)[0, 0]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), probs, idx
    finally:
        handle.remove()


# ------------------------- Detection từ mask -------------------------
def mask_to_boxes(mask, min_area=100):
    m = (np.asarray(mask) > 0).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    boxes = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area >= min_area:
            boxes.append((int(x), int(y), int(x + w), int(y + h)))   # (x1, y1, x2, y2), x2/y2 không bao gồm
    return boxes


def box_iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


# ------------------------- Suy luận ảnh mới -------------------------
def predict_array(img_rgb, cls_model, seg_model, size, device, thr=0.5, min_area=100, with_cam=True):
    img = preprocess_image(img_rgb, size)
    x = normalize_to_tensor(img).unsqueeze(0).to(device)
    res = {'image': img, 'probs': None, 'pred_idx': None, 'cam': None,
           'seg_prob': None, 'mask': None, 'boxes': []}
    if cls_model is not None:
        cls_model.eval()
        with torch.no_grad():
            out = cls_model(x)
            logits = out[1] if isinstance(out, tuple) else out
            probs = torch.softmax(logits.float(), 1)[0].cpu().numpy()
        res['probs'] = probs
        res['pred_idx'] = int(probs.argmax())
        if with_cam and hasattr(cls_model, 'features'):
            cam, _, _ = gradcam(cls_model, x, class_idx=res['pred_idx'])
            res['cam'] = cam
    if seg_model is not None:
        seg_model.eval()
        with torch.no_grad():
            out = seg_model(x)
            seg = out[0] if isinstance(out, tuple) else out
            sp = torch.sigmoid(seg.float())[0, 0].cpu().numpy()
        res['seg_prob'] = sp
        res['mask'] = (sp >= thr).astype(np.uint8)
        res['boxes'] = mask_to_boxes(res['mask'], min_area)
    return res


# ------------------------- Hiển thị -------------------------
def overlay_mask(img, mask, color=(255, 0, 0), alpha=0.4):
    out = img.copy()
    m = np.asarray(mask) > 0
    if m.any():
        out[m] = ((1 - alpha) * out[m] + alpha * np.array(color)).astype(np.uint8)
    return out


def draw_boxes(img, boxes, color=(0, 255, 0), thickness=2):
    out = np.ascontiguousarray(img.copy())
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2) - 1, int(y2) - 1), color, thickness)
    return out


def overlay_cam(img, cam, alpha=0.45):
    heat = cv2.applyColorMap((np.clip(cam, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return ((1 - alpha) * img + alpha * heat).astype(np.uint8)


def visualize_prediction(res, class_names, title=None):
    panels = [('Ảnh đã tiền xử lý', res['image'])]
    if res['mask'] is not None:
        panels.append(('Mask dự đoán', overlay_mask(res['image'], res['mask'])))
        panels.append((f"Bounding box ({len(res['boxes'])})", draw_boxes(res['image'], res['boxes'])))
    if res['cam'] is not None:
        panels.append(('Grad-CAM', overlay_cam(res['image'], res['cam'])))
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.4))
    axes = np.atleast_1d(axes)
    for ax, (t, im) in zip(axes, panels):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
        ax.axis('off')
    if res['probs'] is not None:
        txt = ' | '.join(f'{c}: {p * 100:.1f}%' for c, p in zip(class_names, res['probs']))
        head = f"Dự đoán: {class_names[res['pred_idx']]}  ({txt})"
    else:
        head = ''
    fig.suptitle((title + '\n' if title else '') + head, fontsize=11)
    fig.tight_layout()
    return fig


def log_prediction(log_path, filename, class_names, res):
    header = ['timestamp', 'filename', 'predicted_class'] + ['prob_' + c for c in class_names] + \
             ['mask_area_ratio', 'n_boxes', 'boxes_xyxy']
    new = not os.path.exists(log_path)
    probs = res['probs'] if res['probs'] is not None else [float('nan')] * len(class_names)
    area = float(res['mask'].mean()) if res['mask'] is not None else float('nan')
    row = [datetime.now().isoformat(timespec='seconds'), filename,
           class_names[res['pred_idx']] if res['pred_idx'] is not None else ''] + \
          [f'{float(p):.4f}' for p in probs] + [f'{area:.5f}', len(res['boxes']), str(res['boxes'])]
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if new:
            w.writerow(header)
        w.writerow(row)

In [ ]:
%%writefile bus_trust.py
# bus_trust.py - BUS-Trust: hiệu chuẩn, độ bất định, chọn lọc dự đoán, nhất quán đa mô hình, báo cáo PDF
import numpy as np
from datetime import datetime
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

EPS = 1e-12


# ------------------------- Xác suất, hiệu chuẩn -------------------------
def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def entropy(p):
    p = np.clip(p, EPS, 1.0)
    return -(p * np.log(p)).sum(-1)


def nll(p, y):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1.0)).mean())


def brier(p, y):
    return float(((p - np.eye(p.shape[1])[y]) ** 2).sum(1).mean())


def ece(p, y, n_bins=10):
    conf, pred = p.max(1), p.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(out)


def fit_temperature(logits, y):
    # tìm T > 0 cực tiểu NLL trên tập validation (tìm lưới thô rồi tinh)
    grid = np.exp(np.linspace(np.log(0.2), np.log(10.0), 200))
    losses = [nll(softmax(logits, t), y) for t in grid]
    i = int(np.argmin(losses))
    fine = np.linspace(grid[max(i - 1, 0)], grid[min(i + 1, len(grid) - 1)], 100)
    return float(fine[int(np.argmin([nll(softmax(logits, t), y) for t in fine]))])


def mutual_information(P):
    # P: (S, N, K) các mẫu xác suất (MC dropout) -> thông tin tương hỗ (độ bất định tri thức)
    return entropy(P.mean(0)) - entropy(P).mean(0)


# ------------------------- Chọn lọc dự đoán -------------------------
def risk_coverage(unc, wrong):
    order = np.argsort(unc, kind='stable')
    w = np.asarray(wrong, float)[order]
    n = len(w)
    cov = np.arange(1, n + 1) / n
    risk = np.cumsum(w) / np.arange(1, n + 1)
    return cov, risk, float(risk.mean())        # (coverage, risk, AURC)


def safe_auroc(y, s):
    y, s = np.asarray(y), np.asarray(s, float)
    if len(np.unique(y)) < 2 or not np.isfinite(s).all():
        return float('nan')
    return float(roc_auc_score(y, s))


def accept_by_val_quantile(val_unc, test_unc, coverage):
    thr = float(np.quantile(val_unc, coverage))   # ngưỡng chọn CHỈ trên validation
    return thr, np.asarray(test_unc) <= thr


# ------------------------- Nhất quán phân loại - phân vùng - Grad-CAM -------------------------
def seg_evidence(seg_prob, thr=0.5):
    sp = np.asarray(seg_prob, np.float32)
    if sp.ndim == 2:
        sp = sp[None]
    area = (sp >= thr).mean((1, 2))
    peak = sp.reshape(len(sp), -1).max(1)
    return np.stack([peak, np.log10(area + 1e-4)], 1)


def cam_alignment(cam, mask):
    # tỉ lệ năng lượng Grad-CAM nằm trong mask dự đoán; NaN nếu mask rỗng
    cam = np.asarray(cam, np.float32)
    m = np.asarray(mask) > 0
    if cam.ndim == 2:
        cam, m = cam[None], m[None]
    tot = cam.reshape(len(cam), -1).sum(1) + 1e-8
    inside = (cam * m).reshape(len(cam), -1).sum(1) / tot
    area = m.reshape(len(m), -1).mean(1)
    return np.where(area > 0, inside, np.nan), area


class Standardizer:
    def fit(self, F):
        F = np.asarray(F, float)
        self.mean = np.nanmean(F, 0)
        self.mean = np.where(np.isfinite(self.mean), self.mean, 0.0)
        Fi = np.where(np.isfinite(F), F, self.mean)
        self.std = Fi.std(0) + 1e-8
        return self

    def transform(self, F):
        F = np.asarray(F, float)
        return (np.where(np.isfinite(F), F, self.mean) - self.mean) / self.std


class TumorPresenceMapper:
    # ánh xạ bằng chứng phân vùng -> P(ảnh có khối u); học trên validation để cùng thang xác suất với P(tumor) của bộ phân loại
    def fit(self, ev, has_tumor):
        self.sc = Standardizer().fit(ev)
        self.lr = LogisticRegression(C=1.0).fit(self.sc.transform(ev), has_tumor.astype(int))
        return self

    def predict(self, ev):
        return self.lr.predict_proba(self.sc.transform(ev))[:, 1]

    def to_dict(self):
        return dict(mean=self.sc.mean.tolist(), std=self.sc.std.tolist(), coef=self.lr.coef_[0].tolist(), intercept=float(self.lr.intercept_[0]))

    @staticmethod
    def from_dict(d):
        m = TumorPresenceMapper()
        m.sc = Standardizer(); m.sc.mean, m.sc.std = np.array(d['mean']), np.array(d['std'])
        m.coef, m.intercept = np.array(d['coef']), d['intercept']
        m.predict = lambda ev: 1 / (1 + np.exp(-(((np.asarray(ev) - m.sc.mean) / m.sc.std) @ m.coef + m.intercept)))
        return m


class ErrorPredictor:
    # hồi quy logistic chuẩn hóa, ít đặc trưng, regularization mạnh; học trên validation để dự đoán 'dự đoán của E0 sai'
    def __init__(self, C=0.3):
        self.C = C

    def fit(self, F, wrong):
        self.sc = Standardizer().fit(F)
        self.lr = LogisticRegression(C=self.C, class_weight='balanced', max_iter=1000).fit(self.sc.transform(F), np.asarray(wrong).astype(int))
        return self

    def score(self, F):
        return self.lr.decision_function(self.sc.transform(F))

    def to_dict(self):
        return dict(mean=self.sc.mean.tolist(), std=self.sc.std.tolist(), coef=self.lr.coef_[0].tolist(), intercept=float(self.lr.intercept_[0]), C=self.C)

    @staticmethod
    def from_dict(d):
        p = ErrorPredictor(d.get('C', 0.3))
        mean, std, coef, b = np.array(d['mean']), np.array(d['std']), np.array(d['coef']), d['intercept']
        p.score = lambda F: ((np.where(np.isfinite(np.asarray(F, float)), np.asarray(F, float), mean) - mean) / std) @ coef + b
        return p


# ------------------------- Phân tích một ảnh (dùng trong ứng dụng) -------------------------
def analyze_trust(res, cfg):
    # res: kết quả core.predict_array (cần probs, mask, seg_prob, cam); cfg: trust_config.json đã nạp
    names = cfg['class_names']
    probs = np.clip(np.asarray(res['probs'], float), EPS, 1.0)
    p_cal = softmax(np.log(probs), cfg['temperature'])
    if p_cal.ndim == 2:
        p_cal = p_cal[0]
    out = dict(p_cal=p_cal.tolist(), pred_idx=int(np.argmax(p_cal)), entropy=float(entropy(p_cal)), reasons=[])
    ni = cfg.get('normal_idx')
    ev = seg_evidence(res['seg_prob'], cfg['seg_threshold'])
    if ni is not None and cfg.get('mapper'):
        mapper = TumorPresenceMapper.from_dict(cfg['mapper'])
        p_seg = float(mapper.predict(ev)[0])
        p_cls = float(1.0 - p_cal[ni])
        out.update(p_tumor_cls=p_cls, p_tumor_seg=p_seg, dcs=abs(p_cls - p_seg))
    else:
        out.update(p_tumor_cls=float('nan'), p_tumor_seg=float('nan'), dcs=0.0)
    inside = float('nan')
    if res.get('cam') is not None:
        inside = float(cam_alignment(res['cam'], res['mask'])[0][0])
    out['cam_inside'] = inside
    out['mask_area'] = float((np.asarray(res['mask']) > 0).mean())
    F = np.array([[out['entropy'], out['dcs'], inside]])
    pred = ErrorPredictor.from_dict(cfg['predictor'])
    out['risk_score'] = float(pred.score(F)[0])
    th = cfg['thresholds']
    if out['entropy'] > th['entropy']:
        out['reasons'].append('Độ bất định của dự đoán cao (entropy vượt ngưỡng chọn trên tập validation).')
    if out['dcs'] > th['dcs']:
        out['reasons'].append('Kết quả phân loại và phân vùng chưa nhất quán về việc ảnh có khối u hay không.')
    if np.isfinite(inside) and inside < th['cam_inside']:
        out['reasons'].append('Vùng Grad-CAM ít trùng với vùng được phân vùng.')
    out['flag'] = bool(out['risk_score'] >= th['risk_score'])
    out['pred_name'] = names[out['pred_idx']]
    return out


# ------------------------- Báo cáo PDF (không cần thư viện ngoài) -------------------------
def make_pdf_report(path, filename, res, class_names, trust, meta_text, fig=None):
    import matplotlib
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    with PdfPages(path) as pdf:
        f = plt.figure(figsize=(8.27, 11.69))
        f.text(0.07, 0.95, 'BÁO CÁO PHÂN TÍCH ẢNH SIÊU ÂM VÚ (hỗ trợ nghiên cứu)', fontsize=14, weight='bold')
        lines = [f'Tệp ảnh: {filename}', f'Thời gian: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', meta_text, '',
                 'Xác suất (đã hiệu chuẩn bằng temperature scaling):']
        for c, p in zip(class_names, trust['p_cal']):
            lines.append(f'   - {c}: {p * 100:.1f}%')
        lines += ['', f'Lớp dự đoán: {trust["pred_name"]}', f'Entropy dự đoán: {trust["entropy"]:.3f}',
                  f'Tỉ lệ diện tích vùng phân vùng: {trust["mask_area"] * 100:.2f}%',
                  f'Chênh lệch P(có khối u) giữa phân loại và phân vùng: {trust["dcs"]:.3f}',
                  f'Độ trùng Grad-CAM với vùng phân vùng: ' + ('không xác định (mask rỗng)' if not np.isfinite(trust['cam_inside']) else f'{trust["cam_inside"]:.3f}'),
                  '', 'TRẠNG THÁI: ' + ('CẦN CHUYÊN GIA XEM XÉT' if trust['flag'] else 'Không có cảnh báo từ hệ thống kiểm tra độ tin cậy')]
        for r in trust['reasons']:
            lines.append('   * ' + r)
        lines += ['', 'GIỚI HẠN SỬ DỤNG:',
                  '   - Đây là công cụ hỗ trợ nghiên cứu, KHÔNG phải kết luận chẩn đoán.',
                  '   - Mô hình huấn luyện trên một bộ dữ liệu công khai nhỏ, chưa kiểm định ngoài.',
                  '   - Không có cảnh báo không có nghĩa là dự đoán đúng; mọi kết quả cần được bác sĩ chuyên khoa đánh giá.']
        y = 0.91
        for ln in lines:
            f.text(0.07, y, ln, fontsize=9.5)
            y -= 0.022
        pdf.savefig(f); plt.close(f)
        if fig is not None:
            pdf.savefig(fig)

In [ ]:
%%writefile losses_xai.py
# losses_xai.py - loss phân vùng/phân loại, boundary loss, ràng buộc giải thích không gian, counterfactual faithfulness, nhất quán và hình học
import math
import torch
import torch.nn.functional as F

EPS = 1e-8


# ============================ Phân vùng / phân loại ============================
def dice_loss_from_logits(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float()).flatten(1)
    t = target.float().flatten(1)
    inter = (p * t).sum(1)
    return 1 - ((2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def seg_loss(seg_logits, masks):
    assert seg_logits.shape == masks.shape, f'seg {tuple(seg_logits.shape)} != mask {tuple(masks.shape)}'
    return F.binary_cross_entropy_with_logits(seg_logits.float(), masks.float()) + dice_loss_from_logits(seg_logits, masks)


def boundary_mask(masks):
    # Đường biên hình thái học (dilate - erode) trên mask GT, dùng để tăng trọng số loss tại ranh giới tổn thương
    m = masks.float()
    eroded = 1.0 - dilate(1.0 - m, 1)
    return (dilate(m, 1) - eroded).clamp(0, 1)


def boundary_loss(seg_logits, masks, edge_weight=3.0):
    assert seg_logits.shape == masks.shape
    m = masks.float()
    edge = boundary_mask(m)
    w = 1.0 + edge_weight * edge
    bce = F.binary_cross_entropy_with_logits(seg_logits.float(), m, weight=w)
    p = torch.sigmoid(seg_logits.float())
    inter = (p * m * edge).sum((1, 2, 3))
    den = (p * edge).sum((1, 2, 3)) + (m * edge).sum((1, 2, 3))
    edge_dice = 1.0 - ((2.0 * inter + 1.0) / (den + 1.0)).mean()
    return 0.5 * bce + 0.5 * edge_dice


def cls_loss(logits, labels, class_weights=None, label_smoothing=0.0):
    return F.cross_entropy(logits.float(), labels, weight=class_weights, label_smoothing=float(label_smoothing))


# ============================ ROI: tổn thương, vòng quanh và vùng hậu âm BI-RADS ============================
def downsample_mask(mask, size):
    # mask (B,1,H,W) -> (B,1,h,w) trung bình diện tích, giá trị trong [0,1]
    return F.adaptive_avg_pool2d(mask.float(), size)


def dilate(m, k):
    # giãn nở bằng max-pool (k = số ô đặc trưng); khả vi theo đoạn, dùng cho mask mềm
    return F.max_pool2d(m, kernel_size=2 * k + 1, stride=1, padding=k)


def lesion_ring(m, k):
    # vòng quanh tổn thương: dilate_k(m) - m, nằm trong [0,1]
    return (dilate(m, k) - m).clamp(0, 1)


def lesion_plus_context(m, k):
    return dilate(m, k).clamp(0, 1)


def posterior_zone(m, depth_cells=6):
    # Vùng hậu âm phía dưới tổn thương theo trục dọc chùm tia siêu âm (BI-RADS posterior acoustic shadowing/enhancement).
    # Chỉ phụ thuộc mask dự đoán m (B,1,h,w), hoàn toàn không dùng GT mask lúc suy luận; tách rời với m.
    if depth_cells <= 0:
        return torch.zeros_like(m)
    k_h = 2 * depth_cells + 1
    ker = m.new_zeros((1, 1, k_h, 3))
    ker[0, 0, :depth_cells, :] = 1.0
    raw = F.conv2d(m.float(), ker, padding=(depth_cells, 1))
    return (raw.clamp(0.0, 1.0) * (1.0 - m.float())).clamp(0.0, 1.0)


def normalize_evidence(E):
    # chuẩn hóa tổng = 1 trên không gian: loại bỏ phụ thuộc vào thang đo tuyệt đối
    return E / (E.sum((1, 2, 3), keepdim=True) + EPS)


# ============================ Spatial explanation constraint ============================
def explanation_loss(E, gt_mask, mode='lesion_context', ring_cells=2, delta=0.5, density_margin=1.0, min_roi=1e-3, max_roi=0.9):
    En = normalize_evidence(E.float())
    m = downsample_mask(gt_mask, E.shape[-2:])
    roi = lesion_plus_context(m, ring_cells)
    a_l, a_r = m.mean((1, 2, 3)), roi.mean((1, 2, 3))
    M_l, M_r = (En * m).sum((1, 2, 3)), (En * roi).sum((1, 2, 3))
    if mode == 'none':
        per = torch.zeros_like(M_l)
    elif mode == 'mask_only':
        per = 1 - M_l
    elif mode == 'lesion_context':
        per = F.relu((a_r + delta * (1 - a_r)) - M_r)
    elif mode == 'density':
        rho_in, rho_out = M_r / (a_r + EPS), (1 - M_r).clamp(min=0) / (1 - a_r + EPS)
        per = F.relu(torch.log(rho_out + EPS) - torch.log(rho_in + EPS) + math.log(1 + density_margin))
    else:
        raise ValueError('expl mode không hợp lệ: ' + str(mode))
    valid = ((a_r > min_roi) & (a_r < max_roi)).float()
    loss = (per * valid).sum() / valid.sum().clamp(min=1.0)
    return loss, dict(M_l=M_l.detach(), M_r=M_r.detach(), a_l=a_l.detach(), a_r=a_r.detach(), valid=valid.detach())


# ============================ Counterfactual faithfulness loss (V3) ============================
def counterfactual_faithfulness_loss(logits_full, logits_drop_hi, logits_drop_lo, margin=0.15, low_tol=0.05, conf_thr=0.6):
    # Tối ưu trực tiếp tính trung thực nhân quả: che vùng evidence cao nhất phải làm tụt xác suất lớp dự đoán >= margin,
    # trong khi che vùng evidence thấp nhất chỉ gây thay đổi nhỏ <= low_tol (trên các mẫu có độ tự tin >= conf_thr).
    p_full = F.softmax(logits_full.float(), 1)
    p_hi = F.softmax(logits_drop_hi.float(), 1)
    p_lo = F.softmax(logits_drop_lo.float(), 1)
    cls = p_full.argmax(1)
    idx = torch.arange(p_full.size(0), device=p_full.device)
    pc = p_full[idx, cls]
    d_hi = pc - p_hi[idx, cls]
    d_lo = pc - p_lo[idx, cls]
    per = F.relu(margin - d_hi) + F.relu(d_lo.abs() - low_tol)
    valid = (pc >= conf_thr).float()
    return (per * valid).sum() / valid.sum().clamp(min=1.0)


# ============================ Hình học: affine, inverse warp, mặt nạ pixel hợp lệ ============================
def sample_affine(B, device, max_rot_deg=10.0, scale=(0.9, 1.1), max_trans=0.05, flip_prob=0.5, generator=None):
    def u(lo, hi):
        return lo + (hi - lo) * torch.rand(B, device=device, generator=generator)
    ang = u(-max_rot_deg, max_rot_deg) * math.pi / 180
    s = u(scale[0], scale[1])
    fx = torch.where(torch.rand(B, device=device, generator=generator) < flip_prob, -torch.ones(B, device=device), torch.ones(B, device=device))
    tx, ty = u(-max_trans, max_trans) * 2, u(-max_trans, max_trans) * 2
    c, sn = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = c / s * fx, -sn / s, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sn / s * fx, c / s, ty
    return theta


def hflip_theta(B, device):
    theta = torch.zeros(B, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 1, 1] = -1.0, 1.0
    return theta


def invert_theta(theta):
    B = theta.shape[0]
    A = torch.cat([theta, torch.tensor([0., 0., 1.], device=theta.device).view(1, 1, 3).expand(B, 1, 3)], 1)
    return torch.linalg.inv(A)[:, :2, :]


def warp(t, theta, mode='bilinear', padding_mode='zeros'):
    grid = F.affine_grid(theta, list(t.shape), align_corners=False)
    return F.grid_sample(t, grid, mode=mode, padding_mode=padding_mode, align_corners=False)


def inverse_warp(t_aug, theta, mode='bilinear'):
    return warp(t_aug, invert_theta(theta), mode=mode)


def valid_pixel_mask(theta, size_hw, device=None, thr=0.999):
    ones = torch.ones(theta.shape[0], 1, size_hw[0], size_hw[1], device=theta.device)
    real_content = (warp(ones, theta) > thr).float()
    return (inverse_warp(real_content, theta) > thr).float()


def intensity_jitter(x, contrast=(0.9, 1.1), brightness=0.1, generator=None):
    B = x.shape[0]
    c = contrast[0] + (contrast[1] - contrast[0]) * torch.rand(B, 1, 1, 1, device=x.device, generator=generator)
    b = (torch.rand(B, 1, 1, 1, device=x.device, generator=generator) * 2 - 1) * brightness
    return x * c + b


def apply_label_preserving(x, theta, jitter=True):
    xa = warp(x, theta)
    return intensity_jitter(xa) if jitter else xa


# ============================ Explanation consistency ============================
def prediction_consistency(logits1, logits2):
    p1, p2 = F.softmax(logits1.float(), 1), F.softmax(logits2.float(), 1)
    l1, l2 = F.log_softmax(logits1.float(), 1), F.log_softmax(logits2.float(), 1)
    kl12 = (p1 * (l1 - l2)).sum(1)
    kl21 = (p2 * (l2 - l1)).sum(1)
    return (0.5 * (kl12 + kl21)).mean()


def evidence_alignment(E1, E2, theta, min_valid_frac=0.25):
    E1n, E2n = normalize_evidence(E1.float()), normalize_evidence(E2.float())
    back = inverse_warp(E2n, theta)
    V = valid_pixel_mask(theta, E1.shape[-2:])
    a, b = (E1n * V).flatten(1), (back * V).flatten(1)
    cos = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + EPS)
    ok = (V.mean((1, 2, 3)) >= min_valid_frac).float()
    return cos, ok, back, V


def evidence_consistency(E1, E2, theta):
    cos, ok, _, _ = evidence_alignment(E1, E2, theta)
    return ((1 - cos) * ok).sum() / ok.sum().clamp(min=1.0)


# ============================ Tổng loss ============================
class LossCfg:
    def __init__(self, lam_seg=1.0, lam_cls=1.0, lam_exp=0.5, expl_mode='none', ring_cells=2, delta=0.5, density_margin=1.0,
                 lam_cons_pred=0.0, lam_cons_ev=0.0, label_smoothing=0.0, lam_edge=0.0, lam_faith=0.0, faith_q=0.2, faith_margin=0.15):
        self.__dict__.update(locals()); self.__dict__.pop('self')


def total_loss(out, masks, labels, cfg, class_weights=None, out_aug=None, theta=None, logits_drop_hi=None, logits_drop_lo=None):
    parts = {}
    if out.get('seg') is not None and cfg.lam_seg > 0:
        parts['seg'] = cfg.lam_seg * seg_loss(out['seg'], masks)
    if out.get('seg') is not None and getattr(cfg, 'lam_edge', 0.0) > 0:
        parts['edge'] = cfg.lam_edge * boundary_loss(out['seg'], masks)
    if out.get('logits') is not None and cfg.lam_cls > 0:
        parts['cls'] = cfg.lam_cls * cls_loss(out['logits'], labels, class_weights, getattr(cfg, 'label_smoothing', 0.0))
    if cfg.expl_mode != 'none' and out.get('evidence') is not None and cfg.lam_exp > 0:
        parts['exp'] = cfg.lam_exp * explanation_loss(out['evidence'], masks, cfg.expl_mode, cfg.ring_cells, cfg.delta, cfg.density_margin)[0]
    if out_aug is not None and theta is not None:
        if cfg.lam_cons_pred > 0 and out.get('logits') is not None:
            parts['cons_pred'] = cfg.lam_cons_pred * prediction_consistency(out['logits'], out_aug['logits'])
        if cfg.lam_cons_ev > 0 and out.get('evidence') is not None and out_aug.get('evidence') is not None:
            parts['cons_ev'] = cfg.lam_cons_ev * evidence_consistency(out['evidence'], out_aug['evidence'], theta)
    if getattr(cfg, 'lam_faith', 0.0) > 0 and logits_drop_hi is not None and logits_drop_lo is not None and out.get('logits') is not None:
        parts['faith'] = cfg.lam_faith * counterfactual_faithfulness_loss(
            out['logits'], logits_drop_hi, logits_drop_lo, margin=getattr(cfg, 'faith_margin', 0.15)
        )
    total = sum(parts.values()) if parts else torch.zeros((), requires_grad=True)
    return total, {k: float(v.detach()) for k, v in parts.items()}


In [ ]:
%%writefile models_xai.py
# models_xai.py - BreastVision-XAI V3: SCSE encoder/decoder, BI-RADS posterior zone, evidence coupling toàn diện & phân rã logit
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from bus_core import build_effnet_b3, UNetFamily
from losses_xai import lesion_ring, posterior_zone, EPS


# ============================ Khối chú ý nhẹ & Encoder/Decoder ============================
class SCSEBlock(nn.Module):
    # Concurrent Spatial and Channel Squeeze & Excitation (nhẹ, tăng chất lượng biên tổn thương nhỏ)
    def __init__(self, ch, r=8):
        super().__init__()
        mid = max(4, ch // r)
        self.cse = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
                                 nn.Conv2d(mid, ch, 1), nn.Sigmoid())
        self.sse = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.cse(x) + x * self.sse(x)


class EffNetB3Encoder(nn.Module):
    TAPS = (1, 2, 3, 5, 8)
    CH = {1: 24, 2: 32, 3: 48, 5: 136, 8: 1536}

    def __init__(self, pretrained=True):
        super().__init__()
        net, self.pretrained_loaded = build_effnet_b3(2, pretrained)
        self.features = net.features

    def forward(self, x):
        out = {}
        for i, blk in enumerate(self.features):
            x = blk(x)
            if i in self.TAPS:
                out[i] = x
        return out


def conv_block(cin, cout, use_scse=False):
    layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
              nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    if use_scse:
        layers.append(SCSEBlock(cout))
    return nn.Sequential(*layers)


def _up(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)


class SegDecoder(nn.Module):
    def __init__(self, ch=(256, 128, 64, 32), use_scse=False):
        super().__init__()
        C = EffNetB3Encoder.CH
        self.b4 = conv_block(C[8] + C[5], ch[0], use_scse=use_scse)
        self.b3 = conv_block(ch[0] + C[3], ch[1], use_scse=use_scse)
        self.b2 = conv_block(ch[1] + C[2], ch[2], use_scse=use_scse)
        self.b1 = conv_block(ch[2] + C[1], ch[3], use_scse=use_scse)
        self.out = nn.Conv2d(ch[3], 1, 1)

    def forward(self, f, out_size):
        d = self.b4(torch.cat([_up(f[8], f[5]), f[5]], 1))
        d = self.b3(torch.cat([_up(d, f[3]), f[3]], 1))
        d = self.b2(torch.cat([_up(d, f[2]), f[2]], 1))
        d = self.b1(torch.cat([_up(d, f[1]), f[1]], 1))
        return F.interpolate(self.out(d), size=out_size, mode='bilinear', align_corners=False)


class Neck(nn.Module):
    def __init__(self, dim, use_scse=False):
        super().__init__()
        C = EffNetB3Encoder.CH
        self.l3, self.l5, self.l8 = nn.Conv2d(C[3], dim, 1), nn.Conv2d(C[5], dim, 1), nn.Conv2d(C[8], dim, 1)
        fuse_layers = [nn.Conv2d(dim, dim, 3, padding=1, bias=False), nn.BatchNorm2d(dim), nn.ReLU(inplace=True)]
        if use_scse:
            fuse_layers.append(SCSEBlock(dim))
        self.fuse = nn.Sequential(*fuse_layers)

    def forward(self, f):
        z = self.l3(f[3]) + _up(self.l5(f[5]), f[3]) + _up(self.l8(f[8]), f[3])
        return self.fuse(z)


# ============================ Gộp đa vùng ============================
def masked_avg_pool(Z, m, eps=1e-3):
    return (Z * m).sum((2, 3)) / (m.sum((2, 3)) + eps)


def evidence_zone_weights(Eu, zone_mask, eps=1e-3):
    # V3: trọng số không gian trong vùng giải phẫu dựa trên evidence Eu (B,1,h,w).
    # Nếu zone_mask rỗng -> w = 0 (hữu hạn); nếu khác rỗng -> chuẩn hóa tổng theo không gian (có +eps ổn định).
    raw = Eu * zone_mask
    return raw / (raw.sum((2, 3), keepdim=True) + eps)


class RegionFusion(nn.Module):
    def __init__(self, mode, n_regions, dim):
        super().__init__()
        self.mode, self.n, self.dim = (mode if n_regions > 1 else 'concat'), n_regions, dim
        self.proj = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_regions)])
        if self.mode == 'weighted':
            self.a = nn.Parameter(torch.zeros(n_regions))
        if self.mode == 'gated':
            self.gate = nn.Linear(n_regions * dim, n_regions * dim)
        self.out_dim = n_regions * dim if self.mode == 'concat' else dim

    def forward(self, xs):
        v = [p(x) for p, x in zip(self.proj, xs)]
        if self.mode == 'concat':
            return torch.cat(v, 1)
        if self.mode == 'weighted':
            w = F.softmax(self.a, 0)
            return sum(wi * vi for wi, vi in zip(w, v))
        g = F.softmax(self.gate(torch.cat(xs, 1)).view(-1, self.n, self.dim), 1)
        return (g * torch.stack(v, 1)).sum(1)


# ============================ Mô hình chính ============================
@dataclass
class XAIConfig:
    num_classes: int = 2
    dim: int = 128
    fusion: str = 'concat'          # concat | weighted | gated
    use_lesion: bool = True
    use_peri: bool = True
    evidence_mode: str = 'ewp'      # none | ewp | egr | decoupled
    ring_cells: int = 2
    detach_mask: bool = True
    dropout: float = 0.3
    pretrained: bool = True
    evidence_tau: float = 1.0
    # --- Nâng cấp V3 (mặc định False để X0-X6 giữ nguyên 100%) ---
    use_scse: bool = False
    use_posterior: bool = False
    post_depth: int = 6
    evidence_coupling_all: bool = False


class BreastVisionXAI(nn.Module):
    # Suy luận CHỈ dùng ảnh; mọi vùng giải phẫu (lesion, peri, posterior) được suy ra từ mask DỰ ĐOÁN.
    def __init__(self, cfg: XAIConfig):
        super().__init__()
        assert cfg.evidence_mode in ('none', 'ewp', 'egr', 'decoupled')
        self.cfg = cfg
        self.enc = EffNetB3Encoder(cfg.pretrained)
        self.pretrained_loaded = self.enc.pretrained_loaded
        self.dec = SegDecoder(use_scse=cfg.use_scse)
        self.neck = Neck(cfg.dim, use_scse=cfg.use_scse)
        n = 1 + int(cfg.use_lesion) + int(cfg.use_peri) + int(cfg.use_posterior)
        self.fusion = RegionFusion(cfg.fusion, n, cfg.dim)
        self.head = nn.Sequential(nn.Dropout(cfg.dropout), nn.Linear(self.fusion.out_dim, cfg.num_classes))
        if cfg.evidence_mode != 'none':
            self.ev_head = nn.Sequential(nn.Conv2d(cfg.dim, cfg.dim // 2, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(cfg.dim // 2, 1, 1))

    def _compute_regions(self, Z, m, ring, post, E, ev_mask=None):
        cfg = self.cfg
        if cfg.evidence_mode == 'none' or cfg.evidence_mode == 'decoupled':
            g = Z.mean((2, 3))
            Eu = None
            wg = torch.ones_like(m) / (m.shape[-2] * m.shape[-1])
        else:
            Eu = E if ev_mask is None else E * ev_mask
            wg = Eu if cfg.evidence_mode == 'ewp' else Eu / (Z.shape[-2] * Z.shape[-1])
            g = (Z * Eu).sum((2, 3)) if cfg.evidence_mode == 'ewp' else (Z * Eu).mean((2, 3))
        regions = [g]
        weights = [wg]
        zone_masks = []
        if cfg.use_lesion:
            zone_masks.append(m)
        if cfg.use_peri:
            zone_masks.append(ring)
        if cfg.use_posterior:
            zone_masks.append(post)
        for zm in zone_masks:
            if cfg.evidence_coupling_all and Eu is not None:
                wz = evidence_zone_weights(Eu, zm)
                regions.append((Z * wz).sum((2, 3)))
                weights.append(wz)
            else:
                regions.append(masked_avg_pool(Z, zm))
                weights.append(zm / (zm.sum((2, 3), keepdim=True) + 1e-3))
        return regions, weights

    def forward_heads_from_Z(self, Z, m, ring, post, E, ev_mask=None):
        # Dùng trong counterfactual faithfulness loss: chạy lại phần pooling + fusion + classifier mà KHÔNG chạy lại encoder/decoder
        regions, weights = self._compute_regions(Z, m, ring, post, E, ev_mask=ev_mask)
        h_vec = self.fusion(regions)
        return self.head(h_vec), regions, weights

    def forward(self, x, ev_mask=None):
        cfg = self.cfg
        f = self.enc(x)
        seg = self.dec(f, x.shape[-2:])
        Z = self.neck(f)
        m = F.adaptive_avg_pool2d(torch.sigmoid(seg.float()), Z.shape[-2:])
        if cfg.detach_mask:
            m = m.detach()
        E = None
        if cfg.evidence_mode != 'none':
            s = self.ev_head(Z).float() / cfg.evidence_tau
            B, _, h, w = s.shape
            if cfg.evidence_mode in ('ewp', 'decoupled'):
                E = F.softmax(s.flatten(2), -1).view(B, 1, h, w)
            else:
                E = torch.sigmoid(s)
        ring = lesion_ring(m, cfg.ring_cells) if cfg.use_peri else None
        post = posterior_zone(m, cfg.post_depth) if cfg.use_posterior else None
        regions, weights = self._compute_regions(Z, m, ring, post, E, ev_mask=ev_mask)
        logits = self.head(self.fusion(regions))
        return dict(seg=seg, logits=logits, evidence=E, mask_feat=m, ring=ring, post=post, Z=Z, regions=regions, region_weights=weights)

    @torch.no_grad()
    def infer(self, x):
        self.eval()
        o = self.forward(x)
        return dict(prob=F.softmax(o['logits'].float(), 1), seg_prob=torch.sigmoid(o['seg'].float()), evidence=o['evidence'])

    @torch.no_grad()
    def decompose_logits(self, x):
        # Phân rã chính xác logit:
        # - Khi evidence_coupling_all=False (V2): C_k(x) = E(x)*(W_g(A_g Z(x) + b_g))_k, rest_k chứa các vùng còn lại + bias.
        # - Khi evidence_coupling_all=True  (V3): TẤT CẢ các vùng đều đi qua weights(x) dựa trên E(x), nên
        #   C_k(x) = sum_i w_i(x) * [W_i A_i Z(x)]_k và rest_k = bias_k + sum_i [W_i b_i]_k (hằng số thuần túy!).
        assert self.cfg.evidence_mode == 'ewp' and self.fusion.mode == 'concat', 'phân rã chính xác chỉ có cho ewp + concat'
        self.eval()
        o = self.forward(x)
        D = self.cfg.dim
        W, bias = self.head[1].weight, self.head[1].bias
        if not self.cfg.evidence_coupling_all:
            A, b = self.fusion.proj[0].weight, self.fusion.proj[0].bias
            Wg = W[:, :D]
            C = o['evidence'] * (torch.einsum('kd,bdhw->bkhw', Wg @ A, o['Z']) + (Wg @ b).view(1, -1, 1, 1))
            rest = bias.expand(x.shape[0], -1).clone()
            for i, r in enumerate(o['regions'][1:], start=1):
                v = self.fusion.proj[i](r)
                rest = rest + v @ W[:, i * D:(i + 1) * D].T
            return C, rest, o['logits']

        C = torch.zeros(x.shape[0], self.cfg.num_classes, o['Z'].shape[2], o['Z'].shape[3], device=x.device, dtype=o['Z'].dtype)
        rest = bias.expand(x.shape[0], -1).clone()
        for i, w_map in enumerate(o['region_weights']):
            Ai, bi = self.fusion.proj[i].weight, self.fusion.proj[i].bias
            Wi = W[:, i * D:(i + 1) * D]
            C = C + w_map * torch.einsum('kd,bdhw->bkhw', Wi @ Ai, o['Z'])
            rest = rest + (Wi @ bi).view(1, -1)
        return C, rest, o['logits']


# ============================ Bọc mô hình cũ (E0–E4) để dùng chung engine ============================
class LegacyWrapper(nn.Module):
    def __init__(self, model, task, pretrained_loaded=None):
        super().__init__()
        self.model, self.task = model, task
        self.pretrained_loaded = getattr(model, 'pretrained_loaded', False) if pretrained_loaded is None else bool(pretrained_loaded)

    def forward(self, x, ev_mask=None):
        o = self.model(x)
        if self.task == 'cls':
            return dict(seg=None, logits=o, evidence=None)
        if self.task == 'seg':
            return dict(seg=o, logits=None, evidence=None)
        return dict(seg=o[0], logits=o[1], evidence=None)


# ============================ Danh mục thực nghiệm ============================
EXPERIMENTS = {
    # --- baseline gốc ---
    'E0': dict(kind='legacy', arch='effnet_b3', task='cls', desc='EfficientNet-B3 phân loại'),
    'E1': dict(kind='legacy', arch='unet', task='seg', desc='U-Net'),
    'E2': dict(kind='legacy', arch='attunet', task='seg', desc='Attention U-Net'),
    'E3': dict(kind='legacy', arch='attunet_mt', task='mt', desc='Attention U-Net đa nhiệm'),
    'E4': dict(kind='legacy', arch='attunet', task='seg', augment=False, desc='Attention U-Net không augmentation (ablation)'),
    # --- chuỗi X (V2) ---
    'X0': dict(kind='xai', model=dict(use_lesion=False, use_peri=False, evidence_mode='none'), loss={}, desc='Encoder chung đa nhiệm (baseline)'),
    'X1': dict(kind='xai', model=dict(use_lesion=True, use_peri=False, evidence_mode='none'), loss={}, desc='X0 + gộp global/lesion'),
    'X2': dict(kind='xai', model=dict(use_lesion=True, use_peri=True, evidence_mode='none'), loss={}, desc='X1 + vùng quanh tổn thương'),
    'X3': dict(kind='xai', model=dict(evidence_mode='ewp'), loss={}, desc='X2 + evidence-guided classification (EWP)'),
    'X4': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='lesion_context'), desc='X3 + explanation constraint'),
    'X5': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_pred=0.5, lam_cons_ev=0.5), desc='X3 + consistency'),
    'X6': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5), desc='X3 + constraint + consistency'),
    # --- chuỗi V3 (nâng cấp tính mới & hiệu năng) ---
    'V3_base': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True),
                    loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5),
                    ema=True, tta=True, desc='V3 Base: X6 + SCSE + boundary loss + label smoothing + EMA'),
    'V3_birads': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True, use_posterior=True, evidence_coupling_all=True),
                      loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5),
                      ema=True, tta=True, desc='V3 BI-RADS: + vùng hậu âm siêu âm + evidence coupling toàn vùng'),
    'V3_full': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True, use_posterior=True, evidence_coupling_all=True),
                    loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5, lam_faith=0.3),
                    ema=True, tta=True, desc='V3 Full: + Counterfactual Faithfulness Loss (đề xuất chính V3)'),
    # --- ablation bổ sung ---
    'A_fusion_weighted': dict(kind='xai', model=dict(evidence_mode='none', fusion='weighted'), loss={}, desc='X2 với weighted fusion'),
    'A_fusion_gated': dict(kind='xai', model=dict(evidence_mode='none', fusion='gated'), loss={}, desc='X2 với gated fusion'),
    'A_evidence_egr': dict(kind='xai', model=dict(evidence_mode='egr'), loss={}, desc='X3 formulation đối chứng: evidence-gated features (EGR)'),
    'A_evidence_decoupled': dict(kind='xai', model=dict(evidence_mode='decoupled'), loss=dict(expl_mode='lesion_context'), desc='Evidence KHÔNG nối vào logit (kiểm tra vai trò coupling)'),
    'A_exp_mask_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='mask_only'), desc='X3 + ràng buộc mask-only'),
    'A_exp_density': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='density'), desc='X3 + ràng buộc density-based'),
    'A_cons_pred_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_pred=0.5), desc='X3 + chỉ prediction consistency'),
    'A_cons_ev_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_ev=0.5), desc='X3 + chỉ evidence consistency'),
}
MAIN_CONFIGS = ['X0', 'X3', 'X4', 'X6', 'V3_full']      # cấu hình chính: chạy nhiều seed


def build_experiment(name, num_classes=2, pretrained=True, dim=128, base=32):
    spec = EXPERIMENTS[name]
    if spec['kind'] == 'xai':
        cfg = XAIConfig(num_classes=num_classes, pretrained=pretrained, dim=dim, **spec['model'])
        return BreastVisionXAI(cfg), asdict(cfg)
    from bus_core import build_model
    model, loaded = build_model(spec['arch'], num_classes, base=base, pretrained=pretrained)
    return LegacyWrapper(model, spec['task'], pretrained_loaded=loaded), dict(arch=spec['arch'], task=spec['task'], base=base)


In [ ]:
%%writefile data_audit.py
# data_audit.py - khám phá dataset BUS-BRA / BUSI, kiểm toán dữ liệu, chia theo bệnh nhân, kiểm tra rò rỉ, manifest
import os, re, json, hashlib
import numpy as np
import pandas as pd
import cv2

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
MASK_RE = re.compile(r'^(.*)_mask(?:_\d+)?$', re.IGNORECASE)


# ============================ Tiện ích ============================
def read_rgb(path):
    b = cv2.imread(path, cv2.IMREAD_COLOR)
    return None if b is None else cv2.cvtColor(b, cv2.COLOR_BGR2RGB)


def read_mask(path, hw=None):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    if hw is not None and m.shape != tuple(hw):
        m = cv2.resize(m, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    mx = int(m.max())
    return (m > mx // 2).astype(np.uint8) if mx > 0 else np.zeros_like(m)     # chịu được mask 0/1, 0/255, hay nhãn 0/k


def read_union_mask(paths, hw):
    u = np.zeros(hw, np.uint8)
    for p in paths:
        m = read_mask(p, hw)
        if m is not None:
            u |= m
    return u


def dhash256(img_rgb):
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    s = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return (s[:, 1:] > s[:, :-1]).flatten().astype(np.uint8)


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


class _UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def _key(name):
    s = os.path.splitext(os.path.basename(str(name)))[0].lower()
    s = re.sub(r'^(bus|mask|img|image)[_\-]?', '', s)
    s = re.sub(r'[_\-]?mask$', '', s)
    return s


# ============================ Khám phá BUS-BRA ============================
COLS = dict(image=['ID', 'id', 'Image', 'image', 'Filename', 'filename', 'File', 'file', 'Image_ID'],
            patient=['Case', 'case', 'Patient', 'patient', 'PatientID', 'Patient_ID', 'patient_id', 'Case_ID'],
            label=['Pathology', 'pathology', 'Label', 'label', 'Class', 'class'],
            birads=['BIRADS', 'BI-RADS', 'birads', 'BI_RADS'],
            device=['Device', 'device', 'Scanner', 'scanner'],
            fold=['Fold', 'fold', 'Split', 'split', 'Partition'])


def _pick(df, cands, override=None):
    if override:
        if override not in df.columns:
            raise KeyError(f'Cột {override!r} không có trong CSV. Các cột hiện có: {list(df.columns)}')
        return override
    for c in cands:
        if c in df.columns:
            return c
    return None


def find_busbra_candidates(base):
    out = []
    for dp, dns, fs in os.walk(base):
        low = {d.lower(): d for d in dns}
        if 'images' in low and 'masks' in low:
            out.append(dp)
            dns[:] = []
    return out


def find_csvs(root):
    return [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if f.lower().endswith('.csv')]


def index_busbra(root, csv_path=None, col_overrides=None, allow_missing_patient=False):
    # Dựng bảng (ảnh, mask, bệnh nhân, nhãn). Cấu trúc CSV của BUS-BRA chưa được xác minh trực tiếp bởi tác giả code:
    # hàm dò cột theo danh sách tên phổ biến; nếu không khớp sẽ DỪNG kèm danh sách cột để bạn đặt col_overrides.
    col_overrides = col_overrides or {}
    sub = {d.lower(): d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))}
    if 'images' not in sub or 'masks' not in sub:
        raise FileNotFoundError(f'{root} phải có thư mục Images và Masks. Hiện có: {sorted(os.listdir(root))}')
    img_dir, msk_dir = os.path.join(root, sub['images']), os.path.join(root, sub['masks'])
    parent = os.path.dirname(os.path.abspath(root))
    pool = find_csvs(root) + [os.path.join(parent, f) for f in os.listdir(parent) if f.lower().endswith('.csv')]       # CSV có thể nằm ở thư mục cha
    csvs = [csv_path] if csv_path else [c for c in pool if 'fold' not in os.path.basename(c).lower() and 'split' not in os.path.basename(c).lower()]
    if not csvs:
        raise FileNotFoundError('Không tìm thấy file CSV thông tin trong ' + root)
    csv = None
    for c in csvs:
        d = pd.read_csv(c)
        if _pick(d, COLS['label'], col_overrides.get('label')):
            csv, meta = c, d
            break
    if csv is None:
        raise KeyError('Không CSV nào có cột nhãn (Pathology). Hãy đặt col_overrides={"label": ...}. CSV tìm thấy: ' + str(csvs))
    c_img, c_pat, c_lab = (_pick(meta, COLS[k], col_overrides.get(k)) for k in ('image', 'patient', 'label'))
    if c_img is None:
        raise KeyError(f'Không xác định được cột tên ảnh. Cột hiện có: {list(meta.columns)}; đặt col_overrides={{"image": ...}}')
    if c_pat is None and not allow_missing_patient:
        raise KeyError(f'Không có cột mã bệnh nhân (ví dụ Case). Cột hiện có: {list(meta.columns)}. Đặt col_overrides={{"patient": ...}} '
                       'hoặc allow_missing_patient=True (KHÔNG khuyến nghị: mất khả năng chia theo bệnh nhân).')
    imgs = {_key(f): os.path.join(img_dir, f) for f in os.listdir(img_dir) if os.path.splitext(f)[1].lower() in IMG_EXT}
    msks = {_key(f): os.path.join(msk_dir, f) for f in os.listdir(msk_dir) if os.path.splitext(f)[1].lower() in IMG_EXT}
    rows, missing = [], []
    for _, r in meta.iterrows():
        k = _key(r[c_img])
        if k not in imgs or k not in msks:
            missing.append(str(r[c_img])); continue
        lab = str(r[c_lab]).strip().lower()
        label = 1 if lab.startswith('m') else (0 if lab.startswith('b') else None)
        if label is None:
            missing.append(f'{r[c_img]} (nhãn lạ: {lab})'); continue
        rows.append(dict(image_path=imgs[k], mask_path=msks[k], patient_id=str(r[c_pat]) if c_pat else f'img_{k}', label=label, pathology=lab,
                         birads=r[_pick(meta, COLS['birads'])] if _pick(meta, COLS['birads']) else None,
                         device=r[_pick(meta, COLS['device'])] if _pick(meta, COLS['device']) else None,
                         official_fold=r[_pick(meta, COLS['fold'])] if _pick(meta, COLS['fold']) else None, source='BUS-BRA'))
    df = pd.DataFrame(rows)
    info = dict(csv=csv, columns=dict(image=c_img, patient=c_pat, label=c_lab), n_csv_rows=len(meta), n_indexed=len(df), n_unpaired=len(missing), unpaired_examples=missing[:10])
    if len(df) == 0:
        raise RuntimeError(f'Không ghép được ảnh–mask nào. Ví dụ tên trong CSV: {meta[c_img].head(3).tolist()}; ví dụ tên ảnh: {list(imgs)[:3]}; mask: {list(msks)[:3]}')
    return df, info


# ============================ Khám phá BUSI (chỉ benign/malignant) ============================
def find_busi_candidates(base):
    out = []
    for dp, dns, _ in os.walk(base):
        low = {d.lower() for d in dns}
        if 'benign' in low and 'malignant' in low:
            out.append(dp); dns[:] = []
    return out


def index_busi_binary(root):
    # KHÔNG chuyển normal thành benign: ảnh normal bị loại khỏi tập đánh giá ngoài.
    rows, n_normal = [], 0
    for d in sorted(os.listdir(root)):
        c = d.lower()
        full = os.path.join(root, d)
        if not os.path.isdir(full) or c not in ('benign', 'malignant', 'normal'):
            continue
        imgs, masks = {}, {}
        for f in sorted(os.listdir(full)):
            stem, ext = os.path.splitext(f)
            if ext.lower() not in IMG_EXT:
                continue
            m = MASK_RE.match(stem)
            (masks.setdefault(m.group(1), []).append(os.path.join(full, f)) if m else imgs.__setitem__(stem, os.path.join(full, f)))
        if c == 'normal':
            n_normal += len(imgs); continue
        for stem, p in imgs.items():
            rows.append(dict(image_path=p, mask_paths=sorted(masks.get(stem, [])), patient_id=f'busi_{c}_{stem}', label=int(c == 'malignant'),
                             pathology=c, source='BUSI'))
    return pd.DataFrame(rows), dict(n_normal_excluded=n_normal)


# ============================ Kiểm toán ============================
def audit_dataframe(df, mask_col='mask_path', dup_hamming=8, read_pixels=True):
    # Trả (df_đã_gắn_cột, báo_cáo). Phát hiện: ảnh/mask lỗi, mask rỗng, lệch kích thước, trùng tuyệt đối, gần trùng, nhãn mâu thuẫn.
    df = df.reset_index(drop=True).copy()
    n = len(df)
    if n == 0:
        raise ValueError('audit_dataframe nhận bảng RỖNG: bước khám phá dataset không tìm thấy ảnh nào. Kiểm tra đường dẫn và tên cột.')
    md5, dh, ok, hh, ww, area, msz, issue = [], [], [], [], [], [], [], []
    for r in df.itertuples(index=False):
        iss = []
        img = read_rgb(r.image_path)
        if img is None:
            md5.append(''); dh.append(np.zeros(256, np.uint8)); hh.append(0); ww.append(0); area.append(0); msz.append(False); issue.append('anh_loi'); continue
        mp = getattr(r, mask_col)
        paths = mp if isinstance(mp, (list, tuple)) else [mp]
        m = np.zeros(img.shape[:2], np.uint8)
        mism = False
        for p in paths:
            raw = cv2.imread(p, cv2.IMREAD_GRAYSCALE) if p else None
            if raw is None:
                iss.append('mask_loi_hoac_thieu'); continue
            if raw.shape != img.shape[:2]:
                mism = True
            m |= read_mask(p, img.shape[:2])
        if m.sum() == 0:
            iss.append('mask_rong')
        md5.append(sha256_file(r.image_path)); dh.append(dhash256(img)); hh.append(img.shape[0]); ww.append(img.shape[1])
        area.append(float(m.mean())); msz.append(mism); issue.append(';'.join(iss))
    df['sha256'], df['h'], df['w'], df['mask_ratio'], df['mask_size_mismatch'], df['issue'] = md5, hh, ww, area, msz, issue
    bits = np.stack(dh).astype(np.int32)
    ham = bits @ (1 - bits).T + (1 - bits) @ bits.T
    uf = _UF(n)
    ii, jj = np.where(np.triu(ham <= dup_hamming, k=1))
    for a, b in zip(ii, jj):
        if df.loc[a, 'sha256'] != '' and df.loc[b, 'sha256'] != '':
            uf.union(int(a), int(b))
    df['dup_group'] = [uf.find(i) for i in range(n)]
    gs = df.groupby('dup_group').size()
    exact = df.groupby('sha256').filter(lambda g: len(g) > 1 and g.name != '')
    conflict_groups = df.groupby('dup_group')['label'].nunique()
    rep = dict(n=n, n_bad_image=int((df['issue'].str.contains('anh_loi')).sum()), n_bad_or_missing_mask=int(df['issue'].str.contains('mask_loi').sum()),
               n_empty_mask=int(df['issue'].str.contains('mask_rong').sum()), n_mask_size_mismatch=int(df['mask_size_mismatch'].sum()),
               n_exact_duplicate_images=int(len(exact)), n_neardup_groups=int((gs > 1).sum()), n_images_in_neardup_groups=int(gs[gs > 1].sum()),
               n_label_conflict_groups=int((conflict_groups > 1).sum()), class_counts=df['label'].value_counts().sort_index().to_dict(),
               n_patients=int(df['patient_id'].nunique()) if 'patient_id' in df else None)
    df['_dhash'] = [d.tobytes() for d in dh]
    return df, rep


def drop_invalid(df):
    # loại ảnh/mask lỗi, mask rỗng, nhãn mâu thuẫn trong cùng nhóm trùng; giữ 1 bản cho trùng tuyệt đối cùng nhãn
    bad = df['issue'] != ''
    conflict = df.groupby('dup_group')['label'].transform('nunique') > 1
    keep = ~bad & ~conflict
    out = df[keep].copy()
    if len(out) == 0:
        raise RuntimeError('Sau khi lọc không còn ảnh nào hợp lệ. Số lượng theo lý do: ' + str(df['issue'].replace('', 'ok').value_counts().to_dict())
                           + ' | nhãn mâu thuẫn trong nhóm trùng: ' + str(int(conflict.sum())) + '. Xem hướng dẫn xử lý trong README.')
    out = out.drop_duplicates(subset='sha256', keep='first')
    return out.reset_index(drop=True), dict(dropped_invalid=int(bad.sum()), dropped_label_conflict=int((conflict & ~bad).sum()), dropped_exact_dups=int(keep.sum() - len(out)))


# ============================ Gom nhóm và chia theo bệnh nhân ============================
def build_groups(df):
    # nhóm = hợp của (cùng bệnh nhân) và (ảnh trùng/gần trùng)
    n = len(df)
    uf = _UF(n)
    for col in ('patient_id', 'dup_group'):
        first = {}
        for i, v in enumerate(df[col].values):
            if v in first:
                uf.union(first[v], i)
            else:
                first[v] = i
    return np.array([uf.find(i) for i in range(n)])


def grouped_split(df, seed=42, val_frac=0.15, test_frac=0.15, official_test_fold=None):
    # Chia theo nhóm (bệnh nhân + ảnh trùng), phân tầng theo nhãn. Nếu official_test_fold được cho và cột official_fold có giá trị,
    # test = fold đó nhưng validation vẫn tái tạo từ bệnh nhân KHÔNG thuộc test (tránh lệch bệnh nhân giữa train/val được cung cấp).
    from sklearn.model_selection import StratifiedGroupKFold
    df = df.reset_index(drop=True).copy()
    df['group'] = build_groups(df)
    idx = np.arange(len(df))
    y, g = df['label'].values, df['group'].values
    if official_test_fold is not None and df['official_fold'].notna().any():
        test = np.where(df['official_fold'].astype(str).values == str(official_test_fold))[0]
        tg = set(g[test])
        test = np.array([i for i in idx if g[i] in tg])          # kéo cả nhóm vào test (bệnh nhân không tách đôi)
        dev = np.array([i for i in idx if g[i] not in tg])
    else:
        k1 = max(2, int(round(1 / test_frac)))
        dev, test = next(StratifiedGroupKFold(n_splits=k1, shuffle=True, random_state=seed).split(idx, y, g))
    k2 = max(2, int(round(1 / (val_frac / (1 - test_frac)))))
    tr_rel, va_rel = next(StratifiedGroupKFold(n_splits=k2, shuffle=True, random_state=seed).split(dev, y[dev], g[dev]))
    df['split'] = ''
    df.loc[dev[tr_rel], 'split'] = 'train'; df.loc[dev[va_rel], 'split'] = 'val'; df.loc[test, 'split'] = 'test'
    return df


def leakage_report(df, dup_hamming=8):
    # kiểm tra: nhóm, bệnh nhân, sha256, và cặp gần trùng (dHash) nằm ở các tập khác nhau
    rep = {}
    for col in ('group', 'patient_id', 'sha256'):
        if col in df:
            s = df.groupby(col)['split'].nunique()
            rep[f'{col}_across_splits'] = int((s > 1).sum())
    if '_dhash' in df:
        bits = np.stack([np.frombuffer(b, np.uint8) for b in df['_dhash']]).astype(np.int32)
        ham = bits @ (1 - bits).T + (1 - bits) @ bits.T
        sp = df['split'].values
        i, j = np.where(np.triu(ham <= dup_hamming, k=1))
        rep['neardup_pairs_across_splits'] = int(sum(sp[a] != sp[b] for a, b in zip(i, j)))
    rep['ok'] = all(v == 0 for k, v in rep.items() if k != 'ok')
    return rep


def cross_dataset_duplicates(dfA, dfB, dup_hamming=8):
    # kiểm tra ảnh trùng/gần trùng giữa hai dataset (ví dụ BUS-BRA và BUSI)
    if len(dfA) == 0 or len(dfB) == 0:
        raise ValueError(f'So trùng giữa hai dataset cần cả hai bảng khác rỗng (A={len(dfA)}, B={len(dfB)}). Bảng rỗng nghĩa là dataset đó bị loại hết ở bước audit.')
    out = dict(sha256_shared=int(len(set(dfA['sha256']) & set(dfB['sha256']))))
    A = np.stack([np.frombuffer(b, np.uint8) for b in dfA['_dhash']]).astype(np.int32)
    B = np.stack([np.frombuffer(b, np.uint8) for b in dfB['_dhash']]).astype(np.int32)
    ham = A @ (1 - B).T + (1 - A) @ B.T
    out['neardup_pairs'] = int((ham <= dup_hamming).sum())
    out['ok'] = out['sha256_shared'] == 0 and out['neardup_pairs'] == 0
    return out


def cross_dataset_duplicate_rows(dfA, dfB, dup_hamming=8):
    # mặt nạ boolean trên các dòng của dfB là bản sao/gần trùng của ảnh nào đó trong dfA (dùng để loại khỏi tập đánh giá ngoài)
    if len(dfA) == 0 or len(dfB) == 0:
        raise ValueError(f'So trùng giữa hai dataset cần cả hai bảng khác rỗng (A={len(dfA)}, B={len(dfB)}). Bảng rỗng nghĩa là dataset đó bị loại hết ở bước audit.')
    A = np.stack([np.frombuffer(b, np.uint8) for b in dfA['_dhash']]).astype(np.int32)
    B = np.stack([np.frombuffer(b, np.uint8) for b in dfB['_dhash']]).astype(np.int32)
    ham = A @ (1 - B).T + (1 - A) @ B.T
    near = (ham <= dup_hamming).any(0)
    exact = dfB['sha256'].isin(set(dfA['sha256'])).values
    return near | exact


def write_manifest(df, path):
    cols = [c for c in df.columns if not c.startswith('_')]
    d = df[cols].copy()
    for c in d.columns:
        if d[c].map(lambda v: isinstance(v, (list, tuple))).any():
            d[c] = d[c].map(lambda v: ';'.join(v) if isinstance(v, (list, tuple)) else v)
    d.to_csv(path, index=False)
    h = sha256_file(path)
    json.dump(dict(manifest=os.path.basename(path), sha256=h, n=len(d)), open(path + '.sha256.json', 'w'))
    return h


# ============================ Dữ liệu TỔNG HỢP (chỉ để kiểm thử code; KHÔNG phải dữ liệu y tế) ============================
def _synth_image(rng, malignant, size=(190, 250), seed_bg=None):
    h, w = size
    bg_rng = np.random.RandomState(seed_bg) if seed_bg is not None else rng
    img = cv2.GaussianBlur((bg_rng.rand(h, w) * 90).astype(np.float32), (0, 0), 2.0) + 50
    mask = np.zeros((h, w), np.uint8)
    cx, cy = rng.randint(60, w - 60), rng.randint(50, h - 50)
    ax, ay = rng.randint(14, 36), rng.randint(10, 26)
    cv2.ellipse(mask, (cx, cy), (ax, ay), int(rng.randint(0, 90)), 0, 360, 255, -1)
    img = img + (mask > 0) * (55 if malignant else -30) + rng.rand(h, w) * 8
    return cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR), mask


def make_synthetic_busbra(root, n_patients=30, seed=0):
    # CẤU TRÚC GIẢ ĐỊNH giống BUS-BRA (Images/, Masks/, bus_data.csv với ID, Case, Pathology). Dùng để kiểm thử mã, không đại diện dữ liệu thật.
    rng = np.random.RandomState(seed)
    os.makedirs(os.path.join(root, 'Images'), exist_ok=True); os.makedirs(os.path.join(root, 'Masks'), exist_ok=True)
    rows, k = [], 0
    for p in range(n_patients):
        mal = p % 3 == 0
        for v in range(rng.randint(1, 4)):
            k += 1
            img, m = _synth_image(rng, mal, seed_bg=p)
            name = f'bus_{k:04d}-l'
            cv2.imwrite(os.path.join(root, 'Images', name + '.png'), img); cv2.imwrite(os.path.join(root, 'Masks', name.replace('bus_', 'mask_') + '.png'), m)
            rows.append(dict(ID=name, Case=f'C{p:04d}', Pathology='malignant' if mal else 'benign', BIRADS=4 + int(mal), Device='synthetic'))
    pd.DataFrame(rows).to_csv(os.path.join(root, 'bus_data.csv'), index=False)
    open(os.path.join(root, 'SYNTHETIC_README.txt'), 'w').write('Dữ liệu tổng hợp chỉ để kiểm thử mã. KHÔNG phải ảnh y tế.')
    return root


def make_synthetic_busi(root, n=24, seed=1, with_normal=True):
    rng = np.random.RandomState(seed)
    for c, cnt in (('benign', n), ('malignant', n // 2), ('normal', 6 if with_normal else 0)):
        d = os.path.join(root, c); os.makedirs(d, exist_ok=True)
        for i in range(1, cnt + 1):
            img, m = _synth_image(rng, c == 'malignant')
            if c == 'normal':
                m[:] = 0
            cv2.imwrite(f'{d}/{c} ({i}).png', img); cv2.imwrite(f'{d}/{c} ({i})_mask.png', m)
    open(os.path.join(root, 'SYNTHETIC_README.txt'), 'w').write('Dữ liệu tổng hợp chỉ để kiểm thử mã.')
    return root

In [ ]:
%%writefile bus_core_v2.py
# bus_core_v2.py - dữ liệu, augmentation siêu âm (speckle + affine), EMA, TTA, engine huấn luyện/resume, Grad-CAM
import os, json, time, random, gc, copy
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import bus_core as core
import losses_xai as LX
from data_audit import read_rgb, read_union_mask, read_mask


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================ Nạp và tiền xử lý ============================
def load_arrays(df, img_size, mask_col='mask_path', desc=None):
    N = len(df)
    X = np.zeros((N, img_size, img_size, 3), np.uint8)
    M = np.zeros((N, img_size, img_size), np.uint8)
    for i, r in enumerate(df.itertuples(index=False)):
        img = read_rgb(r.image_path)
        assert img is not None, r.image_path
        mp = getattr(r, mask_col)
        paths = mp if isinstance(mp, (list, tuple)) else [mp]
        X[i] = core.preprocess_image(img, img_size)
        M[i] = core.preprocess_mask(read_union_mask(paths, img.shape[:2]), img_size)
    return X, M, df['label'].values.astype(np.int64)


def augment_pair(img, m, speckle=True):
    # augmentation đồng bộ ảnh–mask + mô phỏng nhiễu nhân speckle đặc trưng siêu âm (tăng chịu đựng domain shift)
    h, w = m.shape
    m0 = m
    if random.random() < 0.5:
        img, m = img[:, ::-1], m[:, ::-1]
    img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
    if random.random() < 0.8:
        A = cv2.getRotationMatrix2D((w / 2, h / 2), random.uniform(-15, 15), random.uniform(0.85, 1.15))
        A[:, 2] += (random.uniform(-0.08, 0.08) * w, random.uniform(-0.08, 0.08) * h)
        it = cv2.warpAffine(img, A, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        mt = cv2.warpAffine(m, A, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        if not (m0.sum() > 0 and mt.sum() == 0):
            img, m = it, mt
    out = img.astype(np.float32)
    if random.random() < 0.8:
        out = out * random.uniform(0.8, 1.2) + random.uniform(-20, 20)
    if random.random() < 0.3:
        out = 255.0 * (np.clip(out, 0, 255) / 255.0) ** random.uniform(0.8, 1.25)
    if speckle and random.random() < 0.25:
        # nhiễu nhân speckle + nhiễu cộng nhẹ
        out = out * (1.0 + np.random.normal(0, random.uniform(0.02, 0.06), out.shape[:2] + (1,)))
    elif random.random() < 0.3:
        out = out + np.random.normal(0, random.uniform(3, 10), out.shape)
    return np.clip(out, 0, 255).astype(np.uint8), m


class BUSDataset(Dataset):
    def __init__(self, X, M, y, indices, augment=False):
        self.X, self.M, self.y, self.indices, self.augment = X, M, y, np.asarray(indices), augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        k = int(self.indices[i])
        img, m = self.X[k], self.M[k]
        if self.augment:
            img, m = augment_pair(img, m)
        return (core.normalize_to_tensor(img), torch.from_numpy(np.ascontiguousarray(m[None].astype(np.float32))),
                torch.tensor(int(self.y[k]), dtype=torch.long), k)


def _winit(w):
    s = (torch.initial_seed() + w) % (2 ** 32)
    random.seed(s); np.random.seed(s)


def make_loader(X, M, y, indices, batch_size, augment, shuffle, num_workers=0, seed=42):
    g = torch.Generator(); g.manual_seed(seed)
    ds = BUSDataset(X, M, y, indices, augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=(shuffle and len(ds) > batch_size), num_workers=num_workers,
                      worker_init_fn=_winit, generator=g, pin_memory=torch.cuda.is_available())


# ============================ Dự đoán (chuẩn & 2-view TTA) ============================
@torch.no_grad()
def predict_arrays(model, X, indices, device, batch_size=32, amp=False, tta=False):
    model.eval()
    probs, segp, evid = [], [], []
    for i in range(0, len(indices), batch_size):
        idx = indices[i:i + batch_size]
        x = torch.stack([core.normalize_to_tensor(X[k]) for k in idx]).to(device)
        with torch.autocast(device_type=device.type, enabled=amp):
            o = model(x)
            if tta:
                of = model(x.flip(-1))
        if o.get('logits') is not None:
            p = F.softmax(o['logits'].float(), 1)
            if tta:
                p = 0.5 * (p + F.softmax(of['logits'].float(), 1))
            probs.append(p.cpu().numpy())
        if o.get('seg') is not None:
            sp = torch.sigmoid(o['seg'].float())
            if tta:
                sp = 0.5 * (sp + torch.sigmoid(of['seg'].float()).flip(-1))
            segp.append(sp.squeeze(1).cpu().numpy().astype(np.float16))
        if o.get('evidence') is not None:
            ev = o['evidence'].float()
            if tta:
                ev = 0.5 * (ev + of['evidence'].float().flip(-1))
                ev = ev / (ev.sum((2, 3), keepdim=True) + 1e-8)
            evid.append(ev.cpu().numpy()[:, 0])
    cat = lambda a: np.concatenate(a) if a else None
    return dict(idx=np.asarray(indices), prob=cat(probs), seg_prob=cat(segp), evidence=cat(evid))


# ============================ EMA helper (V3) ============================
class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def state_dict(self):
        return self.shadow


# ============================ Engine huấn luyện ============================
class TrainCfg:
    def __init__(self, epochs=40, patience=10, batch_size=16, lr_enc=1e-4, lr_new=1e-3, weight_decay=1e-4, grad_clip=1.0,
                 amp=True, num_workers=2, lr_patience=4, nan_limit=5, class_weights=True, use_ema=False, ema_decay=0.995):
        self.__dict__.update(locals()); self.__dict__.pop('self')


def _param_groups(model, cfg):
    enc, new = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (enc if (n.startswith('enc.') or n.startswith('model.features') or '.features.' in n) else new).append(p)
    gs = [{'params': new, 'lr': cfg.lr_new}] if new else []
    if enc:
        gs.append({'params': enc, 'lr': cfg.lr_enc})
    return gs


def _auroc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float('nan')


def val_metrics(pred, M, y):
    out = {}
    idx = pred['idx']
    if pred['prob'] is not None:
        out['auroc'] = _auroc(y[idx], pred['prob'][:, 1]) if pred['prob'].shape[1] == 2 else float('nan')
    if pred['seg_prob'] is not None:
        pb, gb = pred['seg_prob'].astype(np.float32) >= 0.5, M[idx] > 0
        inter = (pb & gb).sum((1, 2)); den = pb.sum((1, 2)) + gb.sum((1, 2))
        has = gb.sum((1, 2)) > 0
        out['dice'] = float((2 * inter / (den + 1e-7))[has].mean()) if has.any() else float('nan')
    return out


def selection_score(m):
    v = [m[k] for k in ('auroc', 'dice') if k in m and np.isfinite(m[k])]
    return float(np.mean(v)) if v else -1.0


def make_loss_cfg(spec):
    return LX.LossCfg(**spec.get('loss', {})) if spec['kind'] == 'xai' else LX.LossCfg()


def train_model(name, model, loss_cfg, data, cfg, out_dir, seed, device, augment=True, resume=True, meta=None, log=print):
    os.makedirs(out_dir, exist_ok=True)
    tag = f'{name}_s{seed}'
    best_p, last_p, hist_p = (os.path.join(out_dir, f'{tag}_{k}') for k in ('best.pt', 'last.pt', 'history.json'))
    seed_everything(seed)
    X, M, y = data['X'], data['M'], data['y']
    use_cons = loss_cfg.lam_cons_pred > 0 or loss_cfg.lam_cons_ev > 0
    use_faith = getattr(loss_cfg, 'lam_faith', 0.0) > 0
    tl = make_loader(X, M, y, data['train'], cfg.batch_size, augment, True, cfg.num_workers, seed)
    model = model.to(device)
    ema = ModelEMA(model, cfg.ema_decay) if getattr(cfg, 'use_ema', False) else None
    cw = None
    if cfg.class_weights:
        cnt = np.bincount(y[data['train']], minlength=int(y.max()) + 1).astype(float)
        cw = torch.tensor(len(data['train']) / (len(cnt) * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(_param_groups(model, cfg), weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=cfg.lr_patience)
    use_amp = cfg.amp and device.type == 'cuda'
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ep0, best, best_ep, bad, hist, done = 0, -1e9, -1, 0, [], False
    if resume and os.path.exists(last_p):
        ck = core.torch_load(last_p, device)
        if not ck.get('compact'):
            model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); scaler.load_state_dict(ck['scaler'])
            if ema is not None and 'ema' in ck:
                ema.shadow = ck['ema']
        ep0, best, best_ep, bad, hist, done = ck['epoch'] + 1, ck['best'], ck['best_ep'], ck['bad'], ck['hist'], ck['done']
        log(f'[{tag}] resume từ epoch {ep0}' + (' (đã hoàn tất)' if done else ''))
    for ep in range(ep0, cfg.epochs):
        if done:
            break
        model.train()
        tot, n, skipped, parts_sum = 0.0, 0, 0, {}
        t0 = time.time()
        for xb, mb, yb, _ in tl:
            xb, mb, yb = xb.to(device, non_blocking=True), mb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out_aug = theta = logits_hi = logits_lo = None
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(xb)
                if use_cons:
                    theta = LX.sample_affine(xb.shape[0], device)
                    out_aug = model(LX.apply_label_preserving(xb, theta))
                if use_faith and out.get('evidence') is not None and hasattr(model, 'forward_heads_from_Z'):
                    E = out['evidence']
                    q = getattr(loss_cfg, 'faith_q', 0.2)
                    thr_hi = torch.quantile(E.flatten(1), 1.0 - q, dim=1, keepdim=True).view(-1, 1, 1, 1)
                    thr_lo = torch.quantile(E.flatten(1), q, dim=1, keepdim=True).view(-1, 1, 1, 1)
                    keep_hi = (E < thr_hi).float()
                    keep_lo = (E > thr_lo).float()
                    logits_hi, _, _ = model.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=keep_hi)
                    logits_lo, _, _ = model.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=keep_lo)
            loss, parts = LX.total_loss(out, mb, yb, loss_cfg, cw, out_aug, theta, logits_hi, logits_lo)
            if not torch.isfinite(loss):
                skipped += 1
                if skipped > cfg.nan_limit:
                    raise FloatingPointError('Quá nhiều batch NaN/Inf')
                continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
            if ema is not None:
                ema.update(model)
            tot += float(loss.detach()) * xb.size(0); n += xb.size(0)
            for k, v in parts.items():
                parts_sum[k] = parts_sum.get(k, 0.0) + v * xb.size(0)
        if n == 0:
            raise RuntimeError('Epoch không có batch hợp lệ')
        if ema is not None:
            cur_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            model.load_state_dict(ema.state_dict())
            vp = predict_arrays(model, X, data['val'], device, amp=use_amp)
            eval_state = {k: v.detach().clone() for k, v in ema.state_dict().items()}
            model.load_state_dict(cur_state)
        else:
            vp = predict_arrays(model, X, data['val'], device, amp=use_amp)
            eval_state = model.state_dict()
        vm = val_metrics(vp, M, y)
        score = selection_score(vm)
        sch.step(score)
        imp = score > best + 1e-4
        if imp:
            best, best_ep, bad = score, ep, 0
            torch.save(dict(model=eval_state, meta=dict(meta or {}, name=name, seed=seed, best_score=best, best_epoch=ep + 1)), best_p)
        else:
            bad += 1
        done = bad >= cfg.patience or ep == cfg.epochs - 1
        rec = dict(epoch=ep + 1, train_loss=tot / n, skipped=skipped, score=float(score), time_s=time.time() - t0,
                   **{f'loss_{k}': v / n for k, v in parts_sum.items()}, **{f'val_{k}': float(v) for k, v in vm.items()})
        hist.append(rec)
        if done:
            torch.save(dict(compact=True, epoch=ep, best=best, best_ep=best_ep, bad=bad, hist=hist, done=True), last_p)
        else:
            ck_last = dict(model=model.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(), scaler=scaler.state_dict(), epoch=ep, best=best,
                           best_ep=best_ep, bad=bad, hist=hist, done=False)
            if ema is not None:
                ck_last['ema'] = ema.state_dict()
            torch.save(ck_last, last_p)
        json.dump(hist, open(hist_p, 'w'))
        log(f'[{tag}] ep {ep + 1:3d}/{cfg.epochs} loss {rec["train_loss"]:.4f} ' + ' '.join(f'{k}={v:.3f}' for k, v in vm.items()) + f' score {score:.4f}' + (' *best*' if imp else ''))
        if bad >= cfg.patience:
            log(f'[{tag}] early stopping'); break
    del opt, tl
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return dict(name=name, seed=seed, best_path=best_p, last_path=last_p, history_path=hist_p, best_score=best, best_epoch=best_ep + 1)


def train_with_oom_fallback(*args, cfg=None, **kw):
    bs = cfg.batch_size
    while True:
        oom = False
        try:
            cfg.batch_size = bs
            return train_model(*args, cfg=cfg, **kw)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and bs > 2:
                oom = True
            else:
                raise
        if oom:
            gc.collect(); torch.cuda.empty_cache(); bs //= 2
            print('!! hết bộ nhớ GPU -> batch', bs)


# ============================ Checkpoint ============================
def save_model_checkpoint(path, model, meta):
    torch.save(dict(model=model.state_dict(), meta=meta), path)


def load_model_from_checkpoint(path, device, pretrained=False):
    from models_xai import build_experiment
    ck = core.torch_load(path, device)
    meta = ck['meta']
    model, _ = build_experiment(meta['experiment'], num_classes=meta.get('num_classes', 2), pretrained=pretrained,
                                dim=meta.get('dim', 128), base=meta.get('base', 32))
    model.load_state_dict(ck['model'])
    return model.to(device).eval(), meta


# ============================ Grad-CAM cho mô hình trả dict ============================
def gradcam_dict(model, x, class_idx=None, layer=None):
    layer = layer if layer is not None else (model.neck if hasattr(model, 'neck') else model.model.features[-1])
    store = {}

    def hook(m, i, o):
        store['a'] = o
        o.register_hook(lambda g: store.__setitem__('g', g))
    h = layer.register_forward_hook(hook)
    try:
        model.eval()
        with torch.enable_grad():
            xin = x.detach().clone().requires_grad_(True)
            lg = model(xin)['logits']
            idx = int(lg.argmax(1)) if class_idx is None else int(class_idx)
            model.zero_grad(set_to_none=True)
            lg[0, idx].backward()
        w = store['g'].detach().mean((2, 3), keepdim=True)
        cam = F.relu((w * store['a'].detach()).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), idx
    finally:
        h.remove()


# ============================ Phân tích một ảnh (dùng cho ứng dụng) ============================
def analyze_image_v2(rgb, model, dcfg, device):
    import bus_trust as T
    size = int(dcfg['img_size'])
    img = core.preprocess_image(rgb, size)
    x = core.normalize_to_tensor(img).unsqueeze(0).to(device)
    model.eval()
    use_tta = bool(dcfg.get('tta', False))
    with torch.no_grad():
        o = model(x)
        if use_tta:
            of = model(x.flip(-1))
            lg = 0.5 * (F.softmax(o['logits'].float(), 1) + F.softmax(of['logits'].float(), 1))
            raw = lg[0].cpu().numpy()
            sp_t = 0.5 * (torch.sigmoid(o['seg'].float()) + torch.sigmoid(of['seg'].float()).flip(-1))
        else:
            raw = F.softmax(o['logits'].float(), 1)[0].cpu().numpy()
            sp_t = torch.sigmoid(o['seg'].float())
    cal = T.softmax(np.log(np.clip(raw, 1e-12, 1)), float(dcfg['temperature']))
    p_mal = float(cal[1])
    conf = float(cal.max())
    res = dict(image=img, p_raw=raw, p_cal=cal, p_malignant=p_mal, label='malignant' if p_mal >= float(dcfg['threshold']) else 'benign',
               confidence=conf, abstain=bool(conf < float(dcfg['abstain_threshold'])), evidence=None, cam=None)
    sp = sp_t[0, 0].cpu().numpy()
    res['seg_prob'] = sp
    res['mask'] = (sp >= float(dcfg['seg_threshold'])).astype(np.uint8)
    res['boxes'] = core.mask_to_boxes(res['mask'], int(dcfg['min_area']))
    if o.get('evidence') is not None:
        E = o['evidence'].float()
        if use_tta:
            E = 0.5 * (E + of['evidence'].float().flip(-1))
        res['evidence'] = F.interpolate(E / (E.sum() + 1e-12), size=(size, size), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
    res['cam'], _ = gradcam_dict(model, x, class_idx=int(np.argmax(cal)))
    return res


In [ ]:
%%writefile evaluation_xai.py
# evaluation_xai.py - đánh giá phân loại/phân vùng, XAI (định vị, trung thực, nhất quán), hiệu chuẩn, chọn lọc dự đoán, thống kê, khóa pipeline
import os, json, time, hashlib
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from scipy import ndimage

import bus_trust as T
import losses_xai as LX
import bus_core as core
from data_audit import sha256_file


# ============================ Phân loại (lớp 1 = malignant) ============================
def classification_metrics(y, p_mal, threshold=0.5):
    from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
    y, p_mal = np.asarray(y), np.asarray(p_mal, float)
    pred = (p_mal >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else float('nan')
    spec = tn / (tn + fp) if tn + fp else float('nan')
    two = len(np.unique(y)) == 2
    return dict(auroc=float(roc_auc_score(y, p_mal)) if two else float('nan'), auprc=float(average_precision_score(y, p_mal)) if two else float('nan'),
                sensitivity_malignant=float(sens), specificity=float(spec), balanced_accuracy=float(np.nanmean([sens, spec])),
                f1=float(f1_score(y, pred, zero_division=0)), accuracy=float((pred == y).mean()), threshold=float(threshold))


def select_threshold_youden(y, p_mal):
    # CHỈ gọi trên validation của BUS-BRA
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, p_mal)
    return float(thr[int(np.argmax(tpr - fpr))])


# ============================ Phân vùng ============================
def hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return float('nan')
    ps = pred ^ ndimage.binary_erosion(pred); gs = gt ^ ndimage.binary_erosion(gt)
    d_to_gt, d_to_pred = ndimage.distance_transform_edt(~gt), ndimage.distance_transform_edt(~pred)
    return float(np.percentile(np.concatenate([d_to_gt[ps], d_to_pred[gs]]), 95))


def seg_metrics(seg_prob, gt, thr=0.5, with_hd95=True):
    pb, gb = seg_prob.astype(np.float32) >= thr, gt > 0
    inter = (pb & gb).sum((1, 2)).astype(float); ps, gs = pb.sum((1, 2)).astype(float), gb.sum((1, 2)).astype(float)
    has = gs > 0
    out = dict(dice=2 * inter / (ps + gs + 1e-7), iou=inter / (ps + gs - inter + 1e-7), has_lesion=has)
    out['hd95'] = np.array([hd95(p, g) for p, g in zip(pb, gb)]) if with_hd95 else None
    hd = out['hd95'][has] if with_hd95 else np.array([])
    ok_hd = np.isfinite(hd)
    out['summary'] = dict(dice=float(out['dice'][has].mean()), iou=float(out['iou'][has].mean()),
                          hd95=float(hd[ok_hd].mean()) if ok_hd.any() else float('nan'),
                          n_hd95_undefined=int((~ok_hd).sum()) if with_hd95 else 0)          # HD95 không xác định khi mask dự đoán rỗng
    return out


# ============================ XAI: định vị ============================
def upsample_evidence(E, size):
    E = torch.as_tensor(E, dtype=torch.float32)
    if E.ndim == 3:
        E = E[:, None]
    U = F.interpolate(E, size=size, mode='bilinear', align_corners=False).clamp(min=0)
    return (U / (U.sum((1, 2, 3), keepdim=True) + 1e-12))[:, 0].numpy()


def localisation_report(E, gt, ring_px=8, min_area=1e-3, max_area=0.9):
    # Relevance Mass (trong tổn thương và trong tổn thương+vòng), Pointing Game, area-adjusted attribution.
    # Đây là đo ĐỊNH VỊ so với mask; KHÔNG chứng minh tính trung thực (faithfulness).
    U = upsample_evidence(E, gt.shape[-2:])
    rm, rmc, pg, aaa, aaac = [], [], [], [], []
    k = np.ones((2 * ring_px + 1, 2 * ring_px + 1), np.uint8)
    for u, g in zip(U, gt):
        g = (g > 0).astype(np.uint8)
        a = g.mean()
        if not (min_area < a < max_area):
            continue
        roi = cv2.dilate(g, k)
        mass_l, mass_r, a_r = float((u * g).sum()), float((u * roi).sum()), float(roi.mean())
        rm.append(mass_l); rmc.append(mass_r)
        aaa.append((mass_l - a) / (1 - a)); aaac.append((mass_r - a_r) / (1 - a_r) if a_r < 1 else float('nan'))
        pg.append(float(roi[np.unravel_index(np.argmax(u), u.shape)] > 0))
    f = lambda v: float(np.nanmean(v)) if len(v) else float('nan')
    return dict(n=len(rm), relevance_mass_lesion=f(rm), relevance_mass_lesion_context=f(rmc), pointing_game=f(pg),
                area_adjusted_lesion=f(aaa), area_adjusted_lesion_context=f(aaac))


# ============================ XAI: trung thực (faithfulness) ============================
def _auc(q, v):
    q, v = np.asarray(q, float), np.asarray(v, float)
    return float(np.sum((q[1:] - q[:-1]) * (v[1:] + v[:-1]) / 2))


@torch.no_grad()
def _prob_of(model, x, cls):
    return float(F.softmax(model(x)['logits'].float(), 1)[0, cls])


@torch.no_grad()
def deletion_insertion_curves(model, x, rank_map, cls, steps=10):
    # rank_map (h,w): điểm quan trọng; xoá/chèn dần các ô quan trọng nhất; baseline = 0 (ảnh trung bình sau chuẩn hóa)
    h, w = rank_map.shape
    order = torch.as_tensor(rank_map).flatten().argsort(descending=True)
    base = torch.zeros_like(x)
    dele, ins = [], []
    for s in range(steps + 1):
        k = int(round(h * w * s / steps))
        cm = torch.zeros(h * w); cm[order[:k]] = 1
        pm = F.interpolate(cm.view(1, 1, h, w).to(x.device), size=x.shape[-2:], mode='nearest')
        dele.append(_prob_of(model, x * (1 - pm) + base * pm, cls))
        ins.append(_prob_of(model, base * (1 - pm) + x * pm, cls))
    q = np.linspace(0, 1, steps + 1)
    return q, np.array(dele), np.array(ins)


def faithfulness_report(model, X, indices, device, steps=8, max_images=40, seed=0, use_cam=True):
    # So sánh xếp hạng theo evidence / Grad-CAM với xếp hạng NGẪU NHIÊN. Deletion AUC thấp hơn và Insertion AUC cao hơn ngẫu nhiên
    # là bằng chứng (chưa phải chứng minh) rằng vùng được xếp hạng ảnh hưởng thật tới dự đoán.
    from bus_core_v2 import gradcam_dict
    rng = np.random.RandomState(seed)
    model.eval()
    rows = []
    for k in indices[:max_images]:
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        with torch.no_grad():
            o = model(x)
        cls = int(o['logits'].argmax(1))
        h, w = (o['evidence'].shape[-2:] if o.get('evidence') is not None else (16, 16))
        maps = {'random': rng.rand(h, w)}
        if o.get('evidence') is not None:
            maps['evidence'] = o['evidence'][0, 0].float().cpu().numpy()
        if use_cam:
            cam, _ = gradcam_dict(model, x, class_idx=cls)
            maps['gradcam'] = cv2.resize(cam, (w, h), interpolation=cv2.INTER_AREA)
        row = {}
        for nm, mp in maps.items():
            q, d, i = deletion_insertion_curves(model, x, mp, cls, steps)
            row[f'deletion_auc_{nm}'], row[f'insertion_auc_{nm}'] = _auc(q, d), _auc(q, i)
        rows.append(row)
    keys = rows[0].keys() if rows else []
    return {k: float(np.mean([r[k] for r in rows])) for k in keys} | {'n': len(rows)}


@torch.no_grad()
def evidence_intervention(model, X, indices, device, q=0.2, max_images=40, seed=0):
    # Can thiệp trực tiếp vào evidence: loại bỏ q% ô có evidence lớn nhất (ev_mask) và đo giảm xác suất lớp dự đoán, so với loại ô ngẫu nhiên cùng số lượng.
    if getattr(getattr(model, 'cfg', None), 'evidence_mode', 'none') not in ('ewp', 'egr'):
        return dict(applicable=False)
    g = torch.Generator().manual_seed(seed)
    model.eval()
    d_top, d_rand = [], []
    for k in indices[:max_images]:
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        o = model(x)
        cls = int(o['logits'].argmax(1)); p0 = float(F.softmax(o['logits'].float(), 1)[0, cls])
        E = o['evidence']; n = E.numel(); kk = max(1, int(q * n))
        top = E.flatten().topk(kk).indices.cpu()
        rnd = torch.randperm(n, generator=g)[:kk]
        for idxs, store in ((top, d_top), (rnd, d_rand)):
            mk = torch.ones(n); mk[idxs] = 0
            p1 = float(F.softmax(model(x, ev_mask=mk.view_as(E).to(device))['logits'].float(), 1)[0, cls])
            store.append(p0 - p1)
    return dict(applicable=True, q=q, n=len(d_top), mean_drop_top_evidence=float(np.mean(d_top)), mean_drop_random=float(np.mean(d_rand)))


# ============================ Phát hiện evidence collapse và kiểm tra nhiễu nền ============================
def evidence_collapse_report(E, uniform_entropy=0.98, ess_uniform=0.9, ess_peaked_cells=2.0):
    E = torch.as_tensor(E, dtype=torch.float32)
    if E.ndim == 4:
        E = E[:, 0]
    N, h, w = E.shape
    P = E.flatten(1).clamp(min=0)
    P = P / (P.sum(1, keepdim=True) + 1e-12)
    H = -(P * torch.log(P + 1e-12)).sum(1) / np.log(h * w)           # entropy chuẩn hóa: 1 = đồng đều
    ess = 1.0 / ((P ** 2).sum(1) + 1e-12)                            # số ô hiệu dụng
    uniform = (H > uniform_entropy) & (ess / (h * w) > ess_uniform)
    peaked = ess < ess_peaked_cells
    return dict(n=N, mean_norm_entropy=float(H.mean()), mean_ess_fraction=float((ess / (h * w)).mean()), uniform_collapse_rate=float(uniform.float().mean()),
                peaked_collapse_rate=float(peaked.float().mean()), collapse_flag=bool(uniform.float().mean() > 0.5 or peaked.float().mean() > 0.5))


@torch.no_grad()
def background_perturbation(model, X, M, indices, device, ring_px=8, max_images=30, seed=0):
    # Xáo trộn pixel NGOÀI vùng (tổn thương + vòng) -> kỳ vọng dự đoán/evidence ít đổi nếu mô hình dựa vào tổn thương.
    # Xáo trộn pixel TRONG tổn thương (đối chứng) -> kỳ vọng thay đổi lớn hơn. Đây là kiểm tra mô tả, không phải chứng minh.
    rng = np.random.RandomState(seed)
    kern = np.ones((2 * ring_px + 1, 2 * ring_px + 1), np.uint8)
    model.eval()
    d_bg, d_les, mass_b, mass_a = [], [], [], []
    for k in indices[:max_images]:
        g = (M[k] > 0).astype(np.uint8)
        if not (0.001 < g.mean() < 0.9):
            continue
        roi = cv2.dilate(g, kern).astype(bool)
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        o = model(x)
        cls = int(o['logits'].argmax(1)); p0 = float(F.softmax(o['logits'].float(), 1)[0, cls])

        def shuffled(region):
            xs = x.clone()
            ii = np.flatnonzero(region.ravel())
            perm = rng.permutation(ii)
            flat = x.view(1, 3, -1)
            xs.view(1, 3, -1)[:, :, torch.as_tensor(ii)] = flat[:, :, torch.as_tensor(perm)]
            return xs
        ob = model(shuffled(~roi)); ol = model(shuffled(g.astype(bool)))
        d_bg.append(abs(p0 - float(F.softmax(ob['logits'].float(), 1)[0, cls]))); d_les.append(abs(p0 - float(F.softmax(ol['logits'].float(), 1)[0, cls])))
        if o.get('evidence') is not None:
            mk = F.interpolate(torch.as_tensor(roi[None, None].astype(np.float32)), size=o['evidence'].shape[-2:], mode='nearest').to(device)
            En = LX.normalize_evidence(o['evidence'].float()); Eb = LX.normalize_evidence(ob['evidence'].float())
            mass_b.append(float((En * mk).sum())); mass_a.append(float((Eb * mk).sum()))
    f = lambda v: float(np.mean(v)) if len(v) else float('nan')
    return dict(n=len(d_bg), mean_abs_dprob_background_shuffle=f(d_bg), mean_abs_dprob_lesion_shuffle=f(d_les),
                evidence_mass_in_roi_before=f(mass_b), evidence_mass_in_roi_after_background_shuffle=f(mass_a))


# ============================ XAI: nhất quán ============================
@torch.no_grad()
def explanation_consistency_eval(model, X, indices, device, batch_size=16, seed=0, hflip_only=False):
    # prediction consistency và evidence consistency được TÁCH RIÊNG; evidence được inverse-warp và chỉ tính trên pixel hợp lệ
    g = torch.Generator(device=device).manual_seed(seed)
    model.eval()
    agree, dprob, cos_all = [], [], []
    for i in range(0, len(indices), batch_size):
        idx = indices[i:i + batch_size]
        x = torch.stack([core.normalize_to_tensor(X[k]) for k in idx]).to(device)
        theta = LX.hflip_theta(len(idx), device) if hflip_only else LX.sample_affine(len(idx), device, generator=g)
        o1, o2 = model(x), model(LX.apply_label_preserving(x, theta))
        p1, p2 = F.softmax(o1['logits'].float(), 1), F.softmax(o2['logits'].float(), 1)
        agree += (p1.argmax(1) == p2.argmax(1)).float().cpu().tolist()
        dprob += (p1[:, 1] - p2[:, 1]).abs().cpu().tolist()
        if o1.get('evidence') is not None:
            cos, ok, _, _ = LX.evidence_alignment(o1['evidence'], o2['evidence'], theta)
            cos_all += cos[ok > 0].cpu().tolist()
    return dict(prediction_agreement=float(np.mean(agree)), mean_abs_dprob_malignant=float(np.mean(dprob)),
                evidence_cosine_after_inverse_warp=float(np.mean(cos_all)) if cos_all else float('nan'), n=len(agree))


# ============================ Hiệu chuẩn, chọn lọc dự đoán ============================
def calibration_report(prob_val, y_val, prob_test, y_test):
    # T chọn trên validation, áp dụng cho test (hoặc cho tập ngoài)
    lg_v, lg_t = np.log(np.clip(prob_val, 1e-12, 1)), np.log(np.clip(prob_test, 1e-12, 1))
    Tm = T.fit_temperature(lg_v, y_val)
    rows = []
    for tag, tt in (('trước', 1.0), ('sau', Tm)):
        p = T.softmax(lg_t, tt)
        rows.append(dict(hiệu_chuẩn=tag, T=tt, NLL=T.nll(p, y_test), Brier=T.brier(p, y_test), ECE=T.ece(p, y_test)))
    return Tm, rows


def selective_report(prob_val, prob_test, y_test, coverages=(0.9, 0.8, 0.7), malignant=1):
    # độ tin cậy = max xác suất; ngưỡng chọn trên validation theo coverage mục tiêu
    cv, ct = prob_val.max(1), prob_test.max(1)
    pred = prob_test.argmax(1)
    wrong = pred != y_test
    cov, risk, aurc = T.risk_coverage(-ct, wrong)
    out = dict(aurc=float(aurc), rows=[])
    for c in coverages:
        thr = float(np.quantile(cv, 1 - c))
        acc = ct >= thr
        mal = y_test == malignant
        out['rows'].append(dict(coverage_target=c, coverage_actual=float(acc.mean()), selective_risk=float(wrong[acc].mean()) if acc.any() else float('nan'),
                                abstain_threshold=thr, sensitivity_malignant_accepted=float((pred[acc & mal] == malignant).mean()) if (acc & mal).any() else float('nan'),
                                malignant_abstained=float((~acc[mal]).mean()) if mal.any() else float('nan')))
    return out


# ============================ Thống kê ============================
def bootstrap_ci(fn, n, B=1000, seed=0):
    rng = np.random.RandomState(seed); v = []
    for _ in range(B):
        x = fn(rng.randint(0, n, n))
        if np.isfinite(x):
            v.append(x)
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) > 20 else (float('nan'), float('nan'))


def paired_bootstrap_diff(fa, fb, n, B=1000, seed=0):
    rng = np.random.RandomState(seed); d = []
    for _ in range(B):
        i = rng.randint(0, n, n)
        x = fa(i) - fb(i)
        if np.isfinite(x):
            d.append(x)
    return (float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))) if len(d) > 20 else (float('nan'),) * 3


def summarize_seeds(values):
    v = np.asarray([x for x in values if np.isfinite(x)], float)
    return dict(n_seeds=int(len(v)), mean=float(v.mean()) if len(v) else float('nan'), sd=float(v.std(ddof=1)) if len(v) > 1 else float('nan'), values=v.tolist())


# ============================ Khóa pipeline và đánh giá ngoài ============================
class LockError(RuntimeError):
    pass


class ExternalEvalLock:
    # Trình tự: (1) chọn mô hình/ngưỡng/temperature CHỈ trên BUS-BRA -> lock(); (2) evaluate_external() kiểm tra khóa rồi mới đọc BUSI;
    # (3) sau khi đánh giá, khóa bị đánh dấu 'đã dùng' và KHÔNG cho khóa lại hoặc đánh giá lại (một lần).
    def __init__(self, path):
        self.path = path

    def read(self):
        if not os.path.exists(self.path):
            raise LockError('Chưa có pipeline lock: không được truy cập dữ liệu ngoài (BUSI).')
        return json.load(open(self.path, encoding='utf-8'))

    def lock(self, ckpt_path, config, temperature, threshold, abstain_threshold, notes=''):
        if os.path.exists(self.path) and self.read().get('external_evaluated'):
            raise LockError('Pipeline đã được dùng để đánh giá ngoài; không được khóa lại.')
        cfg_sha = hashlib.sha256(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()
        json.dump(dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), ckpt=os.path.basename(ckpt_path), ckpt_sha256=sha256_file(ckpt_path), config=config,
                       config_sha256=cfg_sha, temperature=float(temperature), threshold=float(threshold), abstain_threshold=float(abstain_threshold),
                       notes=notes, external_evaluated=False), open(self.path, 'w', encoding='utf-8'), indent=1, default=str)

    def verify(self, ckpt_path):
        L = self.read()
        if sha256_file(ckpt_path) != L['ckpt_sha256']:
            raise LockError('Checkpoint đã thay đổi sau khi khóa (sha256 không khớp).')
        return L

    def guard_access(self, ckpt_path):
        L = self.verify(ckpt_path)
        if L.get('external_evaluated'):
            raise LockError('Đánh giá ngoài chỉ được thực hiện MỘT lần với pipeline đã khóa.')
        return L

    def mark_evaluated(self, summary):
        L = self.read()
        L['external_evaluated'] = True; L['external_summary'] = summary; L['evaluated_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        json.dump(L, open(self.path, 'w', encoding='utf-8'), indent=1, default=str)


def evaluate_external(lock, ckpt_path, load_external_fn, predict_fn):
    # load_external_fn(): đọc BUSI (chỉ được gọi SAU khi qua guard). predict_fn(model_data) -> dict(prob, ...)
    L = lock.guard_access(ckpt_path)
    data = load_external_fn()
    out = predict_fn(data, L)
    lock.mark_evaluated({k: v for k, v in out.items() if isinstance(v, (int, float, str, dict, list))})
    return out

In [ ]:
%%writefile tests_xai.py
# tests_xai.py - kiểm thử đơn vị cho BreastVision-XAI V2. Chạy: python tests_xai.py  (ghi test_results.json)
import os, sys, json, time, tempfile, shutil, inspect, traceback
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bus_core as core
import losses_xai as LX
import models_xai as MX
import data_audit as DA
import bus_core_v2 as C2
import evaluation_xai as EV

DEV = torch.device('cpu')
IMG = 64
TMP = tempfile.mkdtemp(prefix='bvxai_')
RESULTS = []
_cache = {}


def test(fn):
    fn._is_test = True
    return fn


def synth_arrays():
    if 'arrays' not in _cache:
        root = DA.make_synthetic_busbra(os.path.join(TMP, 'busbra'), n_patients=30, seed=0)
        df, info = DA.index_busbra(root)
        df, rep = DA.audit_dataframe(df)
        df, _ = DA.drop_invalid(df)
        df = DA.grouped_split(df, seed=0)
        X, M, y = C2.load_arrays(df, IMG)
        _cache['arrays'] = (df, X, M, y, rep, info)
    return _cache['arrays']


def tiny_model(name='X6'):
    torch.manual_seed(0)
    m, cfg = MX.build_experiment(name, pretrained=False)
    return m.to(DEV)


# ------------------------------------------------------------------ dataset
@test
def test_dataset_pairing_and_audit():
    df, X, M, y, rep, info = synth_arrays()
    assert info['n_unpaired'] == 0 and len(df) > 20
    assert X.shape == (len(df), IMG, IMG, 3) and M.shape == (len(df), IMG, IMG) and set(np.unique(M)) <= {0, 1}
    assert (M.sum((1, 2)) > 0).all(), 'mọi ảnh BUS-BRA giả lập phải có mask khác rỗng'
    # mask phải trùng vị trí khối sáng/tối: tâm khối u của ảnh malignant sáng hơn nền trong vùng mask
    mal = np.where(y == 1)[0][:5]
    for k in mal:
        g = X[k].mean(-1)
        assert g[M[k] > 0].mean() > g[M[k] == 0].mean(), 'mask không khớp ảnh sau letterbox'
    assert rep['n_label_conflict_groups'] == 0
    return f"{len(df)} ảnh, {df['patient_id'].nunique()} bệnh nhân, ghép ảnh–mask 100%"


@test
def test_augmentation_sync():
    img = np.zeros((IMG, IMG, 3), np.uint8); m = np.zeros((IMG, IMG), np.uint8)
    img[20:44, 18:46] = 255; m[20:44, 18:46] = 1
    import random
    random.seed(0); np.random.seed(0)
    ious = []
    for _ in range(100):
        ia, ma = C2.augment_pair(img.copy(), m.copy())
        a = ia.mean(-1) > 110
        ious.append((a & (ma > 0)).sum() / max((a | (ma > 0)).sum(), 1))
    assert np.mean(ious) > 0.8, np.mean(ious)
    return f'IoU trung bình giữa vùng sáng và mask sau 100 lần augment = {np.mean(ious):.3f}'


# ------------------------------------------------------------------ phân vùng
@test
def test_segmentation_output_and_metrics():
    m = tiny_model('X0').eval()
    x = torch.randn(2, 3, IMG, IMG)
    o = m(x)
    assert o['seg'].shape == (2, 1, IMG, IMG) and torch.isfinite(o['seg']).all()
    gt = (torch.rand(2, 1, IMG, IMG) > 0.8).float()
    assert torch.isfinite(LX.seg_loss(o['seg'], gt))
    g = (np.random.rand(3, IMG, IMG) > 0.7).astype(np.uint8)
    sm = EV.seg_metrics(g.astype(np.float32), g)
    assert abs(sm['summary']['dice'] - 1) < 1e-6 and sm['summary']['hd95'] == 0
    return 'seg (B,1,H,W) hữu hạn; Dice(GT,GT)=1; HD95(GT,GT)=0'


# ------------------------------------------------------------------ gộp đa vùng
@test
def test_multi_region_pooling():
    Z = torch.randn(2, 8, 10, 10)
    m = torch.zeros(2, 1, 10, 10); m[0, :, 3:6, 3:7] = 1
    p0 = MX.masked_avg_pool(Z, m, eps=0.0)
    manual = Z[0, :, 3:6, 3:7].mean((1, 2))
    assert torch.allclose(p0[0], manual, atol=1e-5), 'masked pooling sai'
    p = MX.masked_avg_pool(Z, m)                              # eps mặc định
    assert torch.isfinite(p).all() and p[1].abs().max() < 1e-6, 'mask rỗng phải cho vector hữu hạn ~0'
    ring = LX.lesion_ring(m, 2)
    assert (ring * m).sum() == 0, 'vòng không được chồng lên tổn thương'
    assert ((LX.dilate(m, 2) - m).clamp(0, 1) == ring).all()
    assert ring[0].sum() > 0 and ring[1].sum() == 0
    return 'masked pooling khớp tính tay; mask rỗng hữu hạn; ring ∩ lesion = ∅'


@test
def test_fusion_modes_shapes():
    for mode in ('concat', 'weighted', 'gated'):
        f = MX.RegionFusion(mode, 3, 16)
        out = f([torch.randn(4, 16) for _ in range(3)])
        assert out.shape == (4, f.out_dim) and torch.isfinite(out).all()
    return 'concat / weighted / gated cho đầu ra đúng kích thước'


# ------------------------------------------------------------------ evidence–logit coupling
@test
def test_evidence_logit_coupling():
    torch.manual_seed(1)
    m = MX.BreastVisionXAI(MX.XAIConfig(pretrained=False, evidence_mode='ewp', fusion='concat')).eval()
    x = torch.randn(2, 3, IMG, IMG)
    C, rest, lg = m.decompose_logits(x)
    err = (C.sum((2, 3)) + rest - lg).abs().max().item()
    assert err < 1e-4, err
    # can thiệp: xoá ô -> logit đổi đúng bằng phần đóng góp bị loại (trừ hằng số bias của chiếu)
    o = m(x); E = o['evidence']
    mk = torch.ones_like(E); mk[:, :, :3, :3] = 0
    lg2 = m(x, ev_mask=mk)['logits']
    D = m.cfg.dim; W = m.head[1].weight[:, :D]; A = m.fusion.proj[0].weight
    pred_delta = -(torch.einsum('kd,bdhw->bkhw', W @ A, o['Z']) * E * (1 - mk)).sum((2, 3))
    assert torch.allclose(lg2 - lg, pred_delta, atol=1e-4), (lg2 - lg - pred_delta).abs().max()
    # evidence thật sự ảnh hưởng logit; đối chứng decoupled thì không
    m_dec = MX.BreastVisionXAI(MX.XAIConfig(pretrained=False, evidence_mode='decoupled')).eval()
    l1 = m_dec(x)['logits']
    for p in m_dec.ev_head.parameters():
        p.data.add_(torch.randn_like(p))
    assert torch.allclose(l1, m_dec(x)['logits'], atol=1e-6), 'decoupled: evidence không được ảnh hưởng logit'
    l0 = m(x)['logits']
    for p in m.ev_head.parameters():
        p.data.add_(torch.randn_like(p))
    assert (l0 - m(x)['logits']).abs().max() > 1e-4, 'ewp: đổi evidence head phải đổi logit'
    return f'phân rã logit chính xác (sai số {err:.1e}); can thiệp ev_mask khớp dự đoán; decoupled không ảnh hưởng logit'


@test
def test_evidence_collapse_detection():
    N, h, w = 4, 16, 16
    uni = torch.ones(N, h, w) / (h * w)
    peak = torch.zeros(N, h * w); peak[:, 5] = 1; peak = peak.view(N, h, w)
    good = torch.zeros(N, h, w); good[:, 4:9, 4:9] = 1; good = good / good.sum((1, 2), keepdim=True)
    assert EV.evidence_collapse_report(uni)['collapse_flag'] and EV.evidence_collapse_report(uni)['uniform_collapse_rate'] == 1
    assert EV.evidence_collapse_report(peak)['peaked_collapse_rate'] == 1
    assert not EV.evidence_collapse_report(good)['collapse_flag']
    return 'phát hiện đúng collapse đồng đều, collapse một điểm; không báo nhầm map tập trung'


# ------------------------------------------------------------------ explanation loss
@test
def test_explanation_loss():
    B, h = 2, 16
    gt = torch.zeros(B, 1, 64, 64); gt[:, :, 20:36, 20:40] = 1
    uni = torch.ones(B, 1, h, h)
    m = LX.downsample_mask(gt, (h, h)); ideal = m / m.sum((1, 2, 3), keepdim=True)
    for mode in ('mask_only', 'lesion_context', 'density'):
        lu, d = LX.explanation_loss(uni, gt, mode)
        li, _ = LX.explanation_loss(ideal, gt, mode)
        assert lu > 0.05, (mode, float(lu)); assert li < 1e-4, (mode, float(li))
    # mask rỗng -> bị bỏ qua, không NaN
    le, d = LX.explanation_loss(uni, torch.zeros_like(gt), 'lesion_context')
    assert float(le) == 0 and d['valid'].sum() == 0
    # gradient
    E = torch.rand(B, 1, h, h, requires_grad=True)
    for mode in ('mask_only', 'lesion_context', 'density'):
        E.grad = None
        LX.explanation_loss(E, gt, mode)[0].backward()
        assert E.grad is not None and torch.isfinite(E.grad).all() and E.grad.abs().sum() > 0, mode
    # phụ thuộc diện tích ROI: density(đồng đều) = log(1+margin) với mọi diện tích; lesion_context = 0 khi đạt đích điều chỉnh theo diện tích
    vals = []
    for s in (4, 8, 16):
        g2 = torch.zeros(1, 1, 64, 64); g2[:, :, 30 - s:30 + s, 30 - s:30 + s] = 1
        l, dd = LX.explanation_loss(torch.ones(1, 1, h, h), g2, 'density', ring_cells=1)
        vals.append(float(l))
        a = float(dd['a_r'])
        tau = a + 0.5 * (1 - a)
        mr = LX.lesion_plus_context(LX.downsample_mask(g2, (h, h)), 1)
        unif = torch.ones_like(mr) / mr.numel(); ideal_r = mr / mr.sum()
        m_ideal = float((ideal_r * mr).sum()); m_unif = float((unif * mr).sum())        # M_r của hai map; nội suy tuyến tính theo lam
        lam = (tau - m_unif) / (m_ideal - m_unif)
        assert 0 < lam < 1
        lc, _ = LX.explanation_loss((1 - lam) * unif + lam * ideal_r, g2, 'lesion_context', ring_cells=1)
        lc_below, _ = LX.explanation_loss((1 - lam * 0.8) * unif + lam * 0.8 * ideal_r, g2, 'lesion_context', ring_cells=1)
        assert float(lc) < 1e-5 and float(lc_below) > 1e-3, (s, float(lc), float(lc_below))
    assert max(vals) - min(vals) < 1e-3 and abs(vals[0] - np.log(2)) < 1e-3, vals
    # map đồng đều bị phạt
    return f'đồng đều bị phạt, lý tưởng ≈ 0; mask rỗng bỏ qua; gradient hữu hạn; density(đồng đều) bất biến diện tích ({vals[0]:.3f}≈ln2)'


# ------------------------------------------------------------------ consistency
@test
def test_inverse_warp_and_consistency():
    B = 4
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, 32), torch.linspace(-1, 1, 32), indexing='ij')
    pat = (torch.exp(-((xx - 0.2) ** 2 + (yy + 0.1) ** 2) / 0.2) + 0.3 * torch.sin(2 * xx)).expand(B, 1, 32, 32).contiguous()
    # hflip chính xác
    hf = LX.hflip_theta(B, DEV)
    assert torch.allclose(LX.warp(pat, hf), pat.flip(-1), atol=1e-5)
    g = torch.Generator().manual_seed(0)
    th = LX.sample_affine(B, DEV, generator=g)
    back = LX.inverse_warp(LX.warp(pat, th), th)
    V = LX.valid_pixel_mask(th, (32, 32))
    err = ((back - pat).abs() * V).max().item()
    assert err < 0.03, err
    assert 0.4 < V.mean().item() < 1.0, 'mặt nạ hợp lệ phải loại bỏ viền nhưng vẫn giữ đa số pixel'
    # sai số phải giảm khi tăng độ phân giải (phân biệt sai số nội suy với lỗi hệ thống)
    def err_at(n):
        y2, x2 = torch.meshgrid(torch.linspace(-1, 1, n), torch.linspace(-1, 1, n), indexing='ij')
        p2 = (torch.exp(-((x2 - 0.2) ** 2 + (y2 + 0.1) ** 2) / 0.2) + 0.3 * torch.sin(2 * x2)).expand(B, 1, n, n).contiguous()
        return ((LX.inverse_warp(LX.warp(p2, th), th) - p2).abs() * LX.valid_pixel_mask(th, (n, n))).max().item()
    assert err_at(64) < err * 0.5, 'sai số không giảm theo độ phân giải -> lỗi hệ thống'
    A3 = lambda t: torch.cat([t, torch.tensor([0., 0., 1.]).view(1, 1, 3).expand(B, 1, 3)], 1)
    assert (A3(LX.invert_theta(th)) @ A3(th) - torch.eye(3)).abs().max() < 1e-5
    # theta^-1 ∘ theta = I
    A = LX.invert_theta(th)
    # mất nhất quán = 0 khi E2 là bản warp của E1; lớn khi E2 ngẫu nhiên
    E1 = torch.softmax(torch.randn(B, 1 * 32 * 32), -1).view(B, 1, 32, 32) * 1.0
    E1 = pat.clamp(min=0) + 0.01
    E2 = LX.warp(E1, th)
    l_ok = LX.evidence_consistency(E1, E2, th)
    l_bad = LX.evidence_consistency(E1, torch.rand(B, 1, 32, 32), th)
    assert float(l_ok) < 0.05 and float(l_bad) > float(l_ok) + 0.1, (float(l_ok), float(l_bad))
    lg = torch.randn(B, 2)
    assert float(LX.prediction_consistency(lg, lg)) < 1e-7 and float(LX.prediction_consistency(lg, -lg)) > 0.1
    return f'inverse-warp sai số {err:.3f} trên pixel hợp lệ; hflip chính xác; consistency {float(l_ok):.3f} (đúng) vs {float(l_bad):.3f} (sai)'


# ------------------------------------------------------------------ gradient flow
@test
def test_gradient_flow():
    x = torch.randn(4, 3, IMG, IMG); gt = (torch.rand(4, 1, IMG, IMG) > 0.9).float(); gt[:, :, 20:30, 20:30] = 1
    y = torch.tensor([0, 1, 0, 1])

    def grads(model, loss_fn):
        model.train(); model.zero_grad()
        loss_fn(model(x)).backward()
        g = lambda mod: sum(float(p.grad.abs().sum()) for p in mod.parameters() if p.grad is not None)
        return g(model.ev_head) if hasattr(model, 'ev_head') else 0.0, g(model.dec), g(model.enc)
    m = MX.BreastVisionXAI(MX.XAIConfig(pretrained=False, evidence_mode='ewp'))
    ev_g, dec_g, enc_g = grads(m, lambda o: LX.cls_loss(o['logits'], y))
    assert ev_g > 0, 'ewp: classification loss phải truyền gradient tới evidence head'
    assert dec_g == 0, 'detach_mask=True: loss phân loại không được chảy vào decoder phân vùng'
    ev_g, dec_g, enc_g = grads(m, lambda o: LX.seg_loss(o['seg'], gt))
    assert dec_g > 0 and ev_g == 0
    ev_g2, _, _ = grads(m, lambda o: LX.explanation_loss(o['evidence'], gt, 'lesion_context')[0])
    assert ev_g2 > 0, 'explanation loss phải truyền gradient tới evidence head'
    d = MX.BreastVisionXAI(MX.XAIConfig(pretrained=False, evidence_mode='decoupled'))
    ev_d, _, _ = grads(d, lambda o: LX.cls_loss(o['logits'], y))
    assert ev_d == 0, 'decoupled: classification loss KHÔNG được chạm evidence head'
    ev_d2, _, _ = grads(d, lambda o: LX.explanation_loss(o['evidence'], gt, 'lesion_context')[0])
    assert ev_d2 > 0
    # consistency loss: gradient tới evidence head
    th = LX.sample_affine(4, DEV)
    m.train(); m.zero_grad()
    o1, o2 = m(x), m(LX.apply_label_preserving(x, th))
    (LX.evidence_consistency(o1['evidence'], o2['evidence'], th) + LX.prediction_consistency(o1['logits'], o2['logits'])).backward()
    assert sum(float(p.grad.abs().sum()) for p in m.ev_head.parameters() if p.grad is not None) > 0
    return 'cls→evidence(ewp) có; cls→decoder không (detach); cls→evidence(decoupled) không; expl→evidence có; cons→evidence có'


# ------------------------------------------------------------------ checkpoint, suy luận không GT
@test
def test_checkpoint_roundtrip_all_models():
    x = torch.randn(2, 3, IMG, IMG)
    for name in ('E0', 'E1', 'E2', 'E3', 'X0', 'X6', 'V3_full', 'A_evidence_egr'):
        m = tiny_model(name).eval()
        meta = dict(experiment=name, num_classes=2, dim=128, base=32)
        p = os.path.join(TMP, f'{name}.pt')
        C2.save_model_checkpoint(p, m, meta)
        m2, _ = C2.load_model_from_checkpoint(p, DEV)
        o1, o2 = m(x), m2(x)
        for k in ('seg', 'logits', 'evidence'):
            if o1.get(k) is not None:
                assert torch.allclose(o1[k], o2[k], atol=1e-6), (name, k)
    return 'save→load cho E0,E1,E2,E3,X0,X6,A_evidence_egr cho đầu ra giống hệt'


@test
def test_inference_does_not_use_gt_mask():
    sig = list(inspect.signature(MX.BreastVisionXAI.forward).parameters)
    assert sig == ['self', 'x', 'ev_mask'], sig
    assert list(inspect.signature(C2.predict_arrays).parameters) == ['model', 'X', 'indices', 'device', 'batch_size', 'amp', 'tta'], 'predict_arrays không được nhận mask'
    df, X, M, y, *_ = synth_arrays()
    m = tiny_model('X6').eval()
    idx = np.arange(6)
    a = C2.predict_arrays(m, X, idx, DEV)
    M_backup = M.copy(); M[:] = 1 - M                        # đảo toàn bộ GT mask
    b = C2.predict_arrays(m, X, idx, DEV)
    M[:] = M_backup
    assert np.allclose(a['prob'], b['prob']) and np.allclose(a['seg_prob'].astype(float), b['seg_prob'].astype(float)), 'dự đoán phụ thuộc GT mask!'
    return 'forward(x, ev_mask) không có tham số mask GT; đảo toàn bộ GT mask không đổi dự đoán'


# ------------------------------------------------------------------ rò rỉ, trùng lặp
@test
def test_leakage_detection_and_duplicates():
    df, X, M, y, rep, info = synth_arrays()
    lr = DA.leakage_report(df)
    assert lr['ok'], lr
    assert set(df['split']) == {'train', 'val', 'test'}
    # phép thử âm: đẩy 1 ảnh của bệnh nhân train sang test -> phải bị phát hiện
    bad = df.copy()
    cnt = bad[bad['split'] == 'train'].groupby('patient_id').size()
    p = cnt[cnt >= 2].index[0]
    i = bad.index[(bad['patient_id'] == p) & (bad['split'] == 'train')][0]
    bad.loc[i, 'split'] = 'test'
    assert not DA.leakage_report(bad)['ok'], 'không phát hiện rò rỉ bệnh nhân'
    # ảnh trùng khác mã bệnh nhân vẫn phải cùng nhóm
    root = os.path.join(TMP, 'dupset'); shutil.copytree(os.path.join(TMP, 'busbra'), root)
    src = os.path.join(root, 'Images', 'bus_0001-l.png')
    shutil.copy(src, os.path.join(root, 'Images', 'bus_9999-l.png')); shutil.copy(os.path.join(root, 'Masks', 'mask_0001-l.png'), os.path.join(root, 'Masks', 'mask_9999-l.png'))
    import pandas as pd
    meta = pd.read_csv(os.path.join(root, 'bus_data.csv'))
    meta.loc[len(meta)] = dict(ID='bus_9999-l', Case='C9999', Pathology=meta.iloc[0]['Pathology'], BIRADS=4, Device='synthetic')
    meta.to_csv(os.path.join(root, 'bus_data.csv'), index=False)
    d2, _ = DA.index_busbra(root)
    d2, r2 = DA.audit_dataframe(d2)
    assert r2['n_exact_duplicate_images'] >= 2
    g = DA.build_groups(d2)
    a = int(np.where(d2['patient_id'] == 'C9999')[0][0]); b = int(np.where(d2['image_path'].str.contains('bus_0001-l'))[0][0])
    assert g[a] == g[b], 'ảnh trùng không được gom vào cùng nhóm với bản gốc'
    clean, drop = DA.drop_invalid(d2)
    assert drop['dropped_exact_dups'] >= 1
    # nhãn mâu thuẫn
    meta2 = pd.read_csv(os.path.join(root, 'bus_data.csv')); meta2.loc[meta2['ID'] == 'bus_9999-l', 'Pathology'] = 'malignant' if meta.iloc[0]['Pathology'] == 'benign' else 'benign'
    meta2.to_csv(os.path.join(root, 'bus_data.csv'), index=False)
    d3, _ = DA.index_busbra(root); d3, r3 = DA.audit_dataframe(d3)
    assert r3['n_label_conflict_groups'] >= 1
    return 'chia theo bệnh nhân không rò rỉ; phép thử âm bị phát hiện; trùng/gần trùng cùng nhóm; nhãn mâu thuẫn được báo'


@test
def test_busi_binary_excludes_normal_and_cross_dataset():
    root = DA.make_synthetic_busi(os.path.join(TMP, 'busi'), n=12, seed=5)
    df, info = DA.index_busi_binary(root)
    assert info['n_normal_excluded'] == 6 and set(df['label']) <= {0, 1} and (df['pathology'] != 'normal').all()
    dfa, _ = synth_arrays()[0], None
    a, _ = DA.audit_dataframe(DA.index_busbra(os.path.join(TMP, 'busbra'))[0])
    b, _ = DA.audit_dataframe(df, mask_col='mask_paths')
    cd = DA.cross_dataset_duplicates(a, b)
    assert cd['ok'], cd
    # chèn một bản sao để chắc chắn phát hiện được
    b2 = b.copy(); b2.loc[0, 'sha256'] = a.loc[0, 'sha256']; b2.loc[0, '_dhash'] = a.loc[0, '_dhash']
    assert not DA.cross_dataset_duplicates(a, b2)['ok']
    return 'BUSI: loại 6 ảnh normal, không chuyển thành benign; không trùng với BUS-BRA; bản sao chèn vào bị phát hiện'


# ------------------------------------------------------------------ khóa pipeline
@test
def test_external_evaluation_lock():
    lock = EV.ExternalEvalLock(os.path.join(TMP, 'lock.json'))
    ck = os.path.join(TMP, 'X6.pt')
    if not os.path.exists(ck):
        C2.save_model_checkpoint(ck, tiny_model('X6'), dict(experiment='X6', num_classes=2))
    called = []
    loader = lambda: called.append(1) or 'DATA'
    pred = lambda data, L: dict(auroc=0.5, T=L['temperature'])
    try:
        EV.evaluate_external(lock, ck, loader, pred); raise AssertionError('đã đọc dữ liệu ngoài khi chưa khóa')
    except EV.LockError:
        pass
    assert not called, 'load_external_fn bị gọi trước khi khóa'
    lock.lock(ck, dict(img=IMG), temperature=1.3, threshold=0.5, abstain_threshold=0.6)
    out = EV.evaluate_external(lock, ck, loader, pred)
    assert called == [1] and out['T'] == 1.3
    for fn in (lambda: EV.evaluate_external(lock, ck, loader, pred), lambda: lock.lock(ck, {}, 1, 0.5, 0.5)):
        try:
            fn(); raise AssertionError('cho phép đánh giá/khóa lại')
        except EV.LockError:
            pass
    # sửa checkpoint sau khóa
    lock2 = EV.ExternalEvalLock(os.path.join(TMP, 'lock2.json')); lock2.lock(ck, {}, 1, 0.5, 0.5)
    with open(ck, 'ab') as f:
        f.write(b'x')
    try:
        lock2.guard_access(ck); raise AssertionError('không phát hiện checkpoint bị sửa')
    except EV.LockError:
        pass
    return 'từ chối khi chưa khóa (không đọc dữ liệu); đánh giá 1 lần; khóa lại/đánh giá lại bị chặn; sửa checkpoint bị phát hiện'


# ------------------------------------------------------------------ mô hình đầy đủ
@test
def test_all_registered_experiments_forward():
    x = torch.randn(2, 3, IMG, IMG)
    shapes = {}
    for name in MX.EXPERIMENTS:
        m = tiny_model(name).eval()
        o = m(x)
        if o.get('logits') is not None:
            assert o['logits'].shape == (2, 2)
        if o.get('seg') is not None:
            assert o['seg'].shape == (2, 1, IMG, IMG)
        shapes[name] = 'ok'
        del m
    return f'{len(shapes)} cấu hình trong registry (E0–E4, X0–X6, ablation) forward đúng kích thước'


@test
def test_training_smoke_and_resume():
    df, X, M, y, *_ = synth_arrays()
    tr, va = np.where(df['split'] == 'train')[0], np.where(df['split'] == 'val')[0]
    data = dict(X=X, M=M, y=y, train=tr, val=va)
    spec = MX.EXPERIMENTS['X6']
    m = tiny_model('X6')
    cfg = C2.TrainCfg(epochs=2, patience=5, batch_size=4, amp=False, num_workers=0)
    out_dir = os.path.join(TMP, 'train')
    r = C2.train_model('X6', m, C2.make_loss_cfg(spec), data, cfg, out_dir, 0, DEV, meta=dict(experiment='X6', num_classes=2), log=lambda *a: None)
    h = json.load(open(r['history_path']))
    assert len(h) == 2 and all(np.isfinite(e['train_loss']) for e in h)
    for k in ('loss_seg', 'loss_cls', 'loss_exp', 'loss_cons_pred', 'loss_cons_ev'):
        assert k in h[0], k
    assert os.path.exists(r['best_path']) and os.path.exists(r['last_path'])
    r2 = C2.train_model('X6', tiny_model('X6'), C2.make_loss_cfg(spec), data, cfg, out_dir, 0, DEV, meta=dict(experiment='X6', num_classes=2), log=lambda *a: None)
    assert len(json.load(open(r2['history_path']))) == 2, 'resume phải bỏ qua các epoch đã xong'
    mb, _ = C2.load_model_from_checkpoint(r['best_path'], DEV)
    p = C2.predict_arrays(mb, X, va, DEV)
    assert np.isfinite(p['prob']).all()
    _cache['trained'] = (mb, data, p)
    return 'X6 (constraint + consistency) huấn luyện 2 epoch: loss hữu hạn, đủ các thành phần loss, checkpoint + resume hoạt động'


@test
def test_evaluation_functions_run():
    df, X, M, y, *_ = synth_arrays()
    mb, data, p = _cache['trained'] if 'trained' in _cache else (tiny_model('X6').eval(), None, None)
    va = np.where(df['split'] == 'val')[0]; te = np.where(df['split'] == 'test')[0]
    pv = C2.predict_arrays(mb, X, va, DEV); pt = C2.predict_arrays(mb, X, te, DEV)
    cm = EV.classification_metrics(y[te], pt['prob'][:, 1]); assert 'auroc' in cm
    loc = EV.localisation_report(pt['evidence'], M[te]); assert loc['n'] > 0
    fa = EV.faithfulness_report(mb, X, te, DEV, steps=3, max_images=3); assert 'deletion_auc_evidence' in fa and np.isfinite(fa['deletion_auc_random'])
    iv = EV.evidence_intervention(mb, X, te, DEV, max_images=3); assert iv['applicable']
    bg = EV.background_perturbation(mb, X, M, te, DEV, max_images=3)
    cs = EV.explanation_consistency_eval(mb, X, te, DEV); assert 0 <= cs['prediction_agreement'] <= 1
    Tm, rows = EV.calibration_report(pv['prob'], y[va], pt['prob'], y[te]); assert Tm > 0 and len(rows) == 2
    sel = EV.selective_report(pv['prob'], pt['prob'], y[te]); assert len(sel['rows']) == 3
    sm = EV.seg_metrics(pt['seg_prob'], M[te]); assert np.isfinite(sm['summary']['dice'])
    st = EV.summarize_seeds([0.7, 0.72, 0.71]); assert st['n_seeds'] == 3
    return 'phân loại, định vị, faithfulness, can thiệp evidence, nhiễu nền, nhất quán, hiệu chuẩn, chọn lọc, phân vùng, thống kê đều chạy'


@test
def test_gradcam_dict():
    m = tiny_model('X3').eval()
    cam, idx = C2.gradcam_dict(m, torch.randn(1, 3, IMG, IMG))
    assert cam.shape == (IMG, IMG) and np.isfinite(cam).all() and idx in (0, 1)
    return 'Grad-CAM cho mô hình trả dict hoạt động'


@test
def test_analyze_image_v2():
    df, X, M, y, *_ = synth_arrays()
    m = tiny_model('X6').eval()
    dcfg = dict(img_size=IMG, temperature=1.7, threshold=0.5, abstain_threshold=0.99, seg_threshold=0.5, min_area=5)
    rgb = np.random.RandomState(0).randint(0, 255, (120, 150, 3)).astype(np.uint8)
    r = C2.analyze_image_v2(rgb, m, dcfg, DEV)
    assert abs(r['p_cal'].sum() - 1) < 1e-6 and r['label'] in ('benign', 'malignant') and r['abstain'] is True       # ngưỡng 0.99 -> chắc chắn từ chối
    assert r['mask'].shape == (IMG, IMG) and r['evidence'].shape == (IMG, IMG) and r['cam'].shape == (IMG, IMG)
    assert list(inspect.signature(C2.analyze_image_v2).parameters) == ['rgb', 'model', 'dcfg', 'device'], 'không được có tham số mask'
    return 'analyze_image_v2: xác suất hiệu chuẩn, abstention, mask, box, evidence, Grad-CAM; chỉ nhận ảnh'


@test
def test_resume_midway_and_compact_marker():
    df, X, M, y, *_ = synth_arrays()
    tr, va = np.where(df['split'] == 'train')[0], np.where(df['split'] == 'val')[0]
    data = dict(X=X, M=M, y=y, train=tr, val=va)
    spec = MX.EXPERIMENTS['X3']; out_dir = os.path.join(TMP, 'resume_mid')
    meta = dict(experiment='X3', num_classes=2)
    cfg = C2.TrainCfg(epochs=3, patience=5, batch_size=4, amp=False, num_workers=0)

    class Interrupt(Exception):
        pass

    def die_after_first_epoch(msg):
        raise Interrupt()                      # log() được gọi SAU khi checkpoint 'last' đã lưu -> mô phỏng phiên bị ngắt thật
    try:
        C2.train_model('X3', tiny_model('X3'), C2.make_loss_cfg(spec), data, cfg, out_dir, 2, DEV, meta=meta, log=die_after_first_epoch)
        raise AssertionError('lẽ ra phải bị ngắt')
    except Interrupt:
        pass
    last = os.path.join(out_dir, 'X3_s2_last.pt')
    ck = C2.core.torch_load(last, DEV)
    assert ck['done'] is False and not ck.get('compact') and 'opt' in ck and ck['epoch'] == 0, 'checkpoint giữa chừng phải đầy đủ'
    r = C2.train_model('X3', tiny_model('X3'), C2.make_loss_cfg(spec), data, cfg, out_dir, 2, DEV, meta=meta, log=lambda *a: None)
    h = json.load(open(r['history_path']))
    assert [e['epoch'] for e in h] == [1, 2, 3], 'resume phải tiếp tục từ epoch 2 và chạy đủ 3 epoch'
    ck2 = C2.core.torch_load(last, DEV)
    assert ck2.get('compact') is True and ck2['done'] is True and os.path.getsize(last) < 1e6, 'xong rồi thì checkpoint cuối phải gọn'
    r3 = C2.train_model('X3', tiny_model('X3'), C2.make_loss_cfg(spec), data, cfg, out_dir, 2, DEV, meta=meta, log=lambda *a: None)
    assert len(json.load(open(r3['history_path']))) == 3, 'chạy lại sau khi xong không được huấn luyện thêm'
    return 'ngắt thật sau epoch 1 → resume chạy tiếp epoch 2–3; xong thì checkpoint cuối gọn (<1MB); chạy lại không huấn luyện thêm'


@test
def test_mask_encodings_and_empty_dataset_errors():
    import cv2, pandas as pd
    root = DA.make_synthetic_busbra(os.path.join(TMP, 'busbra_01'), n_patients=8, seed=3)
    for f in os.listdir(os.path.join(root, 'Masks')):                       # đổi mask sang mã hoá 0/1 (không phải 0/255)
        p = os.path.join(root, 'Masks', f); m = cv2.imread(p, cv2.IMREAD_GRAYSCALE); cv2.imwrite(p, (m > 127).astype(np.uint8))
    df, info = DA.index_busbra(root); a, rep = DA.audit_dataframe(df)
    assert rep['n_empty_mask'] == 0, 'mask 0/1 bị coi là rỗng'
    clean, drop = DA.drop_invalid(a); assert len(clean) == len(df)
    # nếu loại hết: phải báo lỗi rõ ràng, không để lỗi khó hiểu ở bước sau
    empty_masks = a.copy(); empty_masks['issue'] = 'mask_rong'
    try:
        DA.drop_invalid(empty_masks); raise AssertionError('lẽ ra phải báo lỗi khi loại hết')
    except RuntimeError as e:
        assert 'không còn ảnh nào' in str(e)
    for fn in (lambda: DA.cross_dataset_duplicates(clean.iloc[:0], clean), lambda: DA.cross_dataset_duplicate_rows(clean.iloc[:0], clean), lambda: DA.audit_dataframe(df.iloc[:0])):
        try:
            fn(); raise AssertionError('đầu vào rỗng phải báo lỗi rõ ràng')
        except ValueError as e:
            assert 'rỗng' in str(e).lower()
    return 'mask 0/1 không bị loại nhầm; loại hết/bảng rỗng → báo lỗi tiếng Việt rõ nguyên nhân thay vì "argmin of an empty sequence"'



# ------------------------------------------------------------------ V3 innovations tests
@test
def test_v3_posterior_zone_and_boundary_loss():
    m = torch.zeros(2, 1, 16, 16)
    m[0, 0, 3:6, 6:10] = 1.0
    pz = LX.posterior_zone(m, depth_cells=4)
    assert (pz * m).sum() == 0, 'vùng hậu âm không được chồng lên tổn thương'
    assert pz[0, 0, 6:10, 6:10].mean() > 0.5, 'phải kích hoạt ngay phía dưới khối u'
    assert pz[0, 0, :3, :].sum() == 0, 'không được kích hoạt phía trên khối u'
    assert pz[1].sum() == 0, 'mask rỗng phải cho hậu âm rỗng'
    seg_lg = torch.randn(2, 1, 64, 64, requires_grad=True)
    gt = torch.zeros(2, 1, 64, 64); gt[:, :, 20:40, 20:40] = 1.0
    bl = LX.boundary_loss(seg_lg, gt)
    bl.backward()
    assert torch.isfinite(bl) and seg_lg.grad.abs().sum() > 0
    return 'posterior_zone chỉ nằm dưới khối u và rời rạc với u; boundary_loss hữu hạn và có gradient'


@test
def test_v3_evidence_coupling_all_and_decomposition():
    torch.manual_seed(2)
    m, _ = MX.build_experiment('V3_full', pretrained=False)
    m.eval()
    x = torch.randn(2, 3, IMG, IMG)
    C, rest, lg = m.decompose_logits(x)
    err = (C.sum((2, 3)) + rest - lg).abs().max().item()
    assert err < 1e-4, f'sai số phân rã V3 quá lớn: {err}'
    # Can thiệp ev_mask trên V3 phải thay đổi TẤT CẢ các vùng phụ thuộc evidence
    o = m(x); E = o['evidence']
    mk = torch.ones_like(E); mk[:, :, :4, :4] = 0
    lg2 = m(x, ev_mask=mk)['logits']
    assert (lg2 - lg).abs().max() > 0
    # Đảm bảo E0 legacy wrapper nhận đúng cờ pretrained_loaded
    from unittest.mock import patch
    with patch('bus_core.build_model', return_value=(torch.nn.Linear(2, 2), True)):
        w, _ = MX.build_experiment('E0', pretrained=True)
        assert w.pretrained_loaded is True
    return f'phân rã cộng toàn bộ 4 vùng V3 chính xác (sai số {err:.1e}); sửa cờ pretrained_loaded E0'


@test
def test_v3_counterfactual_faithfulness_and_ema_tta():
    torch.manual_seed(3)
    m, cfgd = MX.build_experiment('V3_full', pretrained=False)
    m.train()
    x = torch.randn(4, 3, IMG, IMG)
    mb = torch.zeros(4, 1, IMG, IMG); mb[:, :, 16:40, 16:40] = 1.0
    yb = torch.tensor([0, 1, 0, 1])
    out = m(x)
    E = out['evidence']
    thr_hi = torch.quantile(E.flatten(1), 0.8, dim=1, keepdim=True).view(-1, 1, 1, 1)
    thr_lo = torch.quantile(E.flatten(1), 0.2, dim=1, keepdim=True).view(-1, 1, 1, 1)
    l_hi, _, _ = m.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=(E < thr_hi).float())
    l_lo, _, _ = m.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=(E > thr_lo).float())
    lcfg = C2.make_loss_cfg(MX.EXPERIMENTS['V3_full'])
    loss, parts = LX.total_loss(out, mb, yb, lcfg, logits_drop_hi=l_hi, logits_drop_lo=l_lo)
    assert 'edge' in parts and 'faith' in parts and torch.isfinite(loss)
    loss.backward()
    assert sum(float(p.grad.abs().sum()) for p in m.ev_head.parameters() if p.grad is not None) > 0
    # TTA & EMA
    ema = C2.ModelEMA(m, decay=0.9)
    ema.update(m)
    X_np = (np.random.rand(4, IMG, IMG, 3) * 255).astype(np.uint8)
    p_tta = C2.predict_arrays(m, X_np, np.arange(4), DEV, tta=True)
    assert p_tta['prob'].shape == (4, 2) and np.allclose(p_tta['evidence'].sum((1, 2)), 1.0, atol=1e-4)
    return 'faithfulness + edge loss có gradient; EMA và 2-view TTA hoạt động chuẩn'

def main():
    tests = [(n, f) for n, f in sorted(globals().items()) if callable(f) and getattr(f, '_is_test', False)]
    order = [n for n, _ in tests]
    # giữ thứ tự có phụ thuộc: huấn luyện trước đánh giá
    tests.sort(key=lambda t: (t[0] == 'test_evaluation_functions_run', t[0] == 'test_training_smoke_and_resume' and False, t[0]))
    for name, fn in tests:
        t0 = time.time()
        try:
            detail = fn() or ''
            RESULTS.append(dict(name=name, status='PASS', seconds=round(time.time() - t0, 1), detail=detail))
        except Exception as e:
            RESULTS.append(dict(name=name, status='FAIL', seconds=round(time.time() - t0, 1), detail=f'{type(e).__name__}: {e}', trace=traceback.format_exc()[-1500:]))
        print(f"[{RESULTS[-1]['status']}] {name} ({RESULTS[-1]['seconds']}s) {RESULTS[-1]['detail']}", flush=True)
    out = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'test_results.json')
    json.dump(dict(device=str(DEV), torch=torch.__version__, n=len(RESULTS), passed=sum(r['status'] == 'PASS' for r in RESULTS), results=RESULTS), open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    shutil.rmtree(TMP, ignore_errors=True)
    print(f"\n{sum(r['status'] == 'PASS' for r in RESULTS)}/{len(RESULTS)} PASS")
    return all(r['status'] == 'PASS' for r in RESULTS)


if __name__ == '__main__':
    sys.exit(0 if main() else 1)

## 3. Cấu hình Kaggle
Mặc định `smoke_test` để tránh vô tình chạy dài. **Để chạy thật:** đặt `RUN_MODE = "full_experiment"`, add Input BUS-BRA và BUSI, bật GPU (và Internet nếu muốn trọng số ImageNet).
Các quyết định được **cố định trước khi huấn luyện** nằm ngay trong cell này và trong `experiment_protocol.md`: mô hình chính (`PRIMARY`), mô hình sẽ bị khóa và đánh giá ngoài (`LOCK_MODELS`), tiêu chí chọn mô hình (trung bình AUROC và Dice trên **validation BUS-BRA**).

In [ ]:
import os, sys, json, time, copy, shutil, random, gc, importlib, cv2
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn.functional as F
sys.path.insert(0, os.getcwd())
# Chống module cũ: kernel Kaggle giữ các module đã import từ notebook/lần chạy trước. Xoá khỏi bộ nhớ đệm để nạp đúng file vừa ghi ở mục 2.
_MODS = ['bus_core', 'bus_trust', 'losses_xai', 'models_xai', 'data_audit', 'bus_core_v2', 'evaluation_xai', 'tests_xai']
for _m in _MODS:
    sys.modules.pop(_m, None)
importlib.invalidate_caches()
for _m in _MODS:                                  # file trên đĩa phải là bản của notebook này (các cell %%writefile ở mục 2 phải đã chạy)
    if not os.path.exists(_m + '.py'):
        raise FileNotFoundError(f'Thiếu {_m}.py: hãy chạy các cell %%writefile ở mục 2 (Run All từ đầu).')
if 'EXPERIMENTS' not in open('models_xai.py', encoding='utf-8').read():
    raise RuntimeError('models_xai.py trên đĩa là BẢN CŨ (của notebook khác). Chạy lại các cell %%writefile ở mục 2 rồi chạy lại cell này.')
import bus_core as core, bus_trust as T, losses_xai as LX, models_xai as MX, data_audit as DA, bus_core_v2 as C2, evaluation_xai as EV
assert hasattr(MX, 'EXPERIMENTS') and hasattr(DA, 'index_busbra'), 'Module trong bộ nhớ là bản cũ: Run -> Restart session rồi Run All.'

# ===================== CẤU HÌNH =====================
RUN_MODE = "full_experiment"          # "smoke_test" | "full_experiment"
BUSBRA_ROOT, BUSI_ROOT = None, None          # None = tự tìm trong /kaggle/input (full) hoặc tự sinh dữ liệu tổng hợp (smoke)
BUSBRA_COL_OVERRIDES = {}                    # ví dụ {'image': 'ID', 'patient': 'Case', 'label': 'Pathology'} nếu tự dò cột sai
OFFICIAL_TEST_FOLD = None                    # đặt số fold chính thức của BUS-BRA nếu muốn dùng làm test (validation vẫn chia theo bệnh nhân)
SPLIT_SEED = 42
PRIMARY = 'V3_full'                               # phương pháp chính (cố định trước)
LOCK_MODELS = ['X0', 'X6', 'V3_full']                   # sẽ bị khóa rồi đánh giá ngoài trên BUSI (cố định trước)
RUN_CONFIGS = ['E0', 'E3', 'X0', 'X3', 'X4', 'X6', 'V3_base', 'V3_birads', 'V3_full']  # đổi thành list(MX.EXPERIMENTS) nếu muốn chạy đủ 23 cấu hình
SEG_THRESHOLD, COVERAGE_TARGET = 0.5, 0.9
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./working')
# =====================================================
SMOKE = RUN_MODE == 'smoke_test'
assert RUN_MODE in ('smoke_test', 'full_experiment')
WORK_DIR = os.path.join(WORK_DIR, RUN_MODE)      # tách thư mục theo chế độ: checkpoint/khóa của smoke_test KHÔNG bị dùng lại khi chạy full_experiment
if SMOKE:
    IMG_SIZE, EPOCHS, PATIENCE, BATCH, SEEDS, MAX_EVAL, NUM_WORKERS, PRETRAINED = 64, 2, 3, 4, [42], 6, 0, False
else:
    IMG_SIZE, EPOCHS, PATIENCE, BATCH, SEEDS, MAX_EVAL, NUM_WORKERS, PRETRAINED = 256, 40, 10, 16, [42, 43, 44], 100, 2, True
MIN_AREA = max(20, int(0.0015 * IMG_SIZE * IMG_SIZE))
DEVICE = C2.get_device()
CKPT_DIR, OUT_DIR, FIG_DIR = (os.path.join(WORK_DIR, d) for d in ('checkpoints', 'outputs', 'figures'))
for d in (WORK_DIR, CKPT_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
TRAIN_CFG = C2.TrainCfg(epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH, amp=DEVICE.type == 'cuda', num_workers=NUM_WORKERS)
C2.seed_everything(SPLIT_SEED)
print(f'RUN_MODE={RUN_MODE} | thiết bị={DEVICE} | IMG_SIZE={IMG_SIZE} | EPOCHS={EPOCHS} | SEEDS={SEEDS} | PRETRAINED={PRETRAINED}')
if SMOKE:
    print('\n' + '!' * 78 + '\n  SMOKE TEST: dữ liệu TỔNG HỢP, chỉ kiểm tra mã/gradient. MỌI SỐ LIỆU DƯỚI ĐÂY KHÔNG CÓ GIÁ TRỊ KHOA HỌC.\n' + '!' * 78)

## 4. Dataset discovery
* **Full:** tự tìm thư mục có `Images/` và `Masks/` (BUS-BRA) và thư mục có `benign/`, `malignant/` (BUSI). Không tìm thấy hoặc nhiều ứng viên → dừng kèm hướng dẫn, không đoán mò.
* **Smoke:** sinh dữ liệu tổng hợp (có file `SYNTHETIC_README.txt`).
* **Lưu ý trung thực:** tên cột CSV của BUS-BRA **chưa được tác giả code xác minh trực tiếp**; `index_busbra` tự dò theo tên phổ biến (`ID`, `Case`, `Pathology`...) và dừng với thông báo rõ nếu không khớp — khi đó đặt `BUSBRA_COL_OVERRIDES`.

In [ ]:
if SMOKE:
    BUSBRA_ROOT = DA.make_synthetic_busbra(os.path.join(WORK_DIR, 'synthetic_busbra'), n_patients=36, seed=0)
    BUSI_ROOT = DA.make_synthetic_busi(os.path.join(WORK_DIR, 'synthetic_busi'), n=20, seed=5)
else:
    def one(cands, what):
        if not cands:
            raise FileNotFoundError(f'Không tìm thấy {what} trong {INPUT_DIR}. Add Input dataset hoặc đặt đường dẫn thủ công.')
        if len(cands) > 1:
            raise RuntimeError(f'Nhiều ứng viên cho {what}: {cands}. Hãy đặt đường dẫn thủ công.')
        return cands[0]
    BUSBRA_ROOT = BUSBRA_ROOT or one(DA.find_busbra_candidates(INPUT_DIR), 'BUS-BRA (thư mục có Images/ và Masks/)')
    BUSI_ROOT = BUSI_ROOT or one(DA.find_busi_candidates(INPUT_DIR), 'BUSI (thư mục có benign/ và malignant/)')
print('BUS-BRA:', BUSBRA_ROOT); print('BUSI   :', BUSI_ROOT)

## 5. Data audit
Kiểm tra: ảnh/mask lỗi, ghép ảnh–mask, mask rỗng, lệch kích thước, trùng tuyệt đối (SHA-256), gần trùng (dHash 256 bit), **nhãn mâu thuẫn**, và **trùng giữa BUS-BRA với BUSI**. Với BUSI, bước audit chỉ đọc để phát hiện trùng lặp; nó không tham gia huấn luyện, chọn mô hình, tiền xử lý hay hiệu chuẩn. Ảnh BUSI trùng với BUS-BRA (nếu có) bị loại khỏi tập đánh giá ngoài và được ghi lại.

In [ ]:
df_raw, info = DA.index_busbra(BUSBRA_ROOT, col_overrides=BUSBRA_COL_OVERRIDES)
print('Chỉ mục BUS-BRA:', {k: v for k, v in info.items() if k != 'unpaired_examples'})
df_a, rep = DA.audit_dataframe(df_raw)
print('Lý do loại (BUS-BRA):', df_a['issue'].replace('', 'hợp lệ').value_counts().to_dict())
df_clean, drop = DA.drop_invalid(df_a)           # báo lỗi rõ ràng nếu loại hết
print('Audit BUS-BRA:', rep); print('Loại bỏ:', drop)
busi_raw, busi_info = DA.index_busi_binary(BUSI_ROOT)
busi_a, busi_rep = DA.audit_dataframe(busi_raw, mask_col='mask_paths')
busi_clean, busi_drop = DA.drop_invalid(busi_a)
dup_rows = DA.cross_dataset_duplicate_rows(df_clean, busi_clean)
print('BUSI (nhị phân):', busi_info, '| audit:', busi_rep, '| trùng với BUS-BRA:', int(dup_rows.sum()), 'ảnh (bị loại khỏi tập ngoài)')
busi_clean = busi_clean[~dup_rows].reset_index(drop=True)
cross = DA.cross_dataset_duplicates(df_clean, busi_clean)
assert cross['ok'], cross
assert set(busi_clean['label'].unique()) <= {0, 1} and (busi_clean['pathology'] != 'normal').all()
json.dump(dict(busbra=rep, busbra_dropped=drop, busi=busi_rep, busi_dropped=busi_drop, busi_info=busi_info, busi_duplicates_with_busbra_removed=int(dup_rows.sum()),
               cross_after=cross), open(os.path.join(OUT_DIR, 'data_audit.json'), 'w'), indent=1, default=str)
print('Sau khi loại trùng: BUSI còn', len(busi_clean), 'ảnh | BUS-BRA còn', len(df_clean), 'ảnh,', df_clean['patient_id'].nunique(), 'bệnh nhân')

## 6. Preprocessing
Letterbox thành hình vuông rồi resize (ảnh và mask xử lý cùng cách), chuẩn hóa ImageNet. Mảng ảnh được nạp vào RAM. **BUSI chưa được nạp** ở bước này.

## 7. BUS-BRA splitting (theo bệnh nhân)
`StratifiedGroupKFold` với nhóm = hợp của (cùng bệnh nhân) và (ảnh trùng/gần trùng). Nếu `OFFICIAL_TEST_FOLD` được đặt, test = fold chính thức nhưng validation **vẫn được tái tạo từ bệnh nhân không thuộc test** (một số công bố ghi nhận ảnh cùng bệnh nhân có thể nằm ở cả train và val trong phân chia được cung cấp). Kiểm tra rò rỉ: nhóm, bệnh nhân, SHA-256, cặp gần trùng giữa các tập.

In [ ]:
df = DA.grouped_split(df_clean, seed=SPLIT_SEED, official_test_fold=OFFICIAL_TEST_FOLD)
leak = DA.leakage_report(df)
print('Báo cáo rò rỉ:', leak); assert leak['ok'], 'Có rò rỉ giữa các tập!'
print(pd.crosstab(df['split'], df['label']).rename(columns={0: 'benign', 1: 'malignant'}).reindex(['train', 'val', 'test']).to_string())
print('Số bệnh nhân:', df.groupby('split')['patient_id'].nunique().to_dict())
h = DA.write_manifest(df, os.path.join(OUT_DIR, 'busbra_split_manifest.csv')); print('Manifest sha256:', h[:16], '...')

X, M, y = C2.load_arrays(df, IMG_SIZE)
train_idx, val_idx, test_idx = (np.where(df['split'].values == s)[0] for s in ('train', 'val', 'test'))
DATA = dict(X=X, M=M, y=y, train=train_idx, val=val_idx)
print('X', X.shape, X.dtype, '| M', M.shape, np.unique(M), '| ảnh có tổn thương rỗng sau resize:', int((M.sum((1, 2)) == 0).sum()))

## 8. DataLoader
`BUSDataset` trả (ảnh, mask, nhãn, chỉ số). Mask chỉ được dùng trong **loss** và **đánh giá**; mô hình không nhận mask. Augmentation ảnh–mask đồng bộ (lật ngang, affine nhỏ, độ sáng/tương phản, gamma, nhiễu).

In [ ]:
dl = C2.make_loader(X, M, y, train_idx, min(BATCH, len(train_idx)), augment=True, shuffle=True, num_workers=0, seed=SPLIT_SEED)
xb, mb, yb, ib = next(iter(dl))
print('ảnh', tuple(xb.shape), xb.dtype, '| mask', tuple(mb.shape), torch.unique(mb).tolist(), '| nhãn', yb.tolist())
assert xb.shape[1] == 3 and mb.shape[1] == 1 and torch.isfinite(xb).all(); del dl

## 9. Baseline models (E0–E4 giữ từ notebook gốc)
| ID | Mô hình | Ghi chú |
|---|---|---|
| E0 | EfficientNet-B3 phân loại | transfer learning |
| E1 | U-Net | phân vùng |
| E2 | Attention U-Net | phân vùng |
| E3 | Attention U-Net đa nhiệm | phân vùng + phân loại |
| E4 | Attention U-Net không augmentation | ablation |
Tất cả được **huấn luyện lại theo giao thức BUS-BRA** (2 lớp benign/malignant). Checkpoint BUSI cũ không được dùng.

In [ ]:
for name in ['E0', 'E1', 'E2', 'E3', 'E4']:
    m, c = MX.build_experiment(name, pretrained=False)
    o = m.eval()(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
    print(name, MX.EXPERIMENTS[name]['desc'], '| tham số', f"{sum(p.numel() for p in m.parameters()) / 1e6:.2f}M",
          '| seg', None if o['seg'] is None else tuple(o['seg'].shape), '| logits', None if o['logits'] is None else tuple(o['logits'].shape))

## 10. Shared encoder
Encoder EfficientNet-B3 (torchvision, trọng số ImageNet nếu có Internet) trích đặc trưng đa tầng: `f1 (24ch,/2)`, `f2 (32,/4)`, `f3 (48,/8)`, `f5 (136,/16)`, `f8 (1536,/32)`. Một encoder dùng chung cho phân vùng và phân loại. Nếu không tải được trọng số, code in cảnh báo và cờ `pretrained_loaded=False` được lưu vào checkpoint (khi đó không phải transfer learning).

## 11. Segmentation decoder
Decoder kiểu U-Net: `d4 = B(↑f8 ⊕ f5)`, `d3 = B(↑d4 ⊕ f3)`, `d2 = B(↑d3 ⊕ f2)`, `d1 = B(↑d2 ⊕ f1)`, `seg = ↑(Conv1×1(d1)) ∈ R^{B×1×H×W}` (logit).

## 12. Multi-region fusion
Neck: `Z = Conv3×3(l3(f3) + ↑l5(f5) + ↑l8(f8)) ∈ R^{B×D×h×w}`, `h = H/8`. Mask **dự đoán** `m = AvgPool(σ(seg)) ∈ [0,1]^{B×1×h×w}` (detach: loss phân loại không đổi decoder; có thể tắt). Ba vùng:
* Global `g = mean_x Z(x)` (hoặc `Σ E(x)Z(x)` khi có evidence).
* Lesion `l = Σ m(x)Z(x) / (Σ m(x) + ε)`.
* Perilesional `p = Σ r(x)Z(x) / (Σ r(x) + ε)`, vòng `r = clamp(dilate_k(m) − m, 0, 1)` (`k = ring_cells`).
Fusion với `v_i = A_i x_i + b_i`: **concat** `h = [v_1;v_2;v_3]`; **weighted** `h = Σ softmax(a)_i v_i`; **gated** `h = Σ softmax_i(W[x_1;x_2;x_3])_i ⊙ v_i`. Mask rỗng cho vector ≈ 0 (hữu hạn). **Không có ground-truth mask ở suy luận.**

Kiến trúc:
```
ảnh x ──► EfficientNet-B3 encoder ──┬─► f1,f2,f3,f5,f8 ──► Seg decoder ──► seg logits ──► σ,AvgPool ──► m ──► ring r
                                    └─► Neck ──► Z (B×D×h×w) ──┬─► global g  (GAP hoặc Σ E·Z)
                                                                ├─► lesion  l = masked-avg(Z, m)
                                                                └─► peri    p = masked-avg(Z, r)
                    Z ──► evidence head ──► s ──► E = softmax_x(s) (ΣE = 1) ──► g_E = Σ E·Z  (đi vào logit)
                    [g_E, l, p] ──► fusion ──► Linear ──► logits (B×2)
```
### Nâng cấp V3 (`V3_base` → `V3_birads` → `V3_full`)
* **SCSE Attention** ở `SegDecoder` & `Neck` + **Boundary-Aware Loss** tại đường biên u (`lam_edge=0.5`).
* **Vùng hậu âm BI-RADS (`posterior_zone`)**: tính trực tiếp từ mask dự đoán `m` phía dưới tổn thương theo trục dọc chùm tia siêu âm (rời rạc với u, 0 tham số dư).
* **Evidence Coupling Toàn Vùng (`evidence_coupling_all=True`)**: bản đồ bằng chứng `E` điều chế cả 4 vùng (`global`, `lesion`, `perilesional`, `posterior`), bảo toàn phân rã cộng chính xác logit trên cả 4 vùng.
* **Counterfactual Faithfulness Loss (`lam_faith=0.3`)**: phạt trực tiếp nếu che 20% ô `E` cao nhất không làm giảm độ tự tin dự đoán hoặc che 20% ô `E` thấp nhất làm đổi dự đoán.
* **Label Smoothing (`0.05`) + Speckle Augmentation + EMA + 2-view TTA**.


In [ ]:
m = MX.build_experiment('X6', pretrained=False)[0].eval()
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
f = m.enc(x); o = m(x)
rows = [('x', x)] + [(f'f{k}', v) for k, v in f.items()] + [('Z', o['Z']), ('seg logits', o['seg']), ('mask m (feat)', o['mask_feat']), ('ring r', o['ring']), ('evidence E', o['evidence']),
        ('g', o['regions'][0]), ('l', o['regions'][1]), ('p', o['regions'][2]), ('logits', o['logits'])]
print(pd.DataFrame([dict(tensor=n, shape=tuple(v.shape)) for n, v in rows]).to_string(index=False))
print('Σ_x E =', o['evidence'].sum((2, 3)).flatten().tolist())

## 13. Evidence-guided classifier (evidence–logit coupling)
**Hai formulation khả thi** (phân tích và chọn **trước** khi có kết quả):

| | **A — Evidence-Weighted Pooling (EWP)** | **B — Evidence-Gated Residual (EGR)** |
|---|---|---|
| Định nghĩa | `E = softmax_x(s(Z)/τ)`, `g_E = Σ_x E(x)Z(x)` | `a = σ(s(Z))`, `g = mean_x(a(x)Z(x))` |
| Đi vào logit | Có, khả vi, `∂L_cls/∂E ≠ 0` | Có, khả vi |
| Chuẩn hóa | `ΣE = 1`, bất biến với diện tích | Không chuẩn hóa → nghiệm tầm thường `a ≡ 1` |
| Phân rã logit chính xác | **Có** (fusion concat): `z_k = Σ_x E(x)·c_k(x) + rest_k` | Không |
| Can thiệp xoá evidence | Giải tích được (`Δz = −Σ_{xoá} E c`) | Chỉ kiểm bằng thực nghiệm |
| Rủi ro | Softmax có thể collapse về vài ô | Dễ collapse đồng đều, phụ thuộc diện tích |

**Tiêu chí chọn chính (cố định trước, không dựa BUSI hay BUS-BRA):** (C1) phân rã cộng chính xác, (C2) chuẩn hóa/bất biến diện tích, (C3) can thiệp giải tích. A thỏa cả ba, B không thỏa → **A là phương pháp chính (X3+)**, B là đối chứng (`A_evidence_egr`). Thêm đối chứng `A_evidence_decoupled`: cùng evidence head nhưng **không** nối vào logit.
Công thức phân rã (fusion `concat`): `z = W_g(A_g g_E + b_g) + W_l v_l + W_p v_p + b`, nên `c_k(x) = E(x)·[W_g(A_g Z(x) + b_g)]_k` và `Σ_x c_k(x) + rest_k = z_k` (kiểm thử sai số ~1e-8).
**Evidence collapse:** `evidence_collapse_report` đo entropy chuẩn hóa `H(E)/log(hw)` (≈1: đồng đều) và số ô hiệu dụng `1/Σ E²` (<2 ô: tập trung một điểm).
**Chưa tuyên bố "faithful"**: chỉ sau khi mục 20 cho kết quả.

In [ ]:
m = MX.build_experiment('X3', pretrained=False)[0].eval()
C, rest, lg = m.decompose_logits(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print('Sai số phân rã logit tối đa:', (C.sum((2, 3)) + rest - lg).abs().max().item())
u = torch.ones(1, 1, 8, 8) / 64; peak = torch.zeros(1, 1, 8, 8); peak[0, 0, 3, 3] = 1
print('collapse (đồng đều):', EV.evidence_collapse_report(u)['collapse_flag'], '| collapse (một điểm):', EV.evidence_collapse_report(peak)['collapse_flag'])

## 14. Explanation constraint (dùng GT mask **chỉ trong loss**)
Định nghĩa trên lưới đặc trưng `h×w`: lesion `m_gt` (mask GT trung bình diện tích), vòng quanh tổn thương `r = clamp(dilate_k(m_gt) − m_gt, 0, 1)`, **lesion-plus-context** `ROI = dilate_k(m_gt)`. Evidence được **chuẩn hóa** `E ← E/ΣE`. `M_l = Σ E·m_gt`, `M_r = Σ E·ROI`, `a_l, a_r` = tỉ lệ diện tích.

| Mode | Loss | Ghi chú |
|---|---|---|
| `none` | 0 | đối chứng |
| `mask_only` | `1 − M_l` | chỉ tổn thương |
| `lesion_context` (**chính, chọn trước**) | `max(0, τ(a_r) − M_r)`, `τ(a) = a + δ(1−a)`, δ=0.5 | đích **điều chỉnh theo diện tích** (mức ngẫu nhiên = a) |
| `density` | `max(0, log ρ_out − log ρ_in + log(1+margin))`, `ρ_in = M_r/a_r`, `ρ_out = (1−M_r)/(1−a_r)` | bất biến diện tích |
Mẫu có ROI rỗng hoặc phủ ≥ 90% ảnh bị **bỏ qua** (không NaN). Đã kiểm thử: map đồng đều bị phạt; map lý tưởng ≈ 0; gradient hữu hạn tới evidence head; loss density của map đồng đều không phụ thuộc diện tích ROI (= ln 2).

In [ ]:
gt = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE); gt[:, :, IMG_SIZE // 3:IMG_SIZE // 2, IMG_SIZE // 3:IMG_SIZE // 2 + 8] = 1
h = IMG_SIZE // 8; uni = torch.ones(2, 1, h, h); ideal = LX.downsample_mask(gt, (h, h)); ideal = ideal / ideal.sum((1, 2, 3), keepdim=True)
for mode in ('mask_only', 'lesion_context', 'density'):
    print(f'{mode:15s} đồng đều = {float(LX.explanation_loss(uni, gt, mode)[0]):.4f} | lý tưởng = {float(LX.explanation_loss(ideal, gt, mode)[0]):.4f} | mask rỗng = {float(LX.explanation_loss(uni, torch.zeros_like(gt), mode)[0]):.4f}')

## 15. Explanation consistency
Biến đổi **bảo toàn nhãn**: lật ngang, affine nhỏ (xoay ≤ 10°, co giãn 0.9–1.1, dịch ≤ 5%), thay đổi cường độ nhẹ (tương phản 0.9–1.1, độ sáng ±0.1). Tách riêng:
* **Prediction consistency:** `½[KL(p‖p′) + KL(p′‖p)]`.
* **Evidence consistency:** `1 − cos(E ⊙ V, warp⁻¹(E′) ⊙ V)` với `V` = **mặt nạ pixel hợp lệ** (vị trí mà nội dung của ảnh biến đổi hoàn toàn là thật, không nhiễm đệm 0). Chỉ tính các mẫu có ≥ 25% pixel hợp lệ.
Đã kiểm thử: hflip chính xác; inverse-warp sai số giảm theo bậc hai khi tăng độ phân giải (loại trừ lỗi hệ thống); consistency ≈ 0 khi E′ là bản warp đúng của E.

In [ ]:
th = LX.sample_affine(4, torch.device('cpu'), generator=torch.Generator().manual_seed(0))
yy, xx = torch.meshgrid(torch.linspace(-1, 1, 64), torch.linspace(-1, 1, 64), indexing='ij')
pat = (torch.exp(-((xx - .2) ** 2 + (yy + .1) ** 2) / .2)).expand(4, 1, 64, 64).contiguous()
back = LX.inverse_warp(LX.warp(pat, th), th); V = LX.valid_pixel_mask(th, (64, 64))
print('sai số inverse-warp trên pixel hợp lệ:', float(((back - pat).abs() * V).max()), '| tỉ lệ pixel hợp lệ:', float(V.mean()))
print('hflip chính xác:', bool(torch.allclose(LX.warp(pat, LX.hflip_theta(4, torch.device('cpu'))), pat.flip(-1), atol=1e-5)))

## 16. Training engine
* PyTorch, **AdamW** (LR encoder nhỏ, phần mới lớn hơn), **ReduceLROnPlateau**, **early stopping**, **gradient clipping**, **mixed precision** (khi có GPU), **checkpoint tốt nhất** và **checkpoint cuối để resume**, kiểm tra loss NaN/Inf, tự giảm batch khi hết bộ nhớ GPU.
* Seed cố định theo từng lần chạy; log từng epoch ghi vào `*_history.json` (gồm từng thành phần loss).
* **Tiêu chí chọn checkpoint và mô hình (cố định trước):** trung bình AUROC và Dice trên **validation BUS-BRA** (chỉ dùng thành phần có sẵn).
* Cấu hình chính (`MAIN_CONFIGS` = X0, X3, X4, X6) chạy **nhiều seed**; các ablation còn lại một seed (ghi rõ trong báo cáo).
* Điều kiện thực nghiệm **giống nhau** giữa mọi cấu hình: cùng split, augmentation, tiền xử lý, LR, số epoch, patience.

In [ ]:
RUNS = {}
def seeds_for(name):
    return SEEDS if name in MX.MAIN_CONFIGS else SEEDS[:1]

def run(name, seed):
    spec = MX.EXPERIMENTS[name]
    model, cfgd = MX.build_experiment(name, pretrained=PRETRAINED)
    meta = dict(experiment=name, num_classes=2, dim=128, base=32, img_size=IMG_SIZE, seed=seed, model_cfg=cfgd, run_mode=RUN_MODE,
                pretrained_loaded=bool(getattr(model, 'pretrained_loaded', False)))
    if PRETRAINED and not meta['pretrained_loaded']:
        print(f'CẢNH BÁO [{name}]: không có trọng số pretrained -> backbone khởi tạo ngẫu nhiên.')
    cfg_run = copy.copy(TRAIN_CFG); cfg_run.use_ema = bool(spec.get('ema', False))
    RUNS[(name, seed)] = C2.train_with_oom_fallback(name, model, C2.make_loss_cfg(spec), DATA, cfg=cfg_run, out_dir=CKPT_DIR, seed=seed,
                                                    device=DEVICE, augment=spec.get('augment', True), meta=meta)
    del model; gc.collect()

def plot_history(name, seed):
    h = pd.DataFrame(json.load(open(RUNS[(name, seed)]['history_path'])))
    cols = [c for c in h.columns if c.startswith('loss_')]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    ax[0].plot(h['epoch'], h['train_loss'], label='tổng')
    for c in cols:
        ax[0].plot(h['epoch'], h[c], label=c.replace('loss_', ''), alpha=.7)
    ax[0].set_title(f'{name} seed {seed}: loss huấn luyện'); ax[0].legend(fontsize=7)
    for c in [c for c in h.columns if c.startswith('val_')] + ['score']:
        ax[1].plot(h['epoch'], h[c], label=c)
    ax[1].set_title('chỉ số validation (BUS-BRA)'); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, f'curves_{name}_s{seed}.png'), dpi=110); plt.show()
print('Engine sẵn sàng. Cấu hình sẽ chạy:', RUN_CONFIGS)

## 17. Baseline experiments (E0–E4)

In [ ]:
for n in ['E0', 'E1', 'E2', 'E3', 'E4']:
    if n in RUN_CONFIGS:
        for s in seeds_for(n):
            run(n, s)

## 18. Ablation experiments (X0–X6 và các ablation riêng)
Chuỗi X: **X0** encoder chung đa nhiệm → **X1** +lesion → **X2** +perilesional → **X3** +evidence–logit coupling → **X4** +explanation constraint → **X5** +consistency → **X6** =X3+constraint+consistency.
Ablation riêng: *lesion region* (X1 vs X0), *perilesional region* (X2 vs X1), *gating* (`A_fusion_weighted/gated` vs X2), *evidence coupling* (X3 vs X2; `A_evidence_decoupled`; `A_evidence_egr`), *explanation loss* (`A_exp_mask_only`, `A_exp_density` vs X4), *consistency loss* (`A_cons_pred_only`, `A_cons_ev_only` vs X5). **Calibration không nằm trong danh sách kiến trúc.**

Chuỗi V3 (mới): **V3_base** (`X6` + SCSE + Boundary loss + Label smoothing + EMA + TTA) → **V3_birads** (+ vùng hậu âm BI-RADS + evidence coupling toàn vùng) → **V3_full** (+ Counterfactual Faithfulness Loss).

In [ ]:
for n in [n for n in RUN_CONFIGS if n.startswith(('X', 'V3_', 'A_'))]:
    for s in seeds_for(n):
        run(n, s)
print('Đã huấn luyện:', sorted(RUNS))
if (PRIMARY, SEEDS[0]) in RUNS:
    plot_history(PRIMARY, SEEDS[0])

## 19. Internal evaluation (BUS-BRA test)
Ngưỡng phân loại chọn bằng Youden **trên validation**. Chỉ số: AUROC, AUPRC, sensitivity (malignant), specificity, balanced accuracy, F1; phân vùng: Dice, IoU, HD95 (pixel, trên ảnh có tổn thương). Tập test chỉ được dùng ở bước này trở đi.

In [ ]:
def load_run(key):
    return C2.load_model_from_checkpoint(RUNS[key]['best_path'], DEVICE)[0]

PRED, THR, rows = {}, {}, []
for key in sorted(RUNS):
    m = load_run(key)
    use_tta = bool(MX.EXPERIMENTS[key[0]].get('tta', False))
    PRED[key] = {s: C2.predict_arrays(m, X, idx, DEVICE, tta=use_tta) for s, idx in (('val', val_idx), ('test', test_idx))}
    del m
    P = PRED[key]; row = dict(model=key[0], seed=key[1])
    if P['val']['prob'] is not None:
        try:
            THR[key] = EV.select_threshold_youden(y[val_idx], P['val']['prob'][:, 1])
        except Exception:
            THR[key] = 0.5
        row.update(EV.classification_metrics(y[test_idx], P['test']['prob'][:, 1], THR[key]))
    if P['test']['seg_prob'] is not None:
        row.update(EV.seg_metrics(P['test']['seg_prob'], M[test_idx])['summary'])
    rows.append(row)
internal = pd.DataFrame(rows)
internal.to_csv(os.path.join(OUT_DIR, 'internal_results_per_seed.csv'), index=False)
print(internal.round(4).to_string(index=False))

agg = []
for name in sorted({r['model'] for r in rows}):
    sub = internal[internal['model'] == name]
    a = dict(model=name, n_seeds=len(sub))
    for k in ('auroc', 'auprc', 'sensitivity_malignant', 'specificity', 'balanced_accuracy', 'f1', 'dice', 'iou', 'hd95'):
        if k in sub and sub[k].notna().any():
            s = EV.summarize_seeds(sub[k].values); a[k + '_mean'], a[k + '_sd'] = s['mean'], s['sd']
    agg.append(a)
agg = pd.DataFrame(agg); agg.to_csv(os.path.join(OUT_DIR, 'internal_results_mean_sd.csv'), index=False)
print('\nTrung bình ± độ lệch chuẩn qua seed (sd = NaN khi chỉ có 1 seed):'); print(agg.round(4).to_string(index=False))

## 20. XAI evaluation
Ba loại đo **được tách biệt**:
* **Localisation** (so với mask): *Relevance Mass* trong lesion và lesion+context, *Pointing Game*, *area-adjusted attribution* `(mass − area)/(1 − area)` (0 = mức ngẫu nhiên). Không chứng minh evidence có ảnh hưởng tới dự đoán.
* **Faithfulness** (ảnh hưởng nhân quả tới dự đoán): *Deletion AUC* (thấp hơn là tốt), *Insertion AUC* (cao hơn là tốt) so với xếp hạng **ngẫu nhiên**; *evidence intervention* (xoá q% ô evidence lớn nhất so với xoá ngẫu nhiên). Kèm *background perturbation* (xáo pixel ngoài ROI vs trong tổn thương, không đổi histogram).
* **Consistency**: cosine evidence sau inverse-warp (trên pixel hợp lệ) và đồng thuận dự đoán, tách riêng.
Cùng với phát hiện **evidence collapse**. Chạy trên seed đầu tiên của X0, X3, X4, X6 (và ảnh test tối đa `MAX_EVAL` cho phần tốn kém).

In [ ]:
XAI, sub = {}, test_idx[:MAX_EVAL]
for n in [n for n in ['X0', 'X3', 'X4', 'X6', 'V3_birads', 'V3_full'] if (n, SEEDS[0]) in RUNS]:
    m = load_run((n, SEEDS[0])); P = PRED[(n, SEEDS[0])]['test']; ent = {}
    if P['evidence'] is not None:
        ent['localisation_evidence'] = EV.localisation_report(P['evidence'], M[test_idx])
        ent['collapse'] = EV.evidence_collapse_report(P['evidence'])
    cams = np.stack([cv2.resize(C2.gradcam_dict(m, core.normalize_to_tensor(X[k]).unsqueeze(0).to(DEVICE))[0], (IMG_SIZE // 8, IMG_SIZE // 8), interpolation=cv2.INTER_AREA) for k in sub])
    ent['localisation_gradcam'] = EV.localisation_report(cams, M[sub])
    ent['faithfulness'] = EV.faithfulness_report(m, X, sub, DEVICE, steps=3 if SMOKE else 8, max_images=len(sub))
    ent['evidence_intervention'] = EV.evidence_intervention(m, X, sub, DEVICE, max_images=len(sub))
    ent['background_perturbation'] = EV.background_perturbation(m, X, M, sub, DEVICE, max_images=len(sub))
    ent['consistency_hflip'] = EV.explanation_consistency_eval(m, X, sub, DEVICE, hflip_only=True)
    ent['consistency_affine'] = EV.explanation_consistency_eval(m, X, sub, DEVICE)
    XAI[n] = ent; del m
flat = [dict(model=n, group=g, metric=k, value=v) for n, ent in XAI.items() for g, d in ent.items() for k, v in d.items() if isinstance(v, (int, float, bool))]
xai_df = pd.DataFrame(flat); xai_df.to_csv(os.path.join(OUT_DIR, 'xai_evaluation.csv'), index=False)
print(xai_df.pivot_table(index=['group', 'metric'], columns='model', values='value').round(4).to_string())

## 21. Calibration
Temperature scaling: `T` chọn bằng cực tiểu NLL **trên validation BUS-BRA**, áp dụng cho test (và, không fit lại, cho BUSI). Chỉ số: **NLL, Brier, ECE** + reliability diagram. Calibration là tầng hậu xử lý, không phải kiến trúc mới.

In [ ]:
def best_seed(name):
    return max([s for (n, s) in RUNS if n == name], key=lambda s: RUNS[(name, s)]['best_score'])     # chọn theo validation BUS-BRA

CAL, TS = {}, {}
for name in sorted({PRIMARY, *LOCK_MODELS} & {n for n, _ in RUNS}):
    s = best_seed(name); pv, pt = PRED[(name, s)]['val']['prob'], PRED[(name, s)]['test']['prob']
    Tm, rws = EV.calibration_report(pv, y[val_idx], pt, y[test_idx])
    CAL[name] = dict(T=Tm, rows=rws, seed=s); TS[name] = Tm
    print(name, f'(seed {s})  T = {Tm:.3f}'); print(pd.DataFrame(rws).round(4).to_string(index=False))
pd.DataFrame([dict(model=n, seed=c['seed'], **r) for n, c in CAL.items() for r in c['rows']]).to_csv(os.path.join(OUT_DIR, 'calibration.csv'), index=False)

if PRIMARY in CAL:
    s = CAL[PRIMARY]['seed']; pt = PRED[(PRIMARY, s)]['test']['prob']
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    for a, (tag, tt) in zip(ax, (('trước hiệu chuẩn', 1.0), (f'sau (T={TS[PRIMARY]:.2f})', TS[PRIMARY]))):
        p = T.softmax(np.log(np.clip(pt, 1e-12, 1)), tt); conf, acc = p.max(1), (p.argmax(1) == y[test_idx]).astype(float)
        edges = np.linspace(0.5, 1, 6); xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mk = (conf > lo) & (conf <= hi)
            if mk.any():
                xs.append(conf[mk].mean()); ys.append(acc[mk].mean())
        a.plot([0.5, 1], [0.5, 1], 'k--'); a.scatter(xs, ys); a.set_title(f'{PRIMARY} {tag}\nECE={T.ece(p, y[test_idx]):.3f}'); a.set_xlabel('độ tin cậy'); a.set_ylabel('độ chính xác')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'reliability_primary.png'), dpi=110); plt.show()

## 22. Selective prediction
Độ tin cậy = max xác suất đã hiệu chuẩn; **ngưỡng từ chối chọn trên validation** theo coverage mục tiêu, áp dụng cho test. Báo cáo: coverage thực, selective risk, độ nhạy malignant trong ca được nhận, tỉ lệ ca malignant bị từ chối, **AURC**.

In [ ]:
SEL = {}
for name in CAL:
    s = CAL[name]['seed']; Tm = TS[name]
    pv = T.softmax(np.log(np.clip(PRED[(name, s)]['val']['prob'], 1e-12, 1)), Tm); pt = T.softmax(np.log(np.clip(PRED[(name, s)]['test']['prob'], 1e-12, 1)), Tm)
    SEL[name] = EV.selective_report(pv, pt, y[test_idx], coverages=(0.9, 0.8, 0.7))
    print(name, 'AURC =', round(SEL[name]['aurc'], 4)); print(pd.DataFrame(SEL[name]['rows']).round(4).to_string(index=False))
pd.DataFrame([dict(model=n, aurc=v['aurc'], **r) for n, v in SEL.items() for r in v['rows']]).to_csv(os.path.join(OUT_DIR, 'selective_prediction.csv'), index=False)

## 23. Pipeline lock
Sau khi **mô hình, seed, ngưỡng phân loại, temperature và ngưỡng từ chối** đã được chọn chỉ bằng BUS-BRA, chúng bị **khóa** (kèm SHA-256 của checkpoint và của cấu hình). Chỉ khi có file khóa hợp lệ, mã đánh giá ngoài mới đọc BUSI. Khóa chỉ cho phép **một lần** đánh giá ngoài; sửa checkpoint sau khi khóa bị phát hiện.

In [ ]:
LOCKS = {}
for name in LOCK_MODELS:
    if name not in CAL:
        continue
    s = CAL[name]['seed']; r = RUNS[(name, s)]; Tm = TS[name]
    pv = T.softmax(np.log(np.clip(PRED[(name, s)]['val']['prob'], 1e-12, 1)), Tm)
    thr = EV.select_threshold_youden(y[val_idx], pv[:, 1]); abst = float(np.quantile(pv.max(1), 1 - COVERAGE_TARGET))
    lock = EV.ExternalEvalLock(os.path.join(OUT_DIR, f'lock_{name}.json'))
    try:
        lock.lock(r['best_path'], dict(experiment=name, seed=s, img_size=IMG_SIZE, seg_threshold=SEG_THRESHOLD, min_area=MIN_AREA, run_mode=RUN_MODE,
                                       selection='checkpoint/seed chọn theo validation BUS-BRA'), Tm, thr, abst, notes='khóa trước khi đọc BUSI')
        print(f'Đã khóa {name} (seed {s}): T={Tm:.3f}, ngưỡng phân loại={thr:.3f}, ngưỡng từ chối={abst:.3f}')
    except EV.LockError as e:
        print(f'{name}: {e}')
    LOCKS[name] = lock

## 24. BUSI external evaluation
Chỉ benign/malignant (Normal đã bị loại, không gán thành Benign). Dùng **đúng** temperature, ngưỡng phân loại và ngưỡng từ chối đã khóa — không fit lại bất cứ thứ gì trên BUSI. Mỗi mô hình bị khóa chỉ được đánh giá **một lần**; chạy lại cell sẽ đọc kết quả đã ghi trong file khóa.

In [ ]:
EXT = {}
for name, lock in LOCKS.items():
    ck = RUNS[(name, CAL[name]['seed'])]['best_path']
    def load_busi():
        return C2.load_arrays(busi_clean, IMG_SIZE, mask_col='mask_paths')          # lần đầu tiên BUSI được nạp vào bộ nhớ
    def predict(data, L, ck=ck):
        Xb, Mb, yb = data
        model, _ = C2.load_model_from_checkpoint(ck, DEVICE)
        P = C2.predict_arrays(model, Xb, np.arange(len(yb)), DEVICE, tta=bool(MX.EXPERIMENTS.get(name, {}).get('tta', False)))
        pc = T.softmax(np.log(np.clip(P['prob'], 1e-12, 1)), L['temperature'])
        out = dict(n=int(len(yb)), n_malignant=int(yb.sum()), **EV.classification_metrics(yb, pc[:, 1], L['threshold']))
        lo, hi = EV.bootstrap_ci(lambda i: EV.classification_metrics(yb[i], pc[i, 1])['auroc'], len(yb), B=500)
        out['auroc_ci95'] = [lo, hi]
        out['calibration_locked_T'] = dict(NLL=T.nll(pc, yb), Brier=T.brier(pc, yb), ECE=T.ece(pc, yb))
        acc = pc.max(1) >= L['abstain_threshold']; pred = pc.argmax(1); mal = yb == 1
        out['selective'] = dict(coverage=float(acc.mean()), selective_risk=float((pred[acc] != yb[acc]).mean()) if acc.any() else float('nan'),
                                sensitivity_malignant_accepted=float((pred[acc & mal] == 1).mean()) if (acc & mal).any() else float('nan'),
                                malignant_abstained=float((~acc[mal]).mean()), aurc=T.risk_coverage(-pc.max(1), pred != yb)[2])
        if P['seg_prob'] is not None:
            out['segmentation'] = EV.seg_metrics(P['seg_prob'], Mb)['summary']
        if P['evidence'] is not None:
            out['localisation_evidence'] = EV.localisation_report(P['evidence'], Mb); out['collapse'] = EV.evidence_collapse_report(P['evidence'])
        return out
    try:
        EXT[name] = EV.evaluate_external(lock, ck, load_busi, predict)
    except EV.LockError as e:
        print(f'{name}: {e} -> dùng kết quả đã lưu trong file khóa')
        EXT[name] = lock.read().get('external_summary')
json.dump(EXT, open(os.path.join(OUT_DIR, 'external_busi_results.json'), 'w'), indent=1, default=str)
for n, r in EXT.items():
    if r:
        print(f"{n}: AUROC={r['auroc']:.4f} (95% CI {r['auroc_ci95'][0]:.3f}-{r['auroc_ci95'][1]:.3f}) | sens={r['sensitivity_malignant']:.3f} | spec={r['specificity']:.3f} | AUPRC={r['auprc']:.3f} | ECE (T khóa)={r['calibration_locked_T']['ECE']:.3f} | coverage={r['selective']['coverage']:.2f}")

## 25. Statistical analysis
* **Theo seed:** trung bình ± độ lệch chuẩn (mục 19; sd chỉ có ý nghĩa khi ≥ 2 seed).
* **So sánh cặp** trên cùng ảnh test (seed đầu tiên): paired bootstrap 1000 lần cho ΔAUROC và ΔDice, kèm khoảng tin cậy 95%. **Nếu CI chứa 0, chưa đủ bằng chứng khác biệt** — không được viết "vượt trội".

In [ ]:
yt = y[test_idx]
def paired(a, b, metric):
    ka, kb = (a, SEEDS[0]), (b, SEEDS[0])
    if ka not in PRED or kb not in PRED:
        return None
    Pa, Pb = PRED[ka]['test'], PRED[kb]['test']
    if metric == 'auroc':
        if Pa['prob'] is None or Pb['prob'] is None:
            return None
        fa = lambda i: EV.classification_metrics(yt[i], Pa['prob'][i, 1])['auroc']; fb = lambda i: EV.classification_metrics(yt[i], Pb['prob'][i, 1])['auroc']
    else:
        if Pa['seg_prob'] is None or Pb['seg_prob'] is None:
            return None
        sa, sb = EV.seg_metrics(Pa['seg_prob'], M[test_idx], with_hd95=False), EV.seg_metrics(Pb['seg_prob'], M[test_idx], with_hd95=False)
        has = sa['has_lesion']
        fa = lambda i: float(sa['dice'][i][has[i]].mean()); fb = lambda i: float(sb['dice'][i][has[i]].mean())
    d, lo, hi = EV.paired_bootstrap_diff(fa, fb, len(yt), B=1000)
    verdict = 'CI chứa 0: chưa đủ bằng chứng khác biệt' if lo <= 0 <= hi else ('CI > 0: A cao hơn B' if lo > 0 else 'CI < 0: B cao hơn A')
    return dict(A=a, B=b, metric=metric, diff=d, ci_low=lo, ci_high=hi, verdict=verdict)

PAIRS = [('X1', 'X0'), ('X2', 'X1'), ('X3', 'X2'), ('X4', 'X3'), ('X5', 'X3'), ('X6', 'X3'), ('X6', 'X0'), ('X6', 'E3'), ('A_evidence_decoupled', 'X3'), ('A_evidence_egr', 'X3'), ('V3_base', 'X6'), ('V3_birads', 'V3_base'), ('V3_full', 'V3_birads'), ('V3_full', 'X6'), ('V3_full', 'X0')]
stats = [r for a, b in PAIRS for mt in ('auroc', 'dice') if (r := paired(a, b, mt))]
stats_df = pd.DataFrame(stats); stats_df.to_csv(os.path.join(OUT_DIR, 'paired_comparisons.csv'), index=False)
print(stats_df.round(4).to_string(index=False) if len(stats_df) else 'Chưa đủ mô hình để so sánh cặp.')

## 26. Visualization

In [ ]:
if (PRIMARY, SEEDS[0]) in RUNS:
    m = load_run((PRIMARY, SEEDS[0])); ks = test_idx[:4]
    fig, axes = plt.subplots(len(ks), 5, figsize=(16, 3.2 * len(ks))); axes = np.atleast_2d(axes)
    for r_, k in enumerate(ks):
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            o = m(x)
        pm = (torch.sigmoid(o['seg'].float())[0, 0].cpu().numpy() >= SEG_THRESHOLD).astype(np.uint8)
        cam, _ = C2.gradcam_dict(m, x); ev = EV.upsample_evidence(o['evidence'].float().cpu(), (IMG_SIZE, IMG_SIZE))[0] if o.get('evidence') is not None else None
        p = float(torch.softmax(o['logits'].float(), 1)[0, 1])
        axes[r_, 0].imshow(X[k]); axes[r_, 0].set_title(f"{['benign', 'malignant'][y[k]]} (P_mal={p:.2f})", fontsize=9)
        axes[r_, 1].imshow(core.overlay_mask(X[k], M[k], color=(0, 255, 0))); axes[r_, 1].set_title('GT mask (chỉ để so sánh)', fontsize=9)
        axes[r_, 2].imshow(core.draw_boxes(core.overlay_mask(X[k], pm), core.mask_to_boxes(pm, MIN_AREA))); axes[r_, 2].set_title('mask + box dự đoán', fontsize=9)
        axes[r_, 3].imshow(core.overlay_cam(X[k], cam)); axes[r_, 3].set_title('Grad-CAM', fontsize=9)
        axes[r_, 4].imshow(core.overlay_cam(X[k], ev / (ev.max() + 1e-12)) if ev is not None else X[k]); axes[r_, 4].set_title('evidence map', fontsize=9)
        for a in axes[r_]: a.axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'examples_primary.png'), dpi=110); plt.show(); del m

## 27. Streamlit export
Cell ghi `app.py` và tạo thư mục `deploy_xai/` gồm: các module, checkpoint của mô hình chính (seed chọn theo validation), `deploy_config.json` (T, ngưỡng, ngưỡng từ chối đã khóa), `results/`, `requirements.txt`. Ứng dụng hiển thị ảnh, dự đoán benign/malignant, xác suất đã hiệu chuẩn, mask, bounding box, Grad-CAM, evidence map, trạng thái từ chối và cảnh báo giới hạn; **không lưu ảnh, không có đăng nhập hay thông tin định danh**.

In [ ]:
%%writefile app.py
# app.py - BreastVision-XAI V3: ứng dụng Streamlit phân tích ảnh siêu âm vú (công cụ hỗ trợ nghiên cứu)
import os, json
import numpy as np
import cv2
import torch
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import bus_core as core
import bus_core_v2 as C2

HERE = os.path.dirname(os.path.abspath(__file__))
RES = os.path.join(HERE, 'results')
st.set_page_config(page_title='BreastVision-XAI V3', layout='wide')
page = st.sidebar.radio('Trang', ['Phân tích ảnh', 'Kết quả nghiên cứu', 'Hướng dẫn và giới hạn'])
st.sidebar.warning('Công cụ hỗ trợ NGHIÊN CỨU. Không phải thiết bị chẩn đoán y tế. Không nhập ảnh có thông tin định danh bệnh nhân.')


def load_cfg():
    p = os.path.join(HERE, 'deploy_config.json')
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None


@st.cache_resource
def get_model(ckpt):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    m, meta = C2.load_model_from_checkpoint(ckpt, device)
    return m, device


if page == 'Phân tích ảnh':
    st.title('BreastVision-XAI V3: phân tích ảnh siêu âm vú')
    cfg = load_cfg()
    if cfg is None:
        st.error('Không tìm thấy deploy_config.json. Hãy chạy notebook BreastVision_XAI_V3.ipynb (mục 27) và đặt các file xuất ra cạnh app.py.'); st.stop()
    if cfg.get('run_mode') != 'full_experiment':
        st.error('MÔ HÌNH NÀY TỪ CHẾ ĐỘ SMOKE TEST (dữ liệu tổng hợp). Kết quả KHÔNG có ý nghĩa khoa học hay y khoa.')
    if not cfg.get('pretrained_loaded', False):
        st.warning('Backbone không dùng trọng số pretrained ImageNet trong lần huấn luyện này.')
    ck = os.path.join(HERE, cfg['checkpoint'])
    if not os.path.exists(ck):
        st.error('Không tìm thấy checkpoint: ' + ck); st.stop()
    model, device = get_model(ck)
    st.sidebar.caption(f"Mô hình: {cfg['experiment']} | img_size {cfg['img_size']} | T={cfg['temperature']:.2f} | thiết bị: {device}")
    up = st.file_uploader('Tải ảnh siêu âm (png/jpg/bmp). Ảnh chỉ được xử lý trong bộ nhớ, không lưu.', type=['png', 'jpg', 'jpeg', 'bmp'])
    if up is None:
        st.info('Hãy tải lên một ảnh siêu âm.')
    else:
        bgr = cv2.imdecode(np.frombuffer(up.read(), np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:
            st.error('Không đọc được ảnh.'); st.stop()
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        with st.spinner('Đang phân tích...'):
            r = C2.analyze_image_v2(rgb, model, cfg, device)
        if r['abstain']:
            st.warning('HỆ THỐNG TỪ CHỐI DỰ ĐOÁN: độ tin cậy sau hiệu chuẩn thấp hơn ngưỡng đã chọn trên tập validation. Cần chuyên gia xem xét.')
        else:
            st.info('Không bị từ chối. Điều này KHÔNG có nghĩa dự đoán là đúng.')
        c1, c2 = st.columns(2)
        with c1:
            st.subheader('Dự đoán của mô hình')
            st.write(f"Nhãn theo ngưỡng đã khóa: **{r['label']}** (kết quả nghiên cứu, không phải chẩn đoán)")
            st.table({'Lớp': ['benign', 'malignant'], 'Xác suất đã hiệu chuẩn (%)': [round(float(p) * 100, 2) for p in r['p_cal']],
                      'Xác suất thô (%)': [round(float(p) * 100, 2) for p in r['p_raw']]})
        with c2:
            st.subheader('Vùng nghi ngờ do mô hình phân vùng')
            st.write(f"Tỉ lệ diện tích mask: {r['mask'].mean() * 100:.2f}% | số vùng (bounding box): {len(r['boxes'])}")
            st.write(r['boxes'])
        panels = [('Ảnh đã tiền xử lý', r['image']), ('Mask dự đoán', core.overlay_mask(r['image'], r['mask'])),
                  (f"Bounding box ({len(r['boxes'])})", core.draw_boxes(r['image'], r['boxes'])), ('Grad-CAM', core.overlay_cam(r['image'], r['cam']))]
        if r['evidence'] is not None:
            ev = r['evidence'] / (r['evidence'].max() + 1e-12)
            panels.append(('Evidence map (nối trực tiếp vào logit)', core.overlay_cam(r['image'], ev)))
        fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.2))
        for a, (t, im) in zip(np.atleast_1d(ax), panels):
            a.imshow(im); a.set_title(t, fontsize=9); a.axis('off')
        st.pyplot(fig)
        st.caption('Grad-CAM và evidence map là giải thích về cách mô hình tính điểm, chưa phải bằng chứng y khoa. Độ trung thực (faithfulness) của chúng chỉ được xác nhận bằng các thử nghiệm trong báo cáo nghiên cứu.')
        st.caption('Giới hạn: huấn luyện trên một bộ dữ liệu công khai, chưa kiểm định lâm sàng; ngưỡng từ chối chỉ là công cụ hỗ trợ.')

elif page == 'Kết quả nghiên cứu':
    st.title('Kết quả thực nghiệm đã lưu')
    cfg = load_cfg()
    if cfg is not None and cfg.get('run_mode') != 'full_experiment':
        st.error('Các tệp kết quả hiện có đến từ SMOKE TEST (dữ liệu tổng hợp): chỉ kiểm tra mã, không có giá trị kết luận.')
    st.caption('Chỉ hiển thị các tệp do notebook tạo ra; không có số liệu nhập tay.')
    shown = False
    if os.path.isdir(RES):
        for fn in sorted(os.listdir(RES)):
            p = os.path.join(RES, fn)
            if fn.endswith('.csv'):
                st.subheader(fn); st.dataframe(pd.read_csv(p)); shown = True
            elif fn.endswith('.png'):
                st.image(p, caption=fn); shown = True
    if not shown:
        st.warning('Chưa có tệp kết quả trong thư mục results/.')

else:
    st.title('Hướng dẫn và giới hạn')
    st.markdown("""
**Cài đặt:** `pip install -r requirements.txt` rồi `streamlit run app.py`.

**Kiến trúc:** encoder EfficientNet-B3 chung → decoder phân vùng → gộp đặc trưng 3 vùng (toàn ảnh, tổn thương, quanh tổn thương) bằng *mask dự đoán* →
evidence map nối trực tiếp vào logit → hiệu chuẩn nhiệt độ và từ chối dự đoán chọn trên validation.

**Giới hạn:**
- Không dùng mask ground-truth khi suy luận; mask hiển thị là do mô hình dự đoán.
- Ứng dụng không lưu ảnh và không ghi thông tin định danh.
- Mô hình chưa được kiểm định lâm sàng; mọi kết quả cần chuyên gia đánh giá.
- Từ chối dự đoán không đảm bảo các dự đoán còn lại là đúng.
""")

In [ ]:
DEP = os.path.join(WORK_DIR, 'deploy_xai'); shutil.rmtree(DEP, ignore_errors=True); os.makedirs(os.path.join(DEP, 'results'))
for f in ['bus_core.py', 'bus_trust.py', 'losses_xai.py', 'models_xai.py', 'data_audit.py', 'bus_core_v2.py', 'evaluation_xai.py', 'app.py']:
    shutil.copy(f, DEP)
if PRIMARY in CAL:
    s = CAL[PRIMARY]['seed']; L = LOCKS[PRIMARY].read(); ckpt = RUNS[(PRIMARY, s)]['best_path']; shutil.copy(ckpt, DEP)
    meta = C2.core.torch_load(ckpt, torch.device('cpu'))['meta']
    json.dump(dict(experiment=PRIMARY, checkpoint=os.path.basename(ckpt), img_size=IMG_SIZE, temperature=L['temperature'], threshold=L['threshold'],
                   abstain_threshold=L['abstain_threshold'], seg_threshold=SEG_THRESHOLD, min_area=MIN_AREA, run_mode=RUN_MODE,
                   pretrained_loaded=bool(meta.get('pretrained_loaded', False)), seed=s, tta=bool(MX.EXPERIMENTS.get(PRIMARY, {}).get('tta', False))), open(os.path.join(DEP, 'deploy_config.json'), 'w'), indent=1)
for d_ in (OUT_DIR, FIG_DIR):
    for f in os.listdir(d_):
        if f.endswith(('.csv', '.png', '.json')) and not f.startswith('lock_'):
            shutil.copy(os.path.join(d_, f), os.path.join(DEP, 'results', f))
open(os.path.join(DEP, 'requirements.txt'), 'w').write('streamlit\ntorch\ntorchvision\nopencv-python-headless\nmatplotlib\nnumpy\npandas\nscikit-learn\nscipy\n')
print('deploy_xai:', sorted(os.listdir(DEP)))

## 28. Unit tests
Chạy toàn bộ bộ kiểm thử (dataset, phân vùng, pooling đa vùng, evidence–logit coupling, explanation loss, consistency + inverse warp, gradient flow, checkpoint, **suy luận không dùng GT mask**, rò rỉ dữ liệu, **khóa đánh giá ngoài**, hiệu chuẩn/chọn lọc, ứng dụng). Kết quả ghi vào `test_results.json`.

In [ ]:
import tests_xai
importlib.reload(tests_xai)
all_ok = tests_xai.main()
shutil.copy(os.path.join(os.path.dirname(tests_xai.__file__), 'test_results.json'), os.path.join(OUT_DIR, 'test_results.json'))
assert all_ok, 'Có test thất bại: xem log phía trên.'

## 29. Research report (tự động từ kết quả đã lưu)

In [ ]:
def md_table(df_, n=40):
    return df_.head(n).round(4).to_string(index=False) if df_ is not None and len(df_) else '(chưa có)'
banner = ('**SMOKE TEST: dữ liệu tổng hợp. Không có kết quả khoa học nào trong báo cáo này.**' if SMOKE else
          '**Full experiment. Kết quả dưới đây là số liệu thực từ lần chạy này; chưa phải kết luận lâm sàng.**')
rep_lines = [f'# Báo cáo thực nghiệm BreastVision-XAI V3 ({RUN_MODE})', '', banner, '', f'Thiết bị: {DEVICE}; IMG_SIZE={IMG_SIZE}; EPOCHS={EPOCHS}; seeds={SEEDS}; pretrained={PRETRAINED}', '',
             '## Dữ liệu', f'- BUS-BRA: {len(df)} ảnh, {df["patient_id"].nunique()} bệnh nhân; split theo bệnh nhân: {df.groupby("split").size().to_dict()}',
             f'- Rò rỉ: {leak}', f'- BUSI (ngoài, nhị phân): {len(busi_clean)} ảnh; Normal bị loại; bản sao với BUS-BRA bị loại: {int(dup_rows.sum())}', '',
             '## Nội bộ (BUS-BRA test): trung bình ± sd qua seed', '```', md_table(agg), '```', '',
             '## So sánh cặp (paired bootstrap)', '```', md_table(stats_df), '```', '',
             '## XAI (localisation / faithfulness / consistency / collapse)', '```', md_table(xai_df.pivot_table(index=['group', 'metric'], columns='model', values='value').reset_index() if len(xai_df) else None, 80), '```', '',
             '## Hiệu chuẩn và chọn lọc dự đoán', '```', md_table(pd.read_csv(os.path.join(OUT_DIR, 'calibration.csv')) if os.path.exists(os.path.join(OUT_DIR, 'calibration.csv')) else None),
             md_table(pd.read_csv(os.path.join(OUT_DIR, 'selective_prediction.csv')) if os.path.exists(os.path.join(OUT_DIR, 'selective_prediction.csv')) else None), '```', '',
             '## Đánh giá ngoài BUSI (pipeline đã khóa)']
for n, r in EXT.items():
    if r:
        rep_lines.append(f"- {n}: AUROC {r['auroc']:.4f} (95% CI {r['auroc_ci95'][0]:.3f}-{r['auroc_ci95'][1]:.3f}), AUPRC {r['auprc']:.4f}, sens {r['sensitivity_malignant']:.3f}, spec {r['specificity']:.3f}, ECE {r['calibration_locked_T']['ECE']:.3f}")
rep_lines += ['', '## Giới hạn', '- BUS-BRA một quốc gia; BUSI không có mã bệnh nhân; chưa kiểm định lâm sàng.',
              '- Nếu CI của hiệu số chứa 0 thì chưa đủ bằng chứng khác biệt; không viết "vượt trội".',
              '- Localisation ≠ consistency ≠ faithfulness: chỉ faithfulness (deletion/insertion/intervention) nói về ảnh hưởng nhân quả tới dự đoán.',
              '- Ứng dụng là công cụ hỗ trợ nghiên cứu, không phải thiết bị chẩn đoán.']
open(os.path.join(OUT_DIR, 'research_report.md'), 'w', encoding='utf-8').write('\n'.join(rep_lines))
print('\n'.join(rep_lines)[:3500])

## 30. ZIP packaging

In [ ]:
PKG = os.path.join(WORK_DIR, 'BreastVision_XAI_V3_run'); shutil.rmtree(PKG, ignore_errors=True); os.makedirs(PKG)
shutil.copytree(os.path.join(WORK_DIR, 'deploy_xai'), os.path.join(PKG, 'deploy_xai'))
shutil.copytree(OUT_DIR, os.path.join(PKG, 'outputs'), ignore=shutil.ignore_patterns('lock_*.json')); shutil.copytree(FIG_DIR, os.path.join(PKG, 'figures'))
for f in os.listdir(os.path.join(PKG, 'outputs')):
    pass
shutil.copy(os.path.join(OUT_DIR, 'lock_' + PRIMARY + '.json'), os.path.join(PKG, 'outputs', 'pipeline_lock_primary.json')) if os.path.exists(os.path.join(OUT_DIR, 'lock_' + PRIMARY + '.json')) else None
z = shutil.make_archive(os.path.join(WORK_DIR, 'BreastVision_XAI_V3_run_results'), 'zip', PKG)
print('Đã tạo:', z, f'({os.path.getsize(z) / 1e6:.1f} MB)')
print('RUN_MODE =', RUN_MODE, '->', 'KHÔNG có kết luận khoa học' if SMOKE else 'kết quả thực của lần chạy này')